# PyTorch in Practice: From Training Craft to Production
### Notebook 2 of 2 · A self-study course for engineering undergraduates

---

In Notebook 1 you learned the core of PyTorch: tensors, autograd, `nn.Module`, losses, `Dataset`/`DataLoader`, and a complete, honest training pipeline for an MLP. This notebook turns that foundation into the toolkit used in industry and in research code.

By the end you will be able to:
- reason about how tensors live in memory, and write formulas from papers directly with `einsum`;
- control training with optimizers, learning-rate schedules, gradient clipping, early stopping and weight averaging;
- save checkpoints that resume training *exactly*, and make runs reproducible;
- build convolutional networks from first principles, and fine-tune pretrained models with frozen layers;
- make training faster with mixed precision and `torch.compile`, debug it systematically, and export models for deployment.

**Prerequisite:** Notebook 1, or equivalent experience with tensors, autograd, `nn.Module` and a basic training loop. Everything Notebook 1 built that this notebook needs is re-created in the setup section, so you can start here in a fresh session.

## Contents and time budget

| Ch. | Topic | You will be able to… | Time |
| --- | --- | --- | --- |
| 0 | Setup and toolkit | Re-create the Notebook 1 pipeline; know the restore points | 10 min |
| 1 | Tensor internals | Predict when operations copy; fix `view` errors; build sliding windows; use `gather`/`scatter`/`index_add_` | 60 min |
| 2 | `einsum` | Translate index formulas from papers into one line of code | 40 min |
| 3 | Optimizers and learning-rate schedules | Configure optimizers, parameter groups, schedules and warm-up; find a learning rate | 50 min |
| 4 | Training control | Clip and accumulate gradients, stop early, run a *controlled* comparison, write a reusable `fit()` | 55 min |
| 5 | Checkpointing and reproducibility | Resume a run exactly; control and report randomness | 40 min |
| 6 | Weight averaging: EMA and SWA | Average weights for smoother, often better models | 35 min |
| 7 | Convolutional networks | Build convolution from scratch; train 2-D and 1-D CNNs, including variable-length signals with masks | 75 min |
| 8 | Transfer learning | Augment data; freeze layers; compare linear probe, partial and full fine-tuning | 70 min |
| 9 | Evaluation beyond accuracy, and regression | Precision/recall/F1, ROC/PR curves, thresholds, imbalance; a regression surrogate model with target scaling | 60 min |
| 10 | Higher-order autograd and `torch.func` | Differentiate through gradients; Jacobians, `vmap`, per-sample gradients; a physics-informed network | 45 min |
| 11 | Performance | Measure correctly; feed the GPU; mixed precision; avoid hidden synchronisations | 40 min |
| 12 | Debugging training | Find silent bugs with hooks, gradient checks and a protocol | 40 min |
| 13 | Production essentials | Package, export (`torch.export`, ONNX) with metadata, measure latency, test | 45 min |
| | | **Core total** | **≈ 11 h** |
| 14 | ★ Optional deep dives | ResNet on CIFAR-10, custom autograd functions, calibration, profiling, multi-GPU, interview questions | 2–3 h |

Sections marked ★ inside the core chapters are optional extensions (about 1.5 h in total); skipping them brings the core to roughly 9–10 hours. The times are estimates for a first pass; GPU waiting time is included.

**GPU:** use a Colab T4 (*Runtime → Change runtime type → T4 GPU*). Chapters 7–8, 11 and 14 do real training; on a CPU they work but are slow.

**Restore points.** Chapters that reuse tools from earlier chapters start with a hidden 🔁 *restore point* cell. If you open the notebook in a fresh session (Colab disconnects after inactivity), run the two Chapter 0 cells and then the restore point of the chapter you want; you do not need to rerun the chapters in between. With `SAVE_TO_DRIVE = True`, trained models are also kept in your Google Drive, so the restore points can load them instead of retraining.

The conventions are the same as in Notebook 1: 🎯 goals, 🤔 **Predict** before running, ⚠️ pitfalls, 🏭 industry notes, 📖 code-reading drills, 🧪 exercises with hidden ✅ solutions (they only execute when `RUN_SOLUTIONS = True`), and ✔️ "Check yourself" questions.

---
# Chapter 0. Setup and the toolkit from Notebook 1
Run both cells. The first sets up the environment; the second re-creates the data pipeline and helpers from Notebook 1.

In [ ]:
# ---- Setup: run this cell first ------------------------------------------------
import copy, math, os, random, sys, time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision import datasets
from torchvision.transforms import v2 as T
from torch.utils.data import DataLoader, Dataset, Subset, TensorDataset
import matplotlib.pyplot as plt

FAST_DEV_RUN = False     # True = tiny data and 1 epoch everywhere: smoke-test the whole notebook in minutes
RUN_SOLUTIONS = False    # True = also execute the hidden exercise solutions during "Run all" (much slower)
SAVE_TO_DRIVE = False    # True (Colab only) = keep checkpoints in Google Drive, so they survive a disconnect

DATA_DIR = Path("./data"); DATA_DIR.mkdir(exist_ok=True)
CKPT_DIR = Path("./checkpoints")
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    CKPT_DIR = Path("/content/drive/MyDrive/pytorch_course/checkpoints")
CKPT_DIR.mkdir(parents=True, exist_ok=True)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
NUM_WORKERS = 2 if sys.platform.startswith("linux") else 0   # worker processes are fragile in notebooks on Windows/macOS
PIN = device.type == "cuda"

# Mixed precision (Chapter 11): bfloat16 on Ampere and newer GPUs, float16 (with a gradient scaler) on older ones
if device.type == "cuda":
    AMP_DTYPE = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
else:
    AMP_DTYPE = None

def sync():
    # GPU work is asynchronous: wait for it to finish before reading a clock
    if device.type == "cuda":
        torch.cuda.synchronize()

def set_seed(seed: int = 0) -> None:
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

def n_epochs(full: int) -> int:
    return 1 if FAST_DEV_RUN else full

def maybe_subset(ds, n: int = 512):
    return Subset(ds, range(min(n, len(ds)))) if FAST_DEV_RUN else ds

def count_params(m, trainable_only=True):
    return sum(p.numel() for p in m.parameters() if p.requires_grad or not trainable_only)

RESULTS = {}             # the numbers YOU measure, collected as you go and summarised at the end
def record(name, value):
    RESULTS[name] = value
    print(f"📝 recorded  {name}: {value:.4f}")

def self_check(name, test):
    fn = globals().get(name)
    if fn is None:
        print(f"ℹ️  Define `{name}` in the cell above first, then run this cell again."); return
    try:
        test(fn); print(f"✅ `{name}` passed all checks.")
    except AssertionError as e:
        print(f"❌ `{name}` failed a check: {e}")
    except Exception as e:
        print(f"❌ `{name}` raised {type(e).__name__}: {e}")

print(f"torch {torch.__version__} | torchvision {torchvision.__version__} | device: {device} | AMP: {AMP_DTYPE}")
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
set_seed(0)

In [ ]:
# ---- Toolkit from Notebook 1: Fashion-MNIST data, evaluation and the MLP builder ---
raw_train = datasets.FashionMNIST(DATA_DIR, train=True, download=True)
raw_test = datasets.FashionMNIST(DATA_DIR, train=False, download=True)
CLASSES = raw_train.classes
_perm = torch.randperm(len(raw_train), generator=torch.Generator().manual_seed(0))
FM_VAL_IDX, FM_TRAIN_IDX = _perm[:5000], _perm[5000:]            # 55,000 train / 5,000 validation
_px = raw_train.data[FM_TRAIN_IDX].float() / 255                 # statistics from the TRAINING split only
FM_MEAN, FM_STD = _px.mean().item(), _px.std().item()
fm_transform = T.Compose([T.ToImage(), T.ToDtype(torch.float32, scale=True), T.Normalize([FM_MEAN], [FM_STD])])


class TensorLoader:
    # Mini-batches sliced from tensors that were preprocessed ONCE (no per-image Python work).
    def __init__(self, x, y, batch_size, shuffle=False, drop_last=False):
        self.x, self.y, self.bs, self.shuffle, self.drop_last = x, y, batch_size, shuffle, drop_last

    def __len__(self):
        n = len(self.x)
        return n // self.bs if self.drop_last else math.ceil(n / self.bs)

    def __iter__(self):
        n = len(self.x)
        idx = torch.randperm(n, device=self.x.device) if self.shuffle else torch.arange(n, device=self.x.device)
        for i in range(len(self)):
            j = idx[i * self.bs:(i + 1) * self.bs]
            yield self.x[j], self.y[j]


def fashion_tensors(split):
    # The whole split as tensors, preprocessed exactly like fm_transform: (x - mean) / std on [0, 1] pixels.
    if split == "test":
        data, targets = raw_test.data, raw_test.targets
    else:
        idx = FM_TRAIN_IDX if split == "train" else FM_VAL_IDX
        data, targets = raw_train.data[idx], raw_train.targets[idx]
    if FAST_DEV_RUN:
        data, targets = data[:512], targets[:512]
    return ((data.float() / 255 - FM_MEAN) / FM_STD).unsqueeze(1), targets


def fashion_loaders(batch_size=128, eval_batch_size=512):
    # The Notebook 1 pipeline: Dataset + transform + DataLoader (used for comparison in Chapter 11).
    full = datasets.FashionMNIST(DATA_DIR, train=True, transform=fm_transform)
    test = datasets.FashionMNIST(DATA_DIR, train=False, transform=fm_transform)
    tr = maybe_subset(Subset(full, FM_TRAIN_IDX.tolist()))
    va = maybe_subset(Subset(full, FM_VAL_IDX.tolist()))
    kw = dict(num_workers=NUM_WORKERS, pin_memory=PIN)
    return (DataLoader(tr, batch_size=batch_size, shuffle=True, drop_last=True, **kw),
            DataLoader(va, batch_size=eval_batch_size, **kw),
            DataLoader(maybe_subset(test), batch_size=eval_batch_size, **kw))


@torch.no_grad()
def evaluate(model, loader, loss_fn=nn.CrossEntropyLoss()):
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        total_loss += loss_fn(logits, y).item() * len(y)
        correct += (logits.argmax(dim=1) == y).sum().item()
        n += len(y)
    return total_loss / n, correct / n


def build_mlp(hidden=(256, 128), p_drop=0.2, batchnorm=False):
    layers, d = [nn.Flatten()], 784
    for h in hidden:
        layers += [nn.Linear(d, h)] + ([nn.BatchNorm1d(h)] if batchnorm else []) + [nn.ReLU(), nn.Dropout(p_drop)]
        d = h
    return nn.Sequential(*layers, nn.Linear(d, 10))


train_loader = TensorLoader(*fashion_tensors("train"), batch_size=128, shuffle=True, drop_last=True)
val_loader = TensorLoader(*fashion_tensors("val"), batch_size=1024)
test_loader = TensorLoader(*fashion_tensors("test"), batch_size=1024)
print(f"Fashion-MNIST ready: {len(train_loader.x):,} train / {len(val_loader.x):,} val / {len(test_loader.x):,} test")

Two changes from Notebook 1:
- **Fashion-MNIST is preprocessed once.** The toolkit converts each split into one normalised tensor (exactly what `fm_transform` would produce, image by image) and `TensorLoader` slices shuffled mini-batches from it. This makes epochs several times faster than decoding every image in Python, which matters because this notebook runs many experiments. Batches still arrive on the CPU and are moved with `.to(device)`, exactly like `DataLoader` batches. Chapter 11 measures the difference and shows how to go further. `fashion_loaders()` still builds the Notebook 1 `DataLoader` pipeline when we need it.
- **The training loader uses `drop_last=True`.** BatchNorm and some techniques in this notebook behave badly with a tiny final batch, and dropping at most 127 of 55,000 examples per epoch costs nothing, because the data is reshuffled every epoch.

---
# Chapter 1. Tensor internals: memory, strides and views

🎯 **In this chapter you will learn to:** explain how a tensor maps indices to memory; predict which operations copy data and which do not; understand and fix the `view` / `contiguous` errors you will meet in other people's code; and use strides to build sliding windows without copying.

In Notebook 1 we said "prefer `reshape` over `view`" without saying why. This chapter explains why. It also explains a line you will find in countless research repositories: `x.transpose(1, 2).contiguous().view(B, T, -1)`.

## 1.1 A tensor is a view onto a block of memory
A tensor's numbers live in a flat, one-dimensional block of memory called its **storage**. The tensor itself is only a small description of how to read that block:

| Field | Meaning | Example for `torch.arange(12).reshape(3, 4)` |
| --- | --- | --- |
| `storage` | The flat block of numbers | `0, 1, 2, …, 11` |
| `storage_offset()` | Where element `[0, 0, …]` sits in the storage | `0` |
| `shape` | Size along each dimension | `(3, 4)` |
| `stride()` | How many storage positions to jump to move **one step** along each dimension | `(4, 1)` |

The address of element `[i, j]` is

$$\text{offset} + i \cdot \text{stride}_0 + j \cdot \text{stride}_1.$$

For a `(3, 4)` matrix stored row by row, moving one column means moving 1 position in memory, and moving one row means skipping a whole row of 4. Hence strides `(4, 1)`.

In [ ]:
x = torch.arange(12).reshape(3, 4)
print(x)
print("shape:", tuple(x.shape), "| stride:", x.stride(), "| offset:", x.storage_offset())

def address(t, idx):
    # Storage position of element t[idx], computed from offset and strides.
    return t.storage_offset() + sum(i * s for i, s in zip(idx, t.stride()))

print("x[2, 1] =", x[2, 1].item(), "lives at storage position", address(x, (2, 1)), "= 0 + 2*4 + 1*1")

For a contiguous tensor, the strides follow directly from the shape: the last dimension has stride 1, and each earlier stride is the product of all later sizes. A `(2, 3, 4)` tensor therefore has strides `(12, 4, 1)`.

## 1.2 Many operations only change the description
Here is the key insight. Operations like **slicing, `transpose`, `permute`, `expand`, `squeeze`/`unsqueeze` and (when possible) `view`** do not move any numbers. They return a **view**: a new description (shape, strides, offset) of the **same storage**. Views are free to create, regardless of the tensor's size, and they share memory with the original.

🤔 **Predict** the strides and offsets in the next cell before you run it.

In [ ]:
x = torch.arange(12).reshape(3, 4)

def describe(name, t):
    print(f"{name:10s} shape={str(tuple(t.shape)):9s} stride={str(t.stride()):8s} offset={t.storage_offset()}  "
          f"shares memory with x: {t.untyped_storage().data_ptr() == x.untyped_storage().data_ptr()}")

describe("x", x)
describe("x[1]", x[1])              # row 1: start at position 4, step 1
describe("x[:, 2]", x[:, 2])        # column 2: start at 2, step 4 (a whole row)
describe("x[::2]", x[::2])          # every other row: row stride doubles to 8
describe("x.T", x.T)                # transpose: the two strides swap
describe("x[None]", x[None])        # a new size-1 dimension (its stride never matters)

Because views share storage, **writing through a view changes the original**. This is efficient, and occasionally a surprise:

In [ ]:
x = torch.zeros(3, 4)
col = x[:, 2]           # a view
col += 7                # in-place write through the view
print(x)                # the original changed

safe = x[:, 2].clone()  # clone() = a real copy with its own storage
safe += 100
print("x unchanged by writing to the clone:", x[:, 2])

⚠️ **Pitfall: a small slice can keep a huge tensor alive.** Since a view shares storage, `small = huge[:10]` keeps all of `huge` in memory as long as `small` exists. If you keep slices of large batches around (for example, saving a few examples for plotting), use `.clone()` so the big storage can be freed.

## 1.3 Contiguity, and why `view` sometimes fails
A tensor is **contiguous** when reading its elements in normal row-major order (last index fastest) walks through storage positions 0, 1, 2, … in order. A freshly created tensor is contiguous. Its transpose is not: reading `x.T` row by row jumps around in memory.

In [ ]:
x = torch.arange(6).reshape(2, 3)
xt = x.T
print("x  =", x.tolist(), "stride", x.stride(), "contiguous:", x.is_contiguous())
print("xt =", xt.tolist(), "stride", xt.stride(), "contiguous:", xt.is_contiguous())
print("reading xt row by row visits storage positions:",
      [address(xt, (i, j)) for i in range(3) for j in range(2)])   # not 0, 1, 2, ...

`view(new_shape)` promises to return a view, without copying. It can only keep that promise if the new shape can be described with strides over the **existing** memory layout. Flattening `xt` into `[0, 3, 1, 4, 2, 5]` would need storage positions in the order 0, 3, 1, 4, 2, 5. No single stride does that, so `view` refuses. The two fixes:

- `.contiguous()` copies the data into a new storage in row-major order (a no-op if the tensor already is contiguous); `view` then works;
- `reshape` does that automatically: it returns a view when possible and a copy when necessary.

In [ ]:
try:
    xt.view(6)
except RuntimeError as e:
    print("view failed:", str(e)[:95], "...")

flat1 = xt.contiguous().view(6)
flat2 = xt.reshape(6)
print("contiguous().view:", flat1.tolist(), "| reshape:", flat2.tolist())
print("reshape of a contiguous tensor is a view (shares memory):",
      x.reshape(6).data_ptr() == x.data_ptr())
print("reshape of xt had to copy (new memory):", flat2.data_ptr() != x.data_ptr())

### 📖 Decoding a line you will see everywhere
In attention code, sequence models and many papers, you will see something like

```python
out = out.transpose(1, 2).contiguous().view(B, T, H * D)     # (B, H, T, D) -> (B, T, H*D)
```

Now you can read it: `transpose(1, 2)` reorders the dimensions to `(B, T, H, D)` without moving data (so the result is non-contiguous); `.contiguous()` physically rearranges the memory; and `view` then merges the last two dimensions. Writing `.reshape(B, T, H * D)` instead would do exactly the same.

🏭 **Why do people still write `view`?** Partly habit (`reshape` was added later), and partly because `view` *guarantees* no copy. If an unexpected copy would be a performance bug, `view` turns it into a loud error. Both are fine; know what each does.

## 1.4 Broadcasting is a stride of 0
How does broadcasting avoid copying data? `expand` (which broadcasting uses internally) sets the stride of a stretched dimension to **0**: moving along that dimension does not move in memory, so the same value is read again and again. `repeat`, in contrast, really copies.

In [ ]:
b = torch.tensor([1.0, 2.0, 3.0])
big_view = b.expand(1000, 3)          # pretend there are 1000 rows
big_copy = b.repeat(1000, 1)          # really make 1000 rows
print("expand: shape", tuple(big_view.shape), "stride", big_view.stride(), "-> no new memory")
print("repeat: shape", tuple(big_copy.shape), "stride", big_copy.stride(),
      f"-> {big_copy.numel() * big_copy.element_size():,} bytes of new memory")

try:
    big_view += 1                     # 1000 'rows' would all write to the same 3 numbers
except RuntimeError as e:
    print("in-place write to an expanded tensor:", str(e)[:80], "...")

## 1.5 Sliding windows from strides: `as_strided` and `unfold`
Strides can do more than describe existing layouts. With `as_strided(size, stride)` you choose them yourself, which lets you build **overlapping windows** without copying a single number.

**Example: a moving average of a sensor signal.** For a signal of length `n` and a window of length `k`, window `i` contains elements `i, i+1, …, i+k−1`. As a `(n−k+1, k)` matrix, moving to the next **window** (row) moves 1 position in memory, and moving to the next element **inside** a window (column) also moves 1. So the strides are `(1, 1)`.

In [ ]:
signal = torch.arange(10.0)
k = 4
windows = signal.as_strided(size=(len(signal) - k + 1, k), stride=(1, 1))
print(windows)
print("moving average:", windows.mean(dim=1))
print("no copy was made:", windows.data_ptr() == signal.data_ptr())

In [ ]:
# A noisy measurement smoothed with a 15-sample moving average
set_seed(0)
t = torch.linspace(0, 4 * math.pi, 400)
noisy = torch.sin(t) + 0.3 * torch.randn(400)
k = 15
smooth = noisy.as_strided((400 - k + 1, k), (1, 1)).mean(dim=1)

plt.figure(figsize=(8, 3))
plt.plot(t, noisy, lw=0.7, alpha=0.6, label="noisy signal")
plt.plot(t[k // 2: k // 2 + len(smooth)], smooth, lw=2, label=f"moving average (k={k})")
plt.legend(); plt.title("Sliding windows via strides"); plt.show()

⚠️ **`as_strided` is a sharp tool.** It does no bounds checking. Wrong sizes or strides read memory outside the tensor (garbage, or a crash), and writing into overlapping windows silently corrupts data. Strides are also measured in **elements of the underlying storage**, so they are wrong if you apply `as_strided` to a non-contiguous tensor without accounting for its strides. In practice, use the safe wrapper **`Tensor.unfold(dimension, size, step)`**, which builds the same windows with bounds checks. Know `as_strided` so you understand what `unfold` does, and so you can read code that uses it.

In [ ]:
w2 = signal.unfold(0, 4, 1)        # dimension 0, windows of size 4, step 1
print("unfold gives the same windows:", torch.equal(w2, windows))
print("non-overlapping windows with step 4:\n", signal.unfold(0, 4, 4))   # the leftover 2 elements are dropped

img = torch.arange(16.0).reshape(4, 4)
patches = img.unfold(0, 2, 2).unfold(1, 2, 2)   # 2x2 patches of a 4x4 image -> (2, 2, 2, 2)
print("patch grid shape:", tuple(patches.shape), "| top-right patch:\n", patches[0, 1])

These patches are the starting point of **convolution**, which in Chapter 7 you will build from exactly this idea: cut the image into overlapping patches, then multiply each patch by a filter.

## 1.6 `gather`, `scatter` and `index_add_`: indexing as an operation
Research code rarely uses Python loops over examples. Three operations do most of the "pick" and "collect" work, and you will meet them in loss functions, in graph neural networks and in any code that groups data.

| Operation | Meaning for a 2-D tensor with `dim=1` | Typical use |
| --- | --- | --- |
| `x.gather(1, idx)` | `out[i, j] = x[i, idx[i, j]]` | pick the logit of the correct class; pick the top-k entries |
| `x.scatter(1, idx, v)` | `out[i, idx[i, j]] = v[i, j]` (writes instead of reads) | one-hot encodings; writing values to chosen positions |
| `out.index_add_(0, ids, v)` | `out[ids[k]] += v[k]` for every row `k` | sums per group: class means, message passing in graphs |

`gather` and `scatter` are inverses of each other: one reads at given positions, the other writes there. Unlike basic slicing, they take a **tensor** of positions, so they always copy (Exercise 1.3).

In [ ]:
logits = torch.tensor([[2.0, 0.5, -1.0],
                       [0.1, 0.2, 3.0],
                       [1.0, 2.0, 0.0]])
y = torch.tensor([0, 2, 1])

picked = logits.gather(1, y[:, None]).squeeze(1)              # idx needs the same number of dims as x
print("gather:                 ", picked)
print("same as advanced indexing:", logits[torch.arange(3), y])

one_hot = torch.zeros(3, 3).scatter(1, y[:, None], 1.0)       # write 1.0 at column y[i] of row i
print("scatter one-hot:\n", one_hot, "\nequals F.one_hot:", torch.equal(one_hot, F.one_hot(y, 3).float()))

**Grouped sums without loops.** Suppose `feats` holds a 2-D feature vector for each of 6 examples and `labels` says which of 3 classes each belongs to. The mean feature vector of every class (a "prototype", used for example in few-shot learning) needs a sum per group. `index_add_` adds row `k` of `feats` into row `labels[k]` of the output; `torch.bincount` counts the members of each group.

In [ ]:
feats = torch.tensor([[1.0, 0.0], [3.0, 2.0], [0.0, 1.0], [2.0, 2.0], [5.0, 5.0], [4.0, 0.0]])
labels = torch.tensor([0, 0, 1, 1, 2, 0])

sums = torch.zeros(3, 2).index_add_(0, labels, feats)       # (num_classes, D)
counts = torch.bincount(labels, minlength=3)                # (num_classes,)
means = sums / counts[:, None]
print("class means:\n", means)
loop = torch.stack([feats[labels == c].mean(dim=0) for c in range(3)])
print("matches a Python loop:", torch.allclose(means, loop))

## 🧪 Exercises

**Exercise 1.1: contiguous strides.** `★☆☆ · ~10 min`
Write `contiguous_strides(shape)` that returns the strides of a contiguous tensor with the given shape, as a tuple. For example `(2, 3, 4)` gives `(12, 4, 1)`. Do not create a tensor.

In [ ]:
# ✍️ Your attempt here.
# Hint: Walk the shape from the right, keeping a running product.

In [ ]:
# 🔎 Self-check for Exercise 1.1
def _test(fn):
    for shape in [(2, 3, 4), (5,), (7, 1, 3), (2, 2, 2, 2)]:
        assert tuple(fn(shape)) == torch.empty(shape).stride(), f"wrong strides for {shape}"
self_check("contiguous_strides", _test)

In [ ]:
# @title ✅ Solution (double-click to reveal) { display-mode: "form" }
# Solutions only execute when RUN_SOLUTIONS = True (setup cell), so "Run all" stays fast.
if RUN_SOLUTIONS:
    def contiguous_strides(shape):
        strides, running = [], 1
        for size in reversed(shape):
            strides.append(running)
            running *= size
        return tuple(reversed(strides))

    print(contiguous_strides((2, 3, 4)))
else:
    print("Solution not executed (RUN_SOLUTIONS = False).")

**Exercise 1.2: framing a signal.** `★★☆ · ~15 min`
Audio and vibration analysis cut a signal into overlapping **frames** of length `frame_len`, starting every `hop` samples (the first step of a spectrogram). Write `frame_signal(x, frame_len, hop)` for a 1-D contiguous tensor `x` that returns a `(num_frames, frame_len)` **view** made with `as_strided`, where `num_frames = 1 + (len(x) − frame_len) // hop`.

In [ ]:
# ✍️ Your attempt here.
# Hint: Moving to the next frame jumps `hop` positions in memory; moving inside a frame jumps 1.

In [ ]:
# 🔎 Self-check for Exercise 1.2
def _test(fn):
    x = torch.arange(20.0)
    for L, h in [(4, 2), (5, 5), (8, 3)]:
        out = fn(x, L, h)
        assert torch.equal(out, x.unfold(0, L, h)), f"wrong frames for frame_len={L}, hop={h}"
        assert out.data_ptr() == x.data_ptr(), "should be a view (no copy)"
self_check("frame_signal", _test)

In [ ]:
# @title ✅ Solution (double-click to reveal) { display-mode: "form" }
# Solutions only execute when RUN_SOLUTIONS = True (setup cell), so "Run all" stays fast.
if RUN_SOLUTIONS:
    def frame_signal(x, frame_len, hop):
        num_frames = 1 + (len(x) - frame_len) // hop
        return x.as_strided(size=(num_frames, frame_len), stride=(hop, 1))

    print(frame_signal(torch.arange(10.0), 4, 3))
else:
    print("Solution not executed (RUN_SOLUTIONS = False).")

**Exercise 1.3: which of these copy?** `★☆☆ · ~10 min`
For `x = torch.randn(8, 16)`, predict whether each result **shares memory** with `x`, then check with `data_ptr()` (careful: a view with a non-zero offset has a different `data_ptr()`; compare `untyped_storage().data_ptr()` instead, as `describe` did):
`x[2:5]`, `x[:, ::2]`, `x.T`, `x.T.reshape(-1)`, `x.reshape(4, 32)`, `x[[0, 2]]`, `x[x > 0]`, `x.flatten()`, `x.unsqueeze(0)`.

In [ ]:
# ✍️ Your attempt here.

In [ ]:
# @title ✅ Solution (double-click to reveal) { display-mode: "form" }
# Solutions only execute when RUN_SOLUTIONS = True (setup cell), so "Run all" stays fast.
if RUN_SOLUTIONS:
    x = torch.randn(8, 16)
    base = x.untyped_storage().data_ptr()
    cases = {"x[2:5]": x[2:5], "x[:, ::2]": x[:, ::2], "x.T": x.T, "x.T.reshape(-1)": x.T.reshape(-1),
             "x.reshape(4, 32)": x.reshape(4, 32), "x[[0, 2]]": x[[0, 2]], "x[x > 0]": x[x > 0],
             "x.flatten()": x.flatten(), "x.unsqueeze(0)": x.unsqueeze(0)}
    for name, t in cases.items():
        print(f"{name:18s} shares memory: {t.untyped_storage().data_ptr() == base}")
    # Basic slicing (ranges, steps), transposes, unsqueeze and reshapes of contiguous tensors are views.
    # Indexing with a LIST/tensor of indices or a boolean MASK ("advanced indexing") always copies,
    # because the selected elements cannot be described with a fixed stride.
    # x.T.reshape(-1) must copy because x.T is not contiguous.
else:
    print("Solution not executed (RUN_SOLUTIONS = False).")

**Exercise 1.4: mean per segment.** `★★☆ · ~15 min`
Graph neural networks and "set" models constantly average values that belong to the same group (the same node, the same document, the same machine). Write `segment_mean(values, segment_ids, num_segments)` for `values` of shape `(N, D)` and integer `segment_ids` of shape `(N,)`, returning `(num_segments, D)` means. Segments without any member must give zeros, not `nan`. No Python loops.

In [ ]:
# ✍️ Your attempt here.
# Hint: index_add_ for the sums, bincount for the counts; clamp the counts to at least 1 before dividing.

In [ ]:
# 🔎 Self-check for Exercise 1.4
def _test(fn):
    v = torch.randn(10, 3)
    ids = torch.tensor([0, 2, 2, 0, 3, 3, 3, 0, 2, 3])          # segment 1 is empty
    out = fn(v, ids, 5)
    assert out.shape == (5, 3), f"shape {tuple(out.shape)}, expected (5, 3)"
    for s in range(5):
        ref = v[ids == s].mean(dim=0) if (ids == s).any() else torch.zeros(3)
        assert torch.allclose(out[s], ref, atol=1e-6), f"segment {s} is wrong"
self_check("segment_mean", _test)

In [ ]:
# @title ✅ Solution (double-click to reveal) { display-mode: "form" }
# Solutions only execute when RUN_SOLUTIONS = True (setup cell), so "Run all" stays fast.
if RUN_SOLUTIONS:
    def segment_mean(values, segment_ids, num_segments):
        sums = torch.zeros(num_segments, values.shape[1], dtype=values.dtype).index_add_(0, segment_ids, values)
        counts = torch.bincount(segment_ids, minlength=num_segments).clamp(min=1)
        return sums / counts[:, None]

    print(segment_mean(torch.ones(4, 2), torch.tensor([0, 0, 2, 2]), 3))
else:
    print("Solution not executed (RUN_SOLUTIONS = False).")

### 📌 Key takeaways
- A tensor = storage + offset + shape + strides. Slicing, `transpose`, `permute`, `expand` and `view` create **views**: no copying, shared memory.
- Advanced indexing (index lists, boolean masks) copies. `gather`/`scatter`/`index_add_` read, write and sum at positions given by a tensor.
- `view` needs a compatible memory layout; `.contiguous()` makes one by copying. `reshape` = view if possible, copy otherwise.
- Broadcasting = stride 0. `unfold` (or, carefully, `as_strided`) builds overlapping windows without copying.

### ✔️ Check yourself

Commit to an answer (say it out loud or write it down) *before* you expand it.

**1.** What are the strides of a contiguous tensor of shape `(4, 5, 6)`, and of its `permute(2, 0, 1)`?

<details>
<summary>Answer</summary>

`(30, 6, 1)`. The permuted tensor has shape `(6, 4, 5)` and strides `(1, 30, 6)`: permuting reorders the strides along with the sizes.

</details>

**2.** Why does `x.T.view(-1)` fail while `x.view(-1)` works?

<details>
<summary>Answer</summary>

`x.T` is not contiguous: its row-major order does not walk memory in order, so no single stride can describe the flattened result without copying. `view` never copies, so it refuses.

</details>

**3.** You keep `examples = batch[:4]` from every training batch in a list for later plotting, and memory usage explodes. Why, and how do you fix it?

<details>
<summary>Answer</summary>

Each slice is a view that keeps its whole batch's storage alive. Use `batch[:4].clone()` (and `.cpu()` / `.detach()` if needed) so only the 4 examples are kept.

</details>

**4.** What is the stride of a dimension created by broadcasting or `expand`?

<details>
<summary>Answer</summary>

0: every step along it reads the same memory, which is why broadcasting uses no extra memory.

</details>

---
# Chapter 2. `einsum`: index notation as code

🎯 **In this chapter you will learn to:** read and write `torch.einsum` expressions; translate formulas written with indices (as in papers) directly into code; and recognise the equivalent `einops` and `permute`/`reshape` idioms in other people's code.

Papers write tensor operations with indices, for example $y_{ni} = \sum_j x_{nj} W_{ij}$. Translating such formulas into `@`, `transpose`, `sum` and broadcasting is error-prone: you have to remember which dimension is which. `einsum` lets you write the indices directly:

```python
y = torch.einsum("nj,ij->ni", x, W)
```

## 2.1 The rules
An einsum string has the form `"inputs->output"`, with one group of letters per input tensor, separated by commas. Each letter names one dimension.

1. **Every letter labels a dimension**, in order. In `"nj,ij->ni"`, `x` has dimensions `(n, j)` and `W` has dimensions `(i, j)`.
2. **A letter that appears in several inputs** must have the same size in each. Those entries are **multiplied** together.
3. **A letter that does not appear in the output is summed over.** Here `j` is summed: that is the $\sum_j$.
4. **The output** has the remaining letters, in the order written after `->`.

In short: *multiply everything along shared letters, then sum out every letter missing from the output.*

Check the example against `nn.Linear`, which computes $y = xW^\top$:

In [ ]:
lin = nn.Linear(5, 3, bias=False)
x = torch.randn(4, 5)
y = torch.einsum("nj,ij->ni", x, lin.weight)          # y_ni = sum_j x_nj W_ij
print(y.shape, "| matches nn.Linear:", torch.allclose(y, lin(x), atol=1e-6))

## 2.2 A translation table
Every row below is checked in the next cell. Read each einsum string out loud using the rules: which letters are shared (multiplied), which disappear (summed)?

| Operation | Math | einsum | Plain PyTorch |
| --- | --- | --- | --- |
| Transpose | $B_{ji} = A_{ij}$ | `"ij->ji"` | `A.T` |
| Sum of all entries | $\sum_{ij} A_{ij}$ | `"ij->"` | `A.sum()` |
| Row sums | $r_i = \sum_j A_{ij}$ | `"ij->i"` | `A.sum(dim=1)` |
| Trace | $\sum_i A_{ii}$ | `"ii->"` | `A.trace()` |
| Diagonal | $d_i = A_{ii}$ | `"ii->i"` | `A.diagonal()` |
| Elementwise product | $C_{ij} = A_{ij} B_{ij}$ | `"ij,ij->ij"` | `A * B` |
| Dot product | $\sum_i u_i v_i$ | `"i,i->"` | `u @ v` |
| Outer product | $C_{ij} = u_i v_j$ | `"i,j->ij"` | `u[:, None] * v[None, :]` |
| Matrix–vector | $y_i = \sum_j A_{ij} v_j$ | `"ij,j->i"` | `A @ v` |
| Matrix–matrix | $C_{ik} = \sum_j A_{ij} B_{jk}$ | `"ij,jk->ik"` | `A @ B` |
| Batched matmul | $C_{bik} = \sum_j A_{bij} B_{bjk}$ | `"bij,bjk->bik"` | `A @ B` (3-D) |
| Row-wise dot products | $s_n = \sum_d X_{nd} Y_{nd}$ | `"nd,nd->n"` | `(X * Y).sum(dim=1)` |
| Bilinear form, per example | $s_n = \sum_{ij} x_{ni} M_{ij} y_{nj}$ | `"ni,ij,nj->n"` | `((x @ M) * y).sum(dim=1)` |

In [ ]:
A, B = torch.randn(3, 3), torch.randn(3, 3)
u, v = torch.randn(3), torch.randn(3)
X, Y = torch.randn(5, 4), torch.randn(5, 4)
M, xb, yb = torch.randn(4, 4), torch.randn(5, 4), torch.randn(5, 4)
Ab, Bb = torch.randn(2, 3, 4), torch.randn(2, 4, 5)

checks = {
    "transpose":      (torch.einsum("ij->ji", A), A.T),
    "sum":            (torch.einsum("ij->", A), A.sum()),
    "row sums":       (torch.einsum("ij->i", A), A.sum(dim=1)),
    "trace":          (torch.einsum("ii->", A), A.trace()),
    "diagonal":       (torch.einsum("ii->i", A), A.diagonal()),
    "elementwise":    (torch.einsum("ij,ij->ij", A, B), A * B),
    "dot":            (torch.einsum("i,i->", u, v), u @ v),
    "outer":          (torch.einsum("i,j->ij", u, v), u[:, None] * v[None, :]),
    "matrix-vector":  (torch.einsum("ij,j->i", A, u), A @ u),
    "matrix-matrix":  (torch.einsum("ij,jk->ik", A, B), A @ B),
    "batched matmul": (torch.einsum("bij,bjk->bik", Ab, Bb), Ab @ Bb),
    "row-wise dots":  (torch.einsum("nd,nd->n", X, Y), (X * Y).sum(dim=1)),
    "bilinear":       (torch.einsum("ni,ij,nj->n", xb, M, yb), ((xb @ M) * yb).sum(dim=1)),
}
for name, (a, b) in checks.items():
    print(f"{name:15s} {str(tuple(a.shape)):10s} agrees: {torch.allclose(a, b, atol=1e-5)}")

## 2.3 From a paper's formula to code
The real value of `einsum` shows up with more than two dimensions, where the plain-PyTorch version needs several `permute`, `reshape` and `unsqueeze` calls. Three examples from real models:

**(a) Gram matrix of feature maps** (used in neural style transfer). For features $F$ with dimensions batch $b$, channel $c$, height $h$, width $w$:
$$G_{bcd} = \frac{1}{HW}\sum_{h,w} F_{bchw}\,F_{bdhw}.$$
Letters `h` and `w` are summed; `c` and `d` are two *different* names for the channel dimension of the two copies, so both survive.

**(b) A 1×1 convolution** mixes channels at every pixel with a weight matrix $W_{oc}$: $Y_{bohw} = \sum_c W_{oc} X_{bchw}$.

**(c) Weighted combination of experts.** $E$ experts each produce a $D$-dimensional output for each example, $O_{ned}$, and a gate gives weights $g_{ne}$: $y_{nd} = \sum_e g_{ne} O_{ned}$.

In [ ]:
Fm = torch.randn(2, 8, 5, 5)                          # (b, c, h, w)
G = torch.einsum("bchw,bdhw->bcd", Fm, Fm) / (5 * 5)
G_plain = Fm.flatten(2) @ Fm.flatten(2).transpose(1, 2) / 25       # the plain version: reshape + batched matmul
print("(a) Gram:", tuple(G.shape), torch.allclose(G, G_plain, atol=1e-5))

W = torch.randn(16, 8)                                # (o, c): 8 -> 16 channels
Yc = torch.einsum("oc,bchw->bohw", W, Fm)
conv = F.conv2d(Fm, W[:, :, None, None])               # the same thing as a real 1x1 convolution
print("(b) 1x1 conv:", tuple(Yc.shape), torch.allclose(Yc, conv, atol=1e-5))

O = torch.randn(4, 3, 6)                              # (n, e, d): 4 examples, 3 experts, 6 dims
g = torch.softmax(torch.randn(4, 3), dim=1)           # (n, e) gate weights
y = torch.einsum("ne,ned->nd", g, O)
print("(c) experts:", tuple(y.shape), torch.allclose(y, (g[:, :, None] * O).sum(dim=1), atol=1e-6))

**The `...` ellipsis.** `"...ij,jk->...ik"` means "any number of leading dimensions, passed through unchanged". It lets one expression handle inputs of shape `(i, j)`, `(b, i, j)` or `(b, t, i, j)`.

## 2.4 Is `einsum` fast?
For two operands, PyTorch rewrites the expression into permutes, reshapes and one batched matrix multiplication, so it is as fast as the hand-written version. With **three or more operands**, the order in which pairs are contracted can make an enormous difference; PyTorch picks a good order automatically when the `opt_einsum` package is installed; `torch.backends.opt_einsum.is_available()` tells you whether it is. The expression `"ni,ij,nj->n"` above is such a case.

One caution: an expression that multiplies first and sums later, such as `"ij,ij->ij"` followed by a separate sum, can materialise large intermediates. Put the whole computation in **one** einsum so PyTorch can plan it.

## 2.5 Reading papers' code: `einops` and its plain-PyTorch equivalents
Many repositories use the `einops` library, whose `rearrange` function describes reshapes with the same kind of letters:

```python
from einops import rearrange
x = rearrange(x, "b c h w -> b (h w) c")             # flatten the pixels, channels last
x = rearrange(x, "b (h p1) (w p2) c -> b (h w) (p1 p2 c)", p1=4, p2=4)   # cut into 4x4 patches
```

Parentheses mean "these dimensions are merged into one". When reading such code, translate it into `permute` + `reshape`; when writing code, use whichever your team prefers. The cell checks both translations in plain PyTorch (and with `einops` if it is installed).

In [ ]:
x = torch.randn(2, 3, 8, 8)                                        # (b, c, h, w)

# "b c h w -> b (h w) c"
seq = x.permute(0, 2, 3, 1).reshape(2, 8 * 8, 3)
print("b c h w -> b (h w) c :", tuple(seq.shape))

# "b c (h p1) (w p2) -> b (h w) (p1 p2 c)" with 4x4 patches (input given channels-first here)
p = 4
patches = (x.reshape(2, 3, 8 // p, p, 8 // p, p)                   # b c h p1 w p2
            .permute(0, 2, 4, 3, 5, 1)                             # b h w p1 p2 c
            .reshape(2, (8 // p) * (8 // p), p * p * 3))           # b (h w) (p1 p2 c)
print("patches:", tuple(patches.shape), "-> 4 patches of 4*4*3 = 48 numbers each")

try:
    from einops import rearrange
    print("einops agrees:",
          torch.equal(seq, rearrange(x, "b c h w -> b (h w) c")),
          torch.equal(patches, rearrange(x, "b c (h p1) (w p2) -> b (h w) (p1 p2 c)", p1=p, p2=p)))
except ImportError:
    print("(einops not installed: `pip install einops` to compare)")

## 🧪 Exercises

**Exercise 2.1: three formulas.** `★★☆ · ~20 min`
Implement each with a **single** `torch.einsum` call:
1. `batch_quadratic(x, A)`: for `x` of shape `(N, D)` and `A` of shape `(D, D)`, return $q_n = x_n^\top A\, x_n$, shape `(N,)`.
2. `pairwise_dots(X, Y)`: for `X` of shape `(N, D)` and `Y` of shape `(M, D)`, return $S_{nm} = \sum_d X_{nd} Y_{md}$, shape `(N, M)`.
3. `channel_mix(x, W)`: for `x` of shape `(B, T, C)` and `W` of shape `(C, K)`, return $y_{btk} = \sum_c x_{btc} W_{ck}$.

In [ ]:
# ✍️ Your attempt here.

In [ ]:
# 🔎 Self-check for Exercise 2.1
def _test_q(fn):
    x, A = torch.randn(6, 4), torch.randn(4, 4)
    assert torch.allclose(fn(x, A), torch.stack([xi @ A @ xi for xi in x]), atol=1e-4), "batch_quadratic is wrong"
def _test_p(fn):
    X, Y = torch.randn(5, 3), torch.randn(7, 3)
    assert torch.allclose(fn(X, Y), X @ Y.T, atol=1e-5), "pairwise_dots is wrong"
def _test_c(fn):
    x, W = torch.randn(2, 5, 3), torch.randn(3, 4)
    assert torch.allclose(fn(x, W), x @ W, atol=1e-5), "channel_mix is wrong"
self_check("batch_quadratic", _test_q); self_check("pairwise_dots", _test_p); self_check("channel_mix", _test_c)

In [ ]:
# @title ✅ Solution (double-click to reveal) { display-mode: "form" }
# Solutions only execute when RUN_SOLUTIONS = True (setup cell), so "Run all" stays fast.
if RUN_SOLUTIONS:
    def batch_quadratic(x, A):
        return torch.einsum("ni,ij,nj->n", x, A, x)

    def pairwise_dots(X, Y):
        return torch.einsum("nd,md->nm", X, Y)       # d shared and summed; n and m both kept

    def channel_mix(x, W):
        return torch.einsum("btc,ck->btk", x, W)

    print(batch_quadratic(torch.randn(3, 2), torch.eye(2)), pairwise_dots(torch.eye(2), torch.eye(2)))
else:
    print("Solution not executed (RUN_SOLUTIONS = False).")

**Exercise 2.2: 📖 read this snippet.** `★★☆ · ~15 min`
This is a simplified piece of attention code, as found in many repositories. `q` and `k` have shape `(B, H, T, D)`, where `B` is the batch, `H` the number of "heads", `T` the sequence length and `D` the feature size.

```python
scores = torch.einsum("bhtd,bhsd->bhts", q, k) / math.sqrt(D)
weights = scores.softmax(dim=-1)
```
(a) What is the shape of `scores`, and what does `scores[b, h, t, s]` mean? (b) Which plain-PyTorch expression computes the same thing? (c) Each row `weights[b, h, t, :]` sums to what?

In [ ]:
# ✍️ Your attempt here.

In [ ]:
# @title ✅ Solution (double-click to reveal) { display-mode: "form" }
# Solutions only execute when RUN_SOLUTIONS = True (setup cell), so "Run all" stays fast.
if RUN_SOLUTIONS:
    B, H, Tn, D = 2, 4, 5, 8
    q, k = torch.randn(B, H, Tn, D), torch.randn(B, H, Tn, D)
    scores = torch.einsum("bhtd,bhsd->bhts", q, k) / math.sqrt(D)
    print("(a)", tuple(scores.shape), "- the scaled dot product of query position t with key position s")
    print("(b)", torch.allclose(scores, q @ k.transpose(-2, -1) / math.sqrt(D), atol=1e-5),
          "- q @ k.transpose(-2, -1) / sqrt(D)")
    print("(c)", scores.softmax(dim=-1).sum(dim=-1)[0, 0], "- every row sums to 1 (softmax over s)")
else:
    print("Solution not executed (RUN_SOLUTIONS = False).")

### 📌 Key takeaways
- Shared letters are multiplied; letters missing from the output are summed; the output order follows `->`.
- Two-operand einsums are as fast as `@`; put a whole computation in one einsum so PyTorch can plan it.
- `rearrange("b c h w -> b (h w) c")` = `permute(0, 2, 3, 1).reshape(B, H*W, C)`.

### ✔️ Check yourself

Commit to an answer (say it out loud or write it down) *before* you expand it.

**1.** What does `torch.einsum('bij,bkj->bik', A, B)` compute in plain PyTorch?

<details>
<summary>Answer</summary>

`A @ B.transpose(1, 2)`: for each batch element, the matrix product of `A` with the transpose of `B` (`j` is shared and summed).

</details>

**2.** Why is `'ii->i'` different from `'ii->'`?

<details>
<summary>Answer</summary>

`'ii->i'` keeps the repeated index, giving the diagonal (a vector). `'ii->'` sums it out too, giving the trace (a scalar).

</details>

**3.** Write the einsum for the outer product of every row of `X` (N, D) with itself, giving (N, D, D).

<details>
<summary>Answer</summary>

`torch.einsum('ni,nj->nij', X, X)`.

</details>

---
# Chapter 3. Optimizers and learning-rate schedules

🎯 **In this chapter you will learn to:** see what an optimizer object stores; choose between SGD with momentum, Adam and AdamW; use parameter groups (for example, no weight decay on biases); schedule the learning rate with warm-up and decay, stepping the scheduler at the right time; and find a good learning rate with a range test.

## 3.1 What an optimizer object holds
An optimizer stores two things:
- **`param_groups`**: a list of dictionaries. Each holds a list of parameters plus the hyper-parameters (`lr`, `weight_decay`, `momentum`, …) to use for them;
- **`state`**: per-parameter memory that the update rule needs, such as momentum buffers. It is created lazily, at the first `step()`.

In [ ]:
model = nn.Linear(3, 2)
opt = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=1e-4)
print("number of groups:", len(opt.param_groups))
print("group 0 settings:", {k: v for k, v in opt.param_groups[0].items() if k != "params"})
print("state before the first step:", len(opt.state), "entries")

loss = model(torch.randn(8, 3)).pow(2).mean()
loss.backward()
opt.step()
print("state after one step:", {tuple(p.shape): list(s.keys()) for p, s in opt.state.items()})

The learning rate is just a number in a dictionary. You can change it at any moment, and that is all a **learning-rate scheduler** does:

```python
for group in optimizer.param_groups:
    group["lr"] = 0.01
```

The `state` matters for checkpoints (Chapter 5): if you resume training without restoring it, momentum restarts from zero.

## 3.2 The three optimizers you need
Write $g$ for the gradient of a parameter $p$, and $\eta$ for the learning rate.

**SGD with momentum.** Keep a running sum of past gradients, $v \leftarrow \mu v + g$, and step along it: $p \leftarrow p - \eta v$. With $\mu = 0.9$ the velocity averages about the last 10 gradients, which damps oscillations across steep directions and speeds up progress along shallow ones (remember the ill-conditioned example of Notebook 1, Exercise 2.2).

**Adam.** Keep running averages of the gradient and of its square:
$$m \leftarrow \beta_1 m + (1-\beta_1) g, \qquad v \leftarrow \beta_2 v + (1-\beta_2) g^2,$$
correct their initial bias towards zero ($\hat m = m / (1-\beta_1^t)$, $\hat v = v / (1-\beta_2^t)$ at step $t$), and step $p \leftarrow p - \eta\, \hat m / (\sqrt{\hat v} + \varepsilon)$. Dividing by $\sqrt{\hat v}$ gives every parameter its own step size of roughly $\eta$, whatever the scale of its gradient. Defaults: $\beta_1 = 0.9$, $\beta_2 = 0.999$, $\varepsilon = 10^{-8}$.

**AdamW.** Adam with **decoupled weight decay**: $p \leftarrow p - \eta\,\lambda\,p$ is applied directly to the weights, separately from the Adam step. (Plain `Adam(weight_decay=λ)` instead adds $\lambda p$ to the gradient, so the decay is then divided by $\sqrt{\hat v}$ and weights with large gradients are barely regularised.) **Use AdamW, not Adam, whenever you want weight decay.**

The update rules are short enough to check by hand. Here is one Adam step computed manually and compared with PyTorch:

In [ ]:
torch.manual_seed(0)
p = torch.randn(4, requires_grad=True)
p0 = p.detach().clone()                                   # the starting point
p_manual = p0.clone()
opt = torch.optim.Adam([p], lr=0.1)
(p ** 2).sum().backward()
g = p.grad.clone()                                        # gradient = 2p
opt.step()

b1, b2, eps, lr, t = 0.9, 0.999, 1e-8, 0.1, 1
m = (1 - b1) * g                                          # m and v start at zero
v = (1 - b2) * g ** 2
m_hat, v_hat = m / (1 - b1 ** t), v / (1 - b2 ** t)
p_manual -= lr * m_hat / (v_hat.sqrt() + eps)
print("PyTorch Adam:", p.detach())
print("by hand:     ", p_manual)
print("max difference:", (p.detach() - p_manual).abs().max().item())
print("size of the first step in each coordinate:", (p.detach() - p0).abs())   # all ≈ lr: m_hat/sqrt(v_hat) = ±1

| Optimizer | Typical starting point | Where you see it |
| --- | --- | --- |
| `AdamW` | `lr=1e-3` (small models) to `3e-4` (larger), `weight_decay=0.01–0.05` | The safe default; transformers; most research code |
| `SGD(momentum=0.9, nesterov=True)` | `lr=0.1` for batch size 256, `weight_decay=5e-4` | CNN image classification; often generalises slightly better with a good schedule |
| `Adam` | as AdamW, without weight decay | Older code |

🏭 **Memory cost.** Adam and AdamW store two extra numbers ($m$ and $v$) per parameter; SGD with momentum stores one. With parameters and gradients, training a model with AdamW needs roughly **4×** the memory of its weights, before counting activations. This is why optimizer choice matters for very large models.

## 3.3 Parameter groups
Pass a **list of dictionaries** instead of `model.parameters()` to treat parameters differently. The two most common uses:

1. **No weight decay on biases and normalisation parameters.** Weight decay pulls weights towards zero to limit model complexity. Biases and BatchNorm/LayerNorm scales do not contribute to that complexity in the same way, and decaying them can hurt. Conveniently, they are exactly the 1-D parameters.
2. **Different learning rates for different parts** of a model, for example a small one for a pretrained backbone and a large one for a new head (Chapter 8).

In [ ]:
from torch.optim import lr_scheduler as S

def param_groups_weight_decay(model, weight_decay):
    decay, no_decay = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue                                     # frozen parameters: leave them out
        (no_decay if p.ndim <= 1 else decay).append(p)   # biases and norm weights are 1-D
    return [{"params": decay, "weight_decay": weight_decay},
            {"params": no_decay, "weight_decay": 0.0}]

In [ ]:
demo = build_mlp(batchnorm=True)
groups = param_groups_weight_decay(demo, 0.05)
opt = torch.optim.AdamW(groups, lr=1e-3)
for i, g in enumerate(opt.param_groups):
    print(f"group {i}: {sum(p.numel() for p in g['params']):>7,} values in {len(g['params'])} tensors, "
          f"weight_decay={g['weight_decay']}, lr={g['lr']}")

Settings given in the dictionaries override the defaults passed to the constructor; everything else (here `lr`) comes from the defaults.

## 3.4 Learning-rate schedules
A constant learning rate is a compromise. Early in training you want large steps to make fast progress; late in training you want small steps to settle into a good minimum instead of bouncing around it. Many modern recipes also **warm up**: they start with a tiny learning rate and increase it over the first few hundred or thousand steps, because large steps on randomly initialised weights (and Adam's noisy early estimates of $v$) can destabilise training.

PyTorch's schedulers live in `torch.optim.lr_scheduler`. Each wraps an optimizer and, every time you call `scheduler.step()`, writes a new learning rate into `optimizer.param_groups`. The helper below records the learning rate over 100 calls, so we can plot the common schedules:

In [ ]:
from torch.optim import lr_scheduler as S

def lr_trace(make_scheduler, steps=100, base_lr=1.0):
    opt = torch.optim.SGD([torch.zeros(1, requires_grad=True)], lr=base_lr)
    sched = make_scheduler(opt)
    lrs = []
    for _ in range(steps):
        lrs.append(opt.param_groups[0]["lr"])   # the lr used for this step
        opt.step()                              # (no gradients here, so nothing changes)
        sched.step()                            # AFTER optimizer.step()
    return lrs

schedules = {
    "StepLR(step_size=30, gamma=0.1)": lambda o: S.StepLR(o, step_size=30, gamma=0.1),
    "ExponentialLR(gamma=0.97)": lambda o: S.ExponentialLR(o, gamma=0.97),
    "CosineAnnealingLR(T_max=100)": lambda o: S.CosineAnnealingLR(o, T_max=100),
    "warm-up 10 + cosine 90 (SequentialLR)": lambda o: S.SequentialLR(
        o, [S.LinearLR(o, start_factor=0.01, total_iters=10), S.CosineAnnealingLR(o, T_max=90)], milestones=[10]),
    "OneCycleLR(max_lr=1, total_steps=100)": lambda o: S.OneCycleLR(o, max_lr=1.0, total_steps=100),
}
fig, axes = plt.subplots(1, 5, figsize=(17, 2.8), sharey=True)
for ax, (name, make) in zip(axes, schedules.items()):
    ax.plot(lr_trace(make)); ax.set_title(name, fontsize=8); ax.set_xlabel("scheduler step")
axes[0].set_ylabel("learning rate (x base)"); plt.tight_layout(); plt.show()

| Schedule | Idea | Typical use |
| --- | --- | --- |
| `StepLR` / `MultiStepLR` | Divide by 10 at fixed epochs | Classic CNN papers ("lr 0.1, divided by 10 at epochs 30, 60, 90") |
| `CosineAnnealingLR` | Smooth decay from the base value to (almost) zero along half a cosine | Modern default for a fixed training length |
| Warm-up + cosine | Linear increase, then cosine decay | Transformers, large batches, AdamW recipes |
| `OneCycleLR` | Warm up to `max_lr`, then anneal far below the start; also cycles momentum | Fast training of small and medium models; called **per batch** |
| `ReduceLROnPlateau` | Divide the lr when a validation metric stops improving | When you do not know the training length in advance |
| `LambdaLR(fn)` | Any formula: lr = base × `fn(step)` | Paper-specific schedules; very common in research code |

### Three rules that prevent most scheduler bugs
1. **Call `scheduler.step()` after `optimizer.step()`.** (PyTorch warns if you swap them, because the first value of the schedule would be skipped.)
2. **Decide whether the schedule counts batches or epochs, and size it in the same unit.** `OneCycleLR(total_steps=epochs * len(train_loader))` is stepped once per batch; `CosineAnnealingLR(T_max=epochs)` is stepped once per epoch. Mixing the two is a classic silent bug, shown below.
3. **`ReduceLROnPlateau` is stepped with the monitored value**: `scheduler.step(val_loss)`, once per epoch.

🤔 **Predict** what happens if a cosine schedule sized for 10 *epochs* is stepped after every *batch* (50 batches per epoch).

In [ ]:
wrong = lr_trace(lambda o: S.CosineAnnealingLR(o, T_max=10), steps=10 * 50)
right = lr_trace(lambda o: S.CosineAnnealingLR(o, T_max=10 * 50), steps=10 * 50)
plt.figure(figsize=(8, 2.8))
plt.plot(wrong, label="T_max=10 (epochs) but stepped every batch: BUG")
plt.plot(right, label="T_max=500 (batches), stepped every batch")
plt.xlabel("batch"); plt.ylabel("lr"); plt.legend(fontsize=8); plt.show()

The cosine formula is periodic, so the buggy schedule rises and falls 25 times. Training "works", just badly, and nothing raises an error. When you use a scheduler, **plot or log the learning rate**.

## 3.5 Finding a learning rate: the range test
The learning rate is the most important hyper-parameter. The **LR range test** finds a sensible value in about a minute: train for a few hundred batches while increasing the learning rate exponentially from tiny to huge, and record the loss. The loss first barely moves (lr too small), then falls quickly, then explodes (lr too large). A good maximum learning rate lies a little before the minimum of the curve, typically about a tenth of the learning rate at which the loss is lowest.

In [ ]:
def lr_range_test(model, loader, make_optimizer, lr_min=1e-6, lr_max=1.0, steps=200, loss_fn=nn.CrossEntropyLoss()):
    model = copy.deepcopy(model).to(device)        # work on a copy: the test wrecks the weights
    opt = make_optimizer(model.parameters(), lr_min)
    factor = (lr_max / lr_min) ** (1 / (steps - 1))
    lrs, losses, smooth, best = [], [], None, float("inf")
    it = iter(loader)
    model.train()
    for i in range(steps):
        try:
            x, y = next(it)
        except StopIteration:
            it = iter(loader); x, y = next(it)
        x, y = x.to(device), y.to(device)
        loss = loss_fn(model(x), y)
        opt.zero_grad(); loss.backward(); opt.step()
        smooth = loss.item() if smooth is None else 0.9 * smooth + 0.1 * loss.item()   # smooth the noisy loss
        lrs.append(opt.param_groups[0]["lr"]); losses.append(smooth)
        best = min(best, smooth)
        if smooth > 4 * best:                      # the loss has exploded: stop
            break
        for g in opt.param_groups:
            g["lr"] *= factor
    return lrs, losses

set_seed(0)
lrs, losses = lr_range_test(build_mlp(), train_loader, lambda p, lr: torch.optim.AdamW(p, lr=lr),
                            steps=20 if FAST_DEV_RUN else 200)
best_lr = lrs[int(np.argmin(losses))]
plt.figure(figsize=(6, 3.2))
plt.semilogx(lrs, losses); plt.axvline(best_lr / 10, color="tab:red", ls="--", label=f"suggested ≈ {best_lr / 10:.0e}")
plt.xlabel("learning rate (log scale)"); plt.ylabel("smoothed training loss"); plt.legend(); plt.title("LR range test, AdamW")
plt.show()
print(f"lowest loss at lr ≈ {best_lr:.1e}; a good maximum lr is about {best_lr / 10:.1e}")

For this MLP the suggestion usually lands within a factor of a few of 1e-3, which is one reason that value is a common AdamW default. The test is a guide, not an oracle: use it to pick the right **order of magnitude**, then confirm with a short real run.

## 🧪 Exercises

**Exercise 3.1: warm-up plus cosine as one formula.** `★★☆ · ~20 min`
Research code often implements warm-up + cosine with a single `LambdaLR`. Write `warmup_cosine(step, warmup_steps, total_steps)` that returns the **multiplier** of the base learning rate:
- during warm-up (`step < warmup_steps`): rises linearly, `(step + 1) / warmup_steps`;
- afterwards: cosine decay from 1 to 0, `0.5 * (1 + cos(π · progress))` with `progress = (step − warmup_steps) / (total_steps − warmup_steps)`.

Then plot it with `lr_trace(lambda o: S.LambdaLR(o, lambda s: warmup_cosine(s, 10, 100)))`.

In [ ]:
# ✍️ Your attempt here.

In [ ]:
# 🔎 Self-check for Exercise 3.1
def _test(fn):
    assert abs(fn(0, 10, 100) - 0.1) < 1e-9, "step 0 should give 1/warmup_steps"
    assert abs(fn(9, 10, 100) - 1.0) < 1e-9, "the last warm-up step should give 1.0"
    assert abs(fn(10, 10, 100) - 1.0) < 1e-9, "decay should start at 1.0"
    assert abs(fn(55, 10, 100) - 0.5) < 1e-9, "halfway through the decay should give 0.5"
    assert fn(99, 10, 100) < 0.01, "near the end the multiplier should be close to 0"
self_check("warmup_cosine", _test)

In [ ]:
# @title ✅ Solution (double-click to reveal) { display-mode: "form" }
# Solutions only execute when RUN_SOLUTIONS = True (setup cell), so "Run all" stays fast.
if RUN_SOLUTIONS:
    def warmup_cosine(step, warmup_steps, total_steps):
        if step < warmup_steps:
            return (step + 1) / warmup_steps
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.5 * (1 + math.cos(math.pi * progress))

    plt.figure(figsize=(5, 2.5))
    plt.plot(lr_trace(lambda o: S.LambdaLR(o, lambda s: warmup_cosine(s, 10, 100))))
    plt.title("LambdaLR(warmup_cosine)"); plt.xlabel("step"); plt.show()
else:
    print("Solution not executed (RUN_SOLUTIONS = False).")

**Exercise 3.2: how much memory does the optimizer need?** `★☆☆ · ~10 min`
For `build_mlp()`, run one training step with `AdamW` and then count the numbers stored in `optimizer.state` (sum the `numel()` of every tensor in every state dictionary). Compare with the number of parameters. Then do the same for `SGD(momentum=0.9)`.

In [ ]:
# ✍️ Your attempt here.

In [ ]:
# @title ✅ Solution (double-click to reveal) { display-mode: "form" }
# Solutions only execute when RUN_SOLUTIONS = True (setup cell), so "Run all" stays fast.
if RUN_SOLUTIONS:
    for name, make in {"AdamW": lambda p: torch.optim.AdamW(p, lr=1e-3),
                       "SGD+momentum": lambda p: torch.optim.SGD(p, lr=0.1, momentum=0.9)}.items():
        m = build_mlp()
        opt = make(m.parameters())
        F.cross_entropy(m(torch.randn(4, 1, 28, 28)), torch.tensor([0, 1, 2, 3])).backward()
        opt.step()
        stored = sum(t.numel() for s in opt.state.values() for t in s.values() if torch.is_tensor(t))
        print(f"{name:13s} parameters {count_params(m):,} | optimizer state {stored:,} numbers "
              f"({stored / count_params(m):.2f} per parameter)")
    # AdamW stores exp_avg and exp_avg_sq (2 per parameter) plus a tiny 'step' counter per tensor;
    # SGD with momentum stores one momentum buffer per parameter.
else:
    print("Solution not executed (RUN_SOLUTIONS = False).")

### 📌 Key takeaways
- An optimizer = `param_groups` (settings per group of parameters) + `state` (momentum and similar). Schedulers only edit `param_groups[i]["lr"]`.
- Default to AdamW. Exclude biases and normalisation parameters from weight decay with parameter groups.
- Step the scheduler after the optimizer, in the unit (batch or epoch) the schedule was sized for, and log the learning rate.
- Find the order of magnitude of the learning rate with a range test.

### ✔️ Check yourself

Commit to an answer (say it out loud or write it down) *before* you expand it.

**1.** Why is `Adam(weight_decay=λ)` not the same as `AdamW(weight_decay=λ)`?

<details>
<summary>Answer</summary>

Adam adds `λp` to the gradient, so the decay is divided by `√v̂` like the rest of the step, and parameters with large gradients are barely decayed. AdamW applies `p ← p − ηλp` separately from the adaptive step, so every weight decays at the same relative rate.

</details>

**2.** You train for 20 epochs with 400 batches each and want OneCycleLR. What `total_steps` do you pass, and where do you call `scheduler.step()`?

<details>
<summary>Answer</summary>

`total_steps = 20 * 400 = 8000`, and `scheduler.step()` after every `optimizer.step()`, inside the batch loop.

</details>

**3.** Why might training diverge in the first few hundred steps without warm-up, yet be fine with it?

<details>
<summary>Answer</summary>

At the start the weights are random and the gradient statistics (Adam's `v`) are poorly estimated, so a full-size learning rate can produce huge, badly scaled updates. Warm-up keeps steps small until the statistics settle and the loss surface becomes better behaved.

</details>

---
# Chapter 4. Training control: clipping, accumulation, early stopping and a reusable `fit()`

🎯 **In this chapter you will learn to:** clip gradients and monitor their norm; simulate large batches with gradient accumulation; stop training when validation stops improving; assemble everything into one reusable `fit()`; and compare training recipes with a **controlled** experiment instead of an anecdote.

🔁 **Starting a fresh session here?** Run the two Chapter 0 cells, then the hidden restore-point cell below. It re-creates the tools from earlier chapters that Chapter 4 uses. In a normal top-to-bottom run it does nothing.

In [ ]:
# @title 🔁 Restore point for Chapter 4 (needed only if you start a fresh session here) { display-mode: "form" }
if "device" not in globals():
    raise RuntimeError("Run the Chapter 0 setup cells first, then this cell.")
if "param_groups_weight_decay" not in globals():
    from torch.optim import lr_scheduler as S

    def param_groups_weight_decay(model, weight_decay):
        decay, no_decay = [], []
        for name, p in model.named_parameters():
            if not p.requires_grad:
                continue                                     # frozen parameters: leave them out
            (no_decay if p.ndim <= 1 else decay).append(p)   # biases and norm weights are 1-D
        return [{"params": decay, "weight_decay": weight_decay},
                {"params": no_decay, "weight_decay": 0.0}]
    print("Restored the tools defined in earlier chapters.")

## 4.1 Gradient clipping
Occasionally one batch produces a huge gradient: an outlier, an unlucky initialisation, or a steep "cliff" in the loss surface. One such step can undo hours of training. **Gradient-norm clipping** rescales the gradient whenever its overall length exceeds a threshold:

$$\|g\| = \sqrt{\textstyle\sum_{\text{all parameters}} \|g_p\|^2}, \qquad g \leftarrow g \cdot \min\!\left(1, \frac{\text{max\_norm}}{\|g\|}\right).$$

All gradients are scaled by the same factor, so the **direction** of the step is unchanged, only its length is capped. It goes between `backward()` and `step()`:

```python
loss.backward()
total_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)   # returns the norm BEFORE clipping
optimizer.step()
```

The returned norm is a valuable signal to log: a gradient norm that suddenly jumps by orders of magnitude is often the first sign of trouble.

In [ ]:
set_seed(0)
m = build_mlp()
x, y = next(iter(train_loader))
F.cross_entropy(m(x), y).backward()

norm_by_hand = torch.sqrt(sum((p.grad ** 2).sum() for p in m.parameters()))
before = [p.grad.clone() for p in m.parameters()]
total = nn.utils.clip_grad_norm_(m.parameters(), max_norm=0.1)
after_norm = torch.sqrt(sum((p.grad ** 2).sum() for p in m.parameters()))
print(f"global gradient norm: by hand {norm_by_hand:.4f}, returned by clip_grad_norm_ {total:.4f}")
print(f"norm after clipping to 0.1: {after_norm:.4f}")
ratios = torch.cat([(a / b).flatten() for a, b in zip([p.grad for p in m.parameters()], before) if b.abs().sum() > 0])
ratios = ratios[torch.isfinite(ratios)]
print(f"every gradient entry scaled by the same factor: {ratios.min():.4f} ... {ratios.max():.4f}")

Typical values of `max_norm` are 0.5–5. Clipping is standard for transformers and recurrent networks, and a cheap safety net elsewhere. (`clip_grad_value_` instead clips each entry separately, which changes the direction; it is rarely what you want.)

## 4.2 Gradient accumulation: large batches on a small GPU
Suppose a recipe needs batch size 256 but only 64 examples fit in GPU memory. Because gradients **add up** across `backward()` calls (Notebook 1, §2.3), you can process 4 micro-batches of 64, calling `backward()` on each, and step once. Divide each micro-batch loss by 4 so that the accumulated gradient is the gradient of the **mean** over all 256 examples:

```python
optimizer.zero_grad()
for i, (x, y) in enumerate(loader):
    loss = loss_fn(model(x), y) / accum_steps
    loss.backward()                          # gradients accumulate
    if (i + 1) % accum_steps == 0:
        optimizer.step()
        optimizer.zero_grad()
```

Let us verify the claim that this gives exactly the large-batch gradient:

In [ ]:
set_seed(0)
m = build_mlp(p_drop=0.0)                     # no dropout, so both runs see the same network
xb, yb = next(iter(train_loader))             # one batch of 128 examples

m.zero_grad()
F.cross_entropy(m(xb), yb).backward()         # (a) one big batch
big = [p.grad.clone() for p in m.parameters()]

m.zero_grad()
for xs, ys in zip(xb.chunk(4), yb.chunk(4)):  # (b) 4 micro-batches of 32, loss divided by 4
    (F.cross_entropy(m(xs), ys) / 4).backward()
acc = [p.grad for p in m.parameters()]
print("accumulated gradient equals the big-batch gradient:",
      all(torch.allclose(a, b, atol=1e-6) for a, b in zip(acc, big)))

⚠️ The equivalence is exact only for layers that treat examples independently. **BatchNorm** computes statistics per micro-batch, so with accumulation it sees batches of 32, not 128 (Exercise 4.2).

## 4.3 Early stopping
In Notebook 1 the validation loss bottomed out after a few epochs and then crept up while the training loss kept falling. **Early stopping** watches a validation metric and stops when it has not improved for `patience` epochs, then restores the weights from the best epoch. It saves compute and acts as a regulariser.

In [ ]:
class EarlyStopping:
    # Stop when `metric` has not improved by more than `min_delta` for `patience` epochs.
    # mode="min" for losses, mode="max" for accuracies. Keeps a copy of the best weights.
    def __init__(self, patience=5, min_delta=0.0, mode="min"):
        self.patience, self.min_delta, self.mode = patience, min_delta, mode
        self.best = math.inf if mode == "min" else -math.inf
        self.best_state, self.best_epoch, self.bad_epochs, self.epoch = None, 0, 0, 0

    def improved(self, value):
        if self.mode == "min":
            return value < self.best - self.min_delta
        return value > self.best + self.min_delta

    def step(self, value, model):
        # Call once per epoch. Returns True when training should stop.
        self.epoch += 1
        if self.improved(value):
            self.best, self.best_epoch, self.bad_epochs = value, self.epoch, 0
            self.best_state = copy.deepcopy(model.state_dict())
        else:
            self.bad_epochs += 1
        return self.bad_epochs >= self.patience

    def restore(self, model):
        model.load_state_dict(self.best_state)

In [ ]:
# A quick test with a made-up sequence of validation losses
es = EarlyStopping(patience=3)
for epoch, val_loss in enumerate([1.0, 0.8, 0.7, 0.72, 0.69, 0.71, 0.70, 0.73, 0.75], start=1):
    stop = es.step(val_loss, nn.Linear(1, 1))
    print(f"epoch {epoch}: val loss {val_loss:.2f} | best {es.best:.2f} (epoch {es.best_epoch}) | "
          f"epochs without improvement {es.bad_epochs}{'  -> STOP' if stop else ''}")
    if stop:
        break

Practical advice:
- **Monitor the validation loss** rather than accuracy when you can: it is smoother and reacts earlier. Accuracy is fine when the loss is distorted (for example by label smoothing).
- **Patience must exceed the natural noise** of the metric, and with `ReduceLROnPlateau` it must exceed the scheduler's patience, or you stop exactly when the learning rate is about to drop.
- With a schedule that decays to zero (cosine, OneCycle), the last epochs are usually the best and early stopping rarely triggers. It then simply acts as "keep the best checkpoint".

## 4.4 A reusable `fit()`
Here is the training machinery the rest of this notebook uses. It extends Notebook 1's version with a scheduler (per batch, per epoch, or on plateau), gradient clipping with norm logging, early stopping, a learning-rate log, and a hook for weight averaging (Chapter 6). Three details are new:

- **Running sums stay on the device.** Notebook 1 called `.item()` on every batch. Each `.item()` makes the CPU wait until the GPU has finished all queued work, which throws away the asynchronous execution you met in Notebook 1, §1.8. Here the loss and the number of correct predictions are summed as tensors on the device and converted **once per epoch**. Chapter 11 measures the difference.
- **Mixed-precision arguments** (`amp_dtype`, `scaler`) are already built in, switched off by default. Chapter 11 explains them; until then, ignore them.
- **`fit` reports the model it keeps.** With early stopping it restores the epoch with the best validation **loss**, and `history["selected"]` holds that epoch's metrics. Report those numbers, not `max(history["val_acc"])`: the maximum may come from a different epoch than the model you keep, and the maximum of noisy values is optimistic.

Read the code once carefully; every option is something you have just learned.

In [ ]:
import contextlib

def train_one_epoch(model, loader, loss_fn, optimizer, scheduler=None, max_grad_norm=None,
                    averaged_model=None, amp_dtype=None, scaler=None):
    # amp_dtype / scaler: mixed precision, explained in Chapter 11. Leave them as None until then.
    model.train()
    total_loss = torch.zeros((), device=device)                  # running sums stay ON the device:
    correct = torch.zeros((), device=device, dtype=torch.long)   # no CPU-GPU synchronisation per step
    n, grad_norms = 0, []
    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        amp = torch.autocast(device_type=device.type, dtype=amp_dtype) if amp_dtype else contextlib.nullcontext()
        with amp:
            logits = model(x)
            loss = loss_fn(logits, y)
        optimizer.zero_grad(set_to_none=True)
        if scaler is not None:                                   # float16 mixed precision (Chapter 11)
            scaler.scale(loss).backward()
            if max_grad_norm is not None:
                scaler.unscale_(optimizer)                       # clip the REAL gradients, not the scaled ones
        else:
            loss.backward()
        if max_grad_norm is not None:
            grad_norms.append(nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm))
        if scaler is not None:
            scaler.step(optimizer)
            scaler.update()
        else:
            optimizer.step()
        if scheduler is not None:                                # per-batch schedules (OneCycleLR, warm-up)
            scheduler.step()
        if averaged_model is not None:                           # weight averaging (Chapter 6)
            averaged_model.update_parameters(model)
        total_loss += loss.detach().float() * len(y)
        correct += (logits.argmax(dim=1) == y).sum()
        n += len(y)
    mean_norm = torch.stack(grad_norms).mean().item() if grad_norms else float("nan")
    return total_loss.item() / n, correct.item() / n, mean_norm   # one synchronisation per epoch


def fit(model, train_loader, val_loader, optimizer, epochs, loss_fn=nn.CrossEntropyLoss(),
        scheduler=None, scheduler_interval="batch", early_stopping=None, eval_model=None, verbose=True, **train_kw):
    # scheduler_interval: "batch", "epoch" or "plateau" (ReduceLROnPlateau).
    # eval_model: evaluate (and select) a different model than the one trained, e.g. an EMA copy (Chapter 6).
    history = {k: [] for k in ["train_loss", "train_acc", "val_loss", "val_acc", "lr", "grad_norm", "time"]}
    eval_model = eval_model if eval_model is not None else model
    for epoch in range(1, epochs + 1):
        t0 = time.perf_counter()
        lr = optimizer.param_groups[0]["lr"]
        tr_loss, tr_acc, gnorm = train_one_epoch(model, train_loader, loss_fn, optimizer,
                                                 scheduler if scheduler_interval == "batch" else None, **train_kw)
        va_loss, va_acc = evaluate(eval_model, val_loader, loss_fn)
        if scheduler is not None and scheduler_interval == "epoch":
            scheduler.step()
        elif scheduler is not None and scheduler_interval == "plateau":
            scheduler.step(va_loss)
        elapsed = time.perf_counter() - t0
        for k, v in zip(history, (tr_loss, tr_acc, va_loss, va_acc, lr, gnorm, elapsed)):
            history[k].append(v)
        if verbose:
            print(f"epoch {epoch:2d} | lr {lr:.2e} | train {tr_loss:.4f} / {tr_acc:.4f} | "
                  f"val {va_loss:.4f} / {va_acc:.4f} | {elapsed:4.1f}s")
        if early_stopping is not None and early_stopping.step(va_loss, eval_model):
            print(f"early stopping: no improvement for {early_stopping.patience} epochs")
            break
    if early_stopping is not None:                       # select the epoch with the best validation LOSS...
        early_stopping.restore(eval_model)
        sel_epoch = early_stopping.best_epoch
    else:
        sel_epoch = len(history["val_loss"])            # ...or keep the last epoch
    sel_loss, sel_acc = history["val_loss"][sel_epoch - 1], history["val_acc"][sel_epoch - 1]
    history["selected"] = {"epoch": sel_epoch, "val_loss": sel_loss, "val_acc": sel_acc}
    if verbose:
        print(f"selected model: epoch {sel_epoch}, val loss {sel_loss:.4f}, val acc {sel_acc:.4f}")
    return history


def plot_history(histories, keys=("val_loss", "val_acc", "lr")):
    fig, axes = plt.subplots(1, len(keys), figsize=(5 * len(keys), 3.2))
    for ax, key in zip(axes, keys):
        for name, h in histories.items():
            ax.plot(range(1, len(h[key]) + 1), h[key], "o-", ms=3, label=name)
        ax.set_title(key); ax.set_xlabel("epoch")
    axes[0].legend(fontsize=8); plt.tight_layout(); plt.show()

### Experiment: does the learning-rate schedule matter? A controlled comparison
A common mistake when comparing training recipes is to change several things at once and then credit the result to one of them. Below, the two configurations are **identical** (same model, optimizer, peak learning rate, weight-decay groups, gradient clipping, number of epochs and selection rule) **except for the schedule**:
- **(a) constant:** learning rate 1e-3 throughout;
- **(b) OneCycle:** warm-up to the same peak of 1e-3, then annealing towards zero.

One run is one noisy sample, so each configuration is trained with **3 seeds** and we compare means relative to the seed-to-seed spread (Chapter 5 returns to this). Each epoch takes a few seconds on a T4, so the whole experiment takes a few minutes.

In [ ]:
EPOCHS = n_epochs(20)
SEEDS = [0] if FAST_DEV_RUN else [0, 1, 2]

def run_config(schedule, seed, early_stopping=None):
    set_seed(seed)
    model = build_mlp().to(device)
    opt = torch.optim.AdamW(param_groups_weight_decay(model, 0.05), lr=1e-3)
    sched = S.OneCycleLR(opt, max_lr=1e-3, total_steps=EPOCHS * len(train_loader)) if schedule == "onecycle" else None
    return fit(model, train_loader, val_loader, opt, EPOCHS, scheduler=sched, max_grad_norm=1.0,
               early_stopping=early_stopping or EarlyStopping(patience=100), verbose=False)   # patience=100: keep the best

runs, summary = {}, {}
for schedule in ("constant", "onecycle"):
    accs = []
    for seed in SEEDS:
        h = run_config(schedule, seed)
        accs.append(h["selected"]["val_acc"])
        print(f"{schedule:9s} seed {seed}: selected epoch {h['selected']['epoch']:2d} | val acc {h['selected']['val_acc']:.4f}")
        runs.setdefault(schedule, h)                  # keep the first seed's history for plotting
    summary[schedule] = torch.tensor(accs)
    record(f"Ch4 {schedule} schedule, val acc (mean over seeds)", summary[schedule].mean().item())

plot_history(runs)
for name, a in summary.items():
    spread = f"± {a.std():.4f}" if len(a) > 1 else "(1 seed: no spread)"
    print(f"{name:9s} mean {a.mean():.4f} {spread}")
print(f"difference of means (onecycle - constant): {(summary['onecycle'].mean() - summary['constant'].mean()):+.4f}")

**How to read the result.** If the difference of the means is clearly larger than the seed-to-seed standard deviations, the schedule made a difference; if not, this experiment cannot tell the two apart. Typically the OneCycle runs end slightly higher: the annealed learning rate lets the weights settle, while the constant-rate runs reach their best epoch earlier and then their validation loss creeps up (see the plot). Your exact numbers will differ; the method is the lesson.

Notice what we may **not** conclude: anything about weight decay or clipping, because they were held fixed. And because the schedule was chosen by its *validation* score, the final, unbiased number still has to come from the test set, once, for the configuration you pick.

## 4.5 ★ Logging to TensorBoard
Printing epoch lines works for one run; with dozens of runs you want a dashboard. **TensorBoard** reads log files written with a `SummaryWriter`: `add_scalar(tag, value, step)` records one number, and runs written to different folders appear as different curves. In a real project you log from inside the training loop (every few hundred steps); here we write the histories we already have.

In [ ]:
from torch.utils.tensorboard import SummaryWriter

for name, h in runs.items():
    writer = SummaryWriter(log_dir=f"runs/ch4_{name}")
    for epoch, (vl, va, lr) in enumerate(zip(h["val_loss"], h["val_acc"], h["lr"]), start=1):
        writer.add_scalar("val/loss", vl, epoch)
        writer.add_scalar("val/accuracy", va, epoch)
        writer.add_scalar("learning_rate", lr, epoch)
    writer.close()
print("logs written to ./runs; the next cell opens the dashboard inside the notebook")

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs

Weights & Biases and MLflow are hosted alternatives with the same idea, and they also store each run's configuration, which makes comparing many runs much easier.

## 🧪 Exercises

**Exercise 4.1: clipping by hand.** `★★☆ · ~15 min`
Write `clip_global_norm_(params, max_norm)` that does what `nn.utils.clip_grad_norm_` does: compute the global L2 norm of all `.grad` tensors, scale them in place if the norm exceeds `max_norm`, and return the norm before clipping (as a tensor). Skip parameters whose `.grad` is `None`.

In [ ]:
# ✍️ Your attempt here.
# Hint: Compute the norm inside torch.no_grad(); scale with g.mul_(factor).

In [ ]:
# 🔎 Self-check for Exercise 4.1
def _test(fn):
    set_seed(1)
    a, b = build_mlp(p_drop=0.0), build_mlp(p_drop=0.0)
    b.load_state_dict(a.state_dict())
    x, y = torch.randn(16, 1, 28, 28), torch.randint(0, 10, (16,))
    for m in (a, b):
        (F.cross_entropy(m(x), y) * 50).backward()
    n1 = fn(list(a.parameters()), 0.5)
    n2 = nn.utils.clip_grad_norm_(b.parameters(), 0.5)
    assert torch.allclose(torch.as_tensor(n1), n2, rtol=1e-5), "the returned norm is wrong"
    assert all(torch.allclose(p.grad, q.grad, atol=1e-7) for p, q in zip(a.parameters(), b.parameters())), \
        "the clipped gradients differ from clip_grad_norm_"
self_check("clip_global_norm_", _test)

In [ ]:
# @title ✅ Solution (double-click to reveal) { display-mode: "form" }
# Solutions only execute when RUN_SOLUTIONS = True (setup cell), so "Run all" stays fast.
if RUN_SOLUTIONS:
    @torch.no_grad()
    def clip_global_norm_(params, max_norm):
        grads = [p.grad for p in params if p.grad is not None]
        total = torch.sqrt(sum((g ** 2).sum() for g in grads))
        factor = max_norm / (total + 1e-6)              # PyTorch adds the same small epsilon
        if factor < 1:
            for g in grads:
                g.mul_(factor)
        return total
else:
    print("Solution not executed (RUN_SOLUTIONS = False).")

**Exercise 4.2: accumulation and BatchNorm.** `★★☆ · ~15 min`
Repeat the verification of §4.2 with `build_mlp(p_drop=0.0, batchnorm=True)`. Are the gradients still equal? Explain why, and name one way to make accumulation and normalisation compatible.

In [ ]:
# ✍️ Your attempt here.

In [ ]:
# @title ✅ Solution (double-click to reveal) { display-mode: "form" }
# Solutions only execute when RUN_SOLUTIONS = True (setup cell), so "Run all" stays fast.
if RUN_SOLUTIONS:
    set_seed(0)
    m = build_mlp(p_drop=0.0, batchnorm=True)
    xb, yb = next(iter(train_loader))
    m.zero_grad(); F.cross_entropy(m(xb), yb).backward()
    big = [p.grad.clone() for p in m.parameters()]
    m.zero_grad()
    for xs, ys in zip(xb.chunk(4), yb.chunk(4)):
        (F.cross_entropy(m(xs), ys) / 4).backward()
    diff = max((p.grad - b).abs().max().item() for p, b in zip(m.parameters(), big))
    print("max gradient difference with BatchNorm:", diff)
    # Not equal: in training mode BatchNorm normalises each micro-batch of 32 with ITS OWN mean and
    # variance, so the forward pass itself differs from the one on 128 examples.
    # Options: accept it (often fine when micro-batches are not tiny); use a per-example normalisation
    # such as LayerNorm or GroupNorm, which does not depend on the batch; or with several GPUs use
    # SyncBatchNorm, which computes statistics across devices.
else:
    print("Solution not executed (RUN_SOLUTIONS = False).")

**Exercise 4.3: ReduceLROnPlateau.** `★☆☆ · ~15 min`
Train `build_mlp()` with `AdamW(lr=1e-3)` and `S.ReduceLROnPlateau(opt, factor=0.3, patience=1)` for 12 epochs, using `fit(..., scheduler_interval="plateau")`. Plot the history together with the runs in `runs` and print the selected epoch's validation accuracy. When did the learning rate drop, and what happened to the validation loss afterwards?

In [ ]:
# ✍️ Your attempt here.

In [ ]:
# @title ✅ Solution (double-click to reveal) { display-mode: "form" }
# Solutions only execute when RUN_SOLUTIONS = True (setup cell), so "Run all" stays fast.
if RUN_SOLUTIONS:
    set_seed(0)
    model_c = build_mlp().to(device)
    opt_c = torch.optim.AdamW(model_c.parameters(), lr=1e-3)
    runs["plateau"] = fit(model_c, train_loader, val_loader, opt_c, n_epochs(12),
                          scheduler=S.ReduceLROnPlateau(opt_c, factor=0.3, patience=1), scheduler_interval="plateau")
    plot_history(runs)
    print("selected:", runs["plateau"]["selected"])
    # The lr drops by 0.3x the epoch after the validation loss fails to improve twice in a row.
    # Each drop is usually followed by a visible dip in the validation loss: the smaller steps let
    # the model settle into the minimum it was bouncing around.
else:
    print("Solution not executed (RUN_SOLUTIONS = False).")

**Exercise 4.4: what does early stopping save?** `★☆☆ · ~10 min`
Rerun configuration (a), seed 0, with `run_config("constant", 0, EarlyStopping(patience=4))`. Compare the number of epochs it trained and its selected validation accuracy with the seed-0 run in `runs["constant"]`.

In [ ]:
# ✍️ Your attempt here.

In [ ]:
# @title ✅ Solution (double-click to reveal) { display-mode: "form" }
# Solutions only execute when RUN_SOLUTIONS = True (setup cell), so "Run all" stays fast.
if RUN_SOLUTIONS:
    h_es = run_config("constant", 0, EarlyStopping(patience=4))
    print(f"with early stopping: {len(h_es['val_loss'])} epochs, selected {h_es['selected']}")
    print(f"without:             {len(runs['constant']['val_loss'])} epochs, selected {runs['constant']['selected']}")
    # Early stopping usually selects the same (or a nearby) epoch, because both runs pick the best
    # validation loss, but it stops a few epochs after that point instead of running all 20: same
    # model, less compute. With a decaying schedule (OneCycle) stopping early would cut the schedule
    # short, which is why it is rarely combined with one.
else:
    print("Solution not executed (RUN_SOLUTIONS = False).")

### 📌 Key takeaways
- Change **one** thing at a time, repeat over seeds, and report the metrics of the model you actually keep.
- `clip_grad_norm_` goes between `backward()` and `step()`; log the norm it returns.
- Gradient accumulation = several `backward()` calls with the loss divided by the number of micro-batches, then one `step()`.
- Early stopping monitors validation, uses patience, and restores the best weights.
- One `fit()` with options beats a new copy of the loop for every experiment.

### ✔️ Check yourself

Commit to an answer (say it out loud or write it down) *before* you expand it.

**1.** Where exactly does `clip_grad_norm_` go, and what would go wrong if you called it before `backward()`?

<details>
<summary>Answer</summary>

After `loss.backward()` and before `optimizer.step()`. Before `backward()` the gradients are from the previous step (or `None`), so you would clip the wrong gradients and the new ones would be unclipped.

</details>

**2.** With 4 accumulation steps, why divide the loss by 4?

<details>
<summary>Answer</summary>

Each micro-batch loss is a mean over its own examples. Summing 4 such gradients gives 4× the gradient of the overall mean; dividing by 4 makes the step identical to one with the full batch.

</details>

**3.** A colleague compares recipe A (constant lr) with recipe B (OneCycle, stronger weight decay, clipping) in one run each and concludes that OneCycle is better. What is wrong?

<details>
<summary>Answer</summary>

Three things changed at once, so the difference cannot be attributed to the schedule; and one run per recipe cannot separate a real effect from seed-to-seed noise. Change one variable, use several seeds, and compare the means with their spread.

</details>

**4.** Early stopping with patience 3 restores the weights from the best epoch. Why is this better than keeping the final weights?

<details>
<summary>Answer</summary>

The final weights come from 3 (or more) epochs without improvement, during which the model may have started to overfit. The best epoch is, by the validation metric, the best model seen.

</details>

---
# Chapter 5. Checkpointing and reproducibility

🎯 **In this chapter you will learn to:** distinguish weights-for-inference from full training checkpoints; save checkpoints safely; resume an interrupted run so that it continues **exactly** as if it had never stopped; load weights that only partly match a model; and control (and report honestly) run-to-run randomness.

Training runs get interrupted: Colab recycles your session, a job hits its time limit, a machine fails. At work, multi-day training runs are routinely resumed from checkpoints. A checkpoint that restores only the weights is **not** enough to continue correctly.

🔁 **Starting a fresh session here?** Run the two Chapter 0 cells, then the hidden restore-point cell below. It re-creates the tools from earlier chapters that Chapter 5 uses. In a normal top-to-bottom run it does nothing.

In [ ]:
# @title 🔁 Restore point for Chapter 5 (needed only if you start a fresh session here) { display-mode: "form" }
if "device" not in globals():
    raise RuntimeError("Run the Chapter 0 setup cells first, then this cell.")
if "fit" not in globals():
    from torch.optim import lr_scheduler as S

    def param_groups_weight_decay(model, weight_decay):
        decay, no_decay = [], []
        for name, p in model.named_parameters():
            if not p.requires_grad:
                continue                                     # frozen parameters: leave them out
            (no_decay if p.ndim <= 1 else decay).append(p)   # biases and norm weights are 1-D
        return [{"params": decay, "weight_decay": weight_decay},
                {"params": no_decay, "weight_decay": 0.0}]

    class EarlyStopping:
        # Stop when `metric` has not improved by more than `min_delta` for `patience` epochs.
        # mode="min" for losses, mode="max" for accuracies. Keeps a copy of the best weights.
        def __init__(self, patience=5, min_delta=0.0, mode="min"):
            self.patience, self.min_delta, self.mode = patience, min_delta, mode
            self.best = math.inf if mode == "min" else -math.inf
            self.best_state, self.best_epoch, self.bad_epochs, self.epoch = None, 0, 0, 0

        def improved(self, value):
            if self.mode == "min":
                return value < self.best - self.min_delta
            return value > self.best + self.min_delta

        def step(self, value, model):
            # Call once per epoch. Returns True when training should stop.
            self.epoch += 1
            if self.improved(value):
                self.best, self.best_epoch, self.bad_epochs = value, self.epoch, 0
                self.best_state = copy.deepcopy(model.state_dict())
            else:
                self.bad_epochs += 1
            return self.bad_epochs >= self.patience

        def restore(self, model):
            model.load_state_dict(self.best_state)

    import contextlib

    def train_one_epoch(model, loader, loss_fn, optimizer, scheduler=None, max_grad_norm=None,
                        averaged_model=None, amp_dtype=None, scaler=None):
        # amp_dtype / scaler: mixed precision, explained in Chapter 11. Leave them as None until then.
        model.train()
        total_loss = torch.zeros((), device=device)                  # running sums stay ON the device:
        correct = torch.zeros((), device=device, dtype=torch.long)   # no CPU-GPU synchronisation per step
        n, grad_norms = 0, []
        for x, y in loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            amp = torch.autocast(device_type=device.type, dtype=amp_dtype) if amp_dtype else contextlib.nullcontext()
            with amp:
                logits = model(x)
                loss = loss_fn(logits, y)
            optimizer.zero_grad(set_to_none=True)
            if scaler is not None:                                   # float16 mixed precision (Chapter 11)
                scaler.scale(loss).backward()
                if max_grad_norm is not None:
                    scaler.unscale_(optimizer)                       # clip the REAL gradients, not the scaled ones
            else:
                loss.backward()
            if max_grad_norm is not None:
                grad_norms.append(nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm))
            if scaler is not None:
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()
            if scheduler is not None:                                # per-batch schedules (OneCycleLR, warm-up)
                scheduler.step()
            if averaged_model is not None:                           # weight averaging (Chapter 6)
                averaged_model.update_parameters(model)
            total_loss += loss.detach().float() * len(y)
            correct += (logits.argmax(dim=1) == y).sum()
            n += len(y)
        mean_norm = torch.stack(grad_norms).mean().item() if grad_norms else float("nan")
        return total_loss.item() / n, correct.item() / n, mean_norm   # one synchronisation per epoch


    def fit(model, train_loader, val_loader, optimizer, epochs, loss_fn=nn.CrossEntropyLoss(),
            scheduler=None, scheduler_interval="batch", early_stopping=None, eval_model=None, verbose=True, **train_kw):
        # scheduler_interval: "batch", "epoch" or "plateau" (ReduceLROnPlateau).
        # eval_model: evaluate (and select) a different model than the one trained, e.g. an EMA copy (Chapter 6).
        history = {k: [] for k in ["train_loss", "train_acc", "val_loss", "val_acc", "lr", "grad_norm", "time"]}
        eval_model = eval_model if eval_model is not None else model
        for epoch in range(1, epochs + 1):
            t0 = time.perf_counter()
            lr = optimizer.param_groups[0]["lr"]
            tr_loss, tr_acc, gnorm = train_one_epoch(model, train_loader, loss_fn, optimizer,
                                                     scheduler if scheduler_interval == "batch" else None, **train_kw)
            va_loss, va_acc = evaluate(eval_model, val_loader, loss_fn)
            if scheduler is not None and scheduler_interval == "epoch":
                scheduler.step()
            elif scheduler is not None and scheduler_interval == "plateau":
                scheduler.step(va_loss)
            elapsed = time.perf_counter() - t0
            for k, v in zip(history, (tr_loss, tr_acc, va_loss, va_acc, lr, gnorm, elapsed)):
                history[k].append(v)
            if verbose:
                print(f"epoch {epoch:2d} | lr {lr:.2e} | train {tr_loss:.4f} / {tr_acc:.4f} | "
                      f"val {va_loss:.4f} / {va_acc:.4f} | {elapsed:4.1f}s")
            if early_stopping is not None and early_stopping.step(va_loss, eval_model):
                print(f"early stopping: no improvement for {early_stopping.patience} epochs")
                break
        if early_stopping is not None:                       # select the epoch with the best validation LOSS...
            early_stopping.restore(eval_model)
            sel_epoch = early_stopping.best_epoch
        else:
            sel_epoch = len(history["val_loss"])            # ...or keep the last epoch
        sel_loss, sel_acc = history["val_loss"][sel_epoch - 1], history["val_acc"][sel_epoch - 1]
        history["selected"] = {"epoch": sel_epoch, "val_loss": sel_loss, "val_acc": sel_acc}
        if verbose:
            print(f"selected model: epoch {sel_epoch}, val loss {sel_loss:.4f}, val acc {sel_acc:.4f}")
        return history


    def plot_history(histories, keys=("val_loss", "val_acc", "lr")):
        fig, axes = plt.subplots(1, len(keys), figsize=(5 * len(keys), 3.2))
        for ax, key in zip(axes, keys):
            for name, h in histories.items():
                ax.plot(range(1, len(h[key]) + 1), h[key], "o-", ms=3, label=name)
            ax.set_title(key); ax.set_xlabel("epoch")
        axes[0].legend(fontsize=8); plt.tight_layout(); plt.show()
    print("Restored the tools defined in earlier chapters.")

## 5.1 Two kinds of saved files

| | Weights for inference | Full training checkpoint |
| --- | --- | --- |
| Purpose | Use or deploy the trained model | Continue training later |
| Contents | `model.state_dict()` + config (architecture settings, class names, preprocessing) | Everything on the left **plus** the optimizer state, scheduler state, epoch counter, best metric so far, and random-number-generator states |
| Size | ≈ size of the weights | ≈ 3× the weights with Adam (two moment buffers per parameter) |

Why each item matters when resuming:
- **Optimizer state.** Adam's moment estimates (or SGD's momentum) took thousands of steps to build up. Without them, the first steps after resuming behave like the start of training, with a jolt in the loss.
- **Scheduler state.** Otherwise the learning-rate schedule restarts from its beginning (for example, a second warm-up).
- **Epoch and best metric.** To know how far to continue and when to save a new best model.
- **RNG states.** Dropout masks and data shuffling depend on random numbers. To reproduce the uninterrupted run exactly, the generators must continue where they stopped.

🏭 **Save the `state_dict`, not the model object.** `torch.save(model)` pickles the class by reference to your source code. The file then breaks as soon as the code is moved or renamed, and loading it requires running arbitrary code. Save `state_dict`s and rebuild the model from code and config.

## 5.2 Saving and loading a full checkpoint

In [ ]:
def save_checkpoint(path, model, optimizer, scheduler=None, epoch=0, best_metric=None,
                    loader_generator=None, config=None):
    np_state = np.random.get_state()
    ckpt = {
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict() if scheduler is not None else None,
        "epoch": epoch,
        "best_metric": best_metric,
        "config": config or {},
        "rng": {
            "torch": torch.get_rng_state(),
            "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [],
            "python": random.getstate(),
            # NumPy's state contains an array; store it as a list so the safe loader accepts it
            "numpy": (np_state[0], np_state[1].tolist(), *np_state[2:]),
        },
        "loader_generator": loader_generator.get_state() if loader_generator is not None else None,
    }
    tmp = Path(str(path) + ".tmp")
    torch.save(ckpt, tmp)             # write to a temporary file first...
    os.replace(tmp, path)             # ...then rename: an interruption can never leave a half-written checkpoint


def load_checkpoint(path, model, optimizer=None, scheduler=None, loader_generator=None):
    ckpt = torch.load(path, map_location=device, weights_only=True)   # safe loading: tensors and plain data only
    model.load_state_dict(ckpt["model"])
    if optimizer is not None:
        optimizer.load_state_dict(ckpt["optimizer"])
    if scheduler is not None and ckpt["scheduler"] is not None:
        scheduler.load_state_dict(ckpt["scheduler"])
    rng = ckpt["rng"]
    torch.set_rng_state(rng["torch"].cpu())
    if torch.cuda.is_available() and rng["cuda"]:
        torch.cuda.set_rng_state_all([s.cpu() for s in rng["cuda"]])
    random.setstate(rng["python"])
    name, keys, *rest = rng["numpy"]
    np.random.set_state((name, np.array(keys, dtype=np.uint32), *rest))
    if loader_generator is not None and ckpt["loader_generator"] is not None:
        loader_generator.set_state(ckpt["loader_generator"].cpu())
    return ckpt

Two details deserve a word:
- **`weights_only=True`** (the default since PyTorch 2.6) makes `torch.load` refuse anything except tensors and basic Python containers and numbers. A checkpoint is a pickle file, and an untrusted pickle can execute arbitrary code when loaded. Treat checkpoints downloaded from the internet like executables, and keep the safe default on.
- **`map_location=device`** loads tensors directly onto the current device, so a checkpoint saved on a GPU loads on a CPU-only machine.

## 5.3 Proof that resuming is exact
The experiment: train for 4 epochs without interruption (run A). Then repeat, but stop after 2 epochs, save a checkpoint, throw away every object, build fresh ones with a **different** seed, load the checkpoint and train 2 more epochs (run B). If the checkpoint is complete, the final weights of A and B are **identical**, bit for bit.

We use a small in-memory subset so this takes seconds, a model with Dropout (which consumes random numbers), AdamW (optimizer state) and a OneCycle schedule (scheduler state). The training loader gets its **own** random generator, so its shuffling can be saved and restored too.

In [ ]:
# Small in-memory subsets (normalised exactly like fm_transform) for quick experiments
def _fm_subset(idx):
    x = ((raw_train.data[idx].float() / 255 - FM_MEAN) / FM_STD).unsqueeze(1)
    return TensorDataset(x, raw_train.targets[idx])

small_ds = _fm_subset(FM_TRAIN_IDX[:4096])      # 4,096 training images
small_val = _fm_subset(FM_VAL_IDX[:2000])       # 2,000 validation images

In [ ]:
RESUME_EPOCHS = 4

def make_run():
    # Everything a run needs, built from scratch.
    gen = torch.Generator().manual_seed(1234)                       # the loader's own RNG
    loader = DataLoader(small_ds, batch_size=128, shuffle=True, generator=gen)
    model = build_mlp(p_drop=0.3).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
    sched = S.OneCycleLR(opt, max_lr=3e-3, total_steps=RESUME_EPOCHS * len(loader))
    return model, opt, sched, loader, gen

def train_epochs(model, opt, sched, loader, epochs):
    for _ in range(epochs):
        train_one_epoch(model, loader, nn.CrossEntropyLoss(), opt, sched)

# Run A: 4 epochs, no interruption
set_seed(0)
model_A, opt_A, sched_A, loader_A, _ = make_run()
train_epochs(model_A, opt_A, sched_A, loader_A, RESUME_EPOCHS)

# Run B: 2 epochs, checkpoint, "crash", rebuild with another seed, load, 2 more epochs
set_seed(0)
model_B, opt_B, sched_B, loader_B, gen_B = make_run()
train_epochs(model_B, opt_B, sched_B, loader_B, 2)
save_checkpoint(CKPT_DIR / "resume_demo.pt", model_B, opt_B, sched_B, epoch=2, loader_generator=gen_B)
del model_B, opt_B, sched_B, loader_B, gen_B                       # the "crash"

set_seed(999)                                                      # deliberately different
model_B, opt_B, sched_B, loader_B, gen_B = make_run()
ckpt = load_checkpoint(CKPT_DIR / "resume_demo.pt", model_B, opt_B, sched_B, loader_generator=gen_B)
train_epochs(model_B, opt_B, sched_B, loader_B, RESUME_EPOCHS - ckpt["epoch"])

same = all(torch.equal(a, b) for a, b in zip(model_A.state_dict().values(), model_B.state_dict().values()))
max_diff = max((a.float() - b.float()).abs().max().item()
               for a, b in zip(model_A.state_dict().values(), model_B.state_dict().values()))
print("run A and resumed run B have identical weights:", same, f"| largest difference: {max_diff:.2e}")

On a CPU this prints `True` with a largest difference of exactly 0: the resumed run is indistinguishable from the uninterrupted one. (This was verified when the notebook was written.) On a GPU, check what **your** run prints. If it prints `False`, the largest difference tells you whether you are looking at floating-point rounding (around 1e-7 to 1e-5), caused by GPU kernels that are not bit-for-bit deterministic (§5.4), or at a real bug (much larger, like the cases in Exercise 5.1). Either way, the *procedure* is what matters: every piece of state is restored.

Exercise 5.1 asks you to remove one item from the checkpoint at a time and watch the equality break. That is the best way to remember why each one is there.

## 5.4 Reproducibility: what you can and cannot control
Sources of randomness in a training run, and how to control each:

| Source | Control |
| --- | --- |
| Weight initialisation, Dropout, `torch.rand*` | `torch.manual_seed(seed)` (also seeds all GPUs) |
| Python `random`, NumPy (often used in data augmentation) | `random.seed(seed)`, `np.random.seed(seed)` |
| Shuffling in the `DataLoader` | `DataLoader(..., generator=torch.Generator().manual_seed(seed))` |
| Random augmentations inside worker processes | Seeded automatically from the loader's generator for `torch` ops; NumPy/`random` inside workers need a `worker_init_fn` (below) |
| Non-deterministic GPU kernels (some reductions use atomic additions, whose order varies) | `torch.use_deterministic_algorithms(True)`; set the environment variable `CUBLAS_WORKSPACE_CONFIG=:4096:8` for CUDA; `torch.backends.cudnn.benchmark = False` |
| Different hardware, driver or library versions | **Cannot be controlled.** Different chips and library versions use different kernels and summation orders. |

The `worker_init_fn` recommended by the PyTorch documentation:

```python
def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32      # already distinct per worker and per epoch
    np.random.seed(worker_seed)
    random.seed(worker_seed)

DataLoader(ds, ..., worker_init_fn=seed_worker, generator=torch.Generator().manual_seed(0))
```

Full determinism makes some operations slower and makes others raise an error, because no deterministic implementation exists. A common professional approach: aim for determinism when debugging, and otherwise accept small nondeterminism and **measure** its effect.

### Seed-to-seed variation: report it
Even a perfectly reproducible run is just one sample. Change the seed and the result changes. Before claiming that a change improved accuracy by 0.3%, check that 0.3% is larger than the seed-to-seed variation. Good papers report a mean ± standard deviation over several seeds.

In [ ]:
accs = []
for seed in range(3 if FAST_DEV_RUN else 5):
    set_seed(seed)
    m = build_mlp().to(device)
    opt = torch.optim.AdamW(m.parameters(), lr=1e-3)
    loader = DataLoader(small_ds, batch_size=128, shuffle=True, generator=torch.Generator().manual_seed(seed))
    for _ in range(n_epochs(5)):
        train_one_epoch(m, loader, nn.CrossEntropyLoss(), opt)
    accs.append(evaluate(m, DataLoader(small_val, batch_size=1000))[1])
accs = torch.tensor(accs)
print("validation accuracy per seed:", [f"{a:.4f}" for a in accs.tolist()])
print(f"mean {accs.mean():.4f} ± std {accs.std():.4f}")

## 5.5 Good checkpointing habits
- **Save `last.pt` every epoch** (for resuming) and **`best.pt`** whenever the validation metric improves (for using). Optionally keep every N-th epoch.
- **On Colab, save to Google Drive** (`from google.colab import drive; drive.mount("/content/drive")`), because the local disk disappears when the session is recycled.
- **Store the config** (all hyper-parameters, and ideally the code version, e.g. the git commit) in the checkpoint. A checkpoint you cannot rebuild is worthless.
- **Resume logic at the top of the training script:** if `last.pt` exists, load it and continue from `ckpt["epoch"] + 1`. Then the same command both starts and resumes a run.

## 5.6 Loading weights that only partly match
You will often load weights into a model that differs slightly: a pretrained backbone with a new classification head, or a checkpoint from code whose layer names have changed. `load_state_dict(sd, strict=False)` loads whatever matches by name and **reports** what did not:

In [ ]:
src = build_mlp(hidden=(256, 128))
dst = nn.Sequential(*list(build_mlp(hidden=(256, 128)).children())[:-1], nn.Linear(128, 5))   # new 5-class head
sd = {k: v for k, v in src.state_dict().items() if not k.startswith("7.")}   # drop the old head (layer 7)'s weights
result = dst.load_state_dict(sd, strict=False)
print("missing keys (not in the file, left at their initial values):", result.missing_keys)
print("unexpected keys (in the file, but not in the model):         ", result.unexpected_keys)

⚠️ **Always print and read `missing_keys` and `unexpected_keys` when you use `strict=False`.** If a naming difference (say, a `"module."` prefix) makes **every** key mismatch, `strict=False` silently loads nothing, and you train from random weights while believing you fine-tuned. Also, `strict=False` does not forgive **shape** mismatches: a key that matches by name but not by shape still raises an error, which is why we removed the old head above.

## 🧪 Exercises

**Exercise 5.1: break the resume, one piece at a time.** `★★☆ · ~20 min`
Repeat run B from §5.3 three times, each time omitting one restoration step: (a) do not load the optimizer state (`load_checkpoint(..., optimizer=None)`), (b) do not load the scheduler, (c) do not restore the loader's generator. In each case, print whether the weights still equal run A's, and the largest absolute difference.

In [ ]:
# ✍️ Your attempt here.

In [ ]:
# @title ✅ Solution (double-click to reveal) { display-mode: "form" }
# Solutions only execute when RUN_SOLUTIONS = True (setup cell), so "Run all" stays fast.
if RUN_SOLUTIONS:
    def resumed_run(**skip):
        model, opt, sched, loader, gen = make_run()
        load_checkpoint(CKPT_DIR / "resume_demo.pt", model,
                        None if skip.get("opt") else opt,
                        None if skip.get("sched") else sched,
                        None if skip.get("gen") else gen)
        train_epochs(model, opt, sched, loader, RESUME_EPOCHS - 2)
        return model

    for label, skip in {"(a) no optimizer state": {"opt": True},
                        "(b) no scheduler state": {"sched": True},
                        "(c) no loader generator": {"gen": True}}.items():
        set_seed(999)
        m = resumed_run(**skip)
        diff = max((a - b).abs().max().item() for a, b in zip(model_A.state_dict().values(), m.state_dict().values()))
        print(f"{label:25s} identical: {diff == 0.0} | max difference {diff:.2e}")
    # (a) Adam restarts with zero moments: the first steps are badly scaled.
    # (b) OneCycle restarts from its beginning: the lr jumps back to the warm-up value.
    # (c) the data order of epochs 3-4 differs from the uninterrupted run.
    # Each one breaks exact reproduction; (a) and (b) can also visibly hurt the training curve.
else:
    print("Solution not executed (RUN_SOLUTIONS = False).")

**Exercise 5.2: a robust partial loader.** `★★☆ · ~20 min`
Write `load_matching(model, state_dict)` that (1) strips a leading `"module."` from every key (added by `nn.DataParallel` and by `AveragedModel`, Chapter 6), (2) loads only the entries whose name **and shape** match the model, and (3) returns the list of model keys that were **not** loaded. It must never raise an error because of a shape mismatch.

In [ ]:
# ✍️ Your attempt here.
# Hint: Compare against model.state_dict(); use load_state_dict(filtered, strict=False).

In [ ]:
# 🔎 Self-check for Exercise 5.2
def _test(fn):
    src = build_mlp(hidden=(256, 128))
    sd = {"module." + k: v for k, v in src.state_dict().items()}
    dst = build_mlp(hidden=(256, 64))                        # second hidden layer has a different size
    not_loaded = fn(dst, sd)
    assert torch.equal(dst.state_dict()["1.weight"], src.state_dict()["1.weight"]), "matching weights were not loaded"
    assert set(not_loaded) == {"4.weight", "4.bias", "7.weight"}, f"wrong list of keys not loaded: {not_loaded}"
self_check("load_matching", _test)

In [ ]:
# @title ✅ Solution (double-click to reveal) { display-mode: "form" }
# Solutions only execute when RUN_SOLUTIONS = True (setup cell), so "Run all" stays fast.
if RUN_SOLUTIONS:
    def load_matching(model, state_dict):
        own = model.state_dict()
        cleaned = {k.removeprefix("module."): v for k, v in state_dict.items()}
        usable = {k: v for k, v in cleaned.items() if k in own and own[k].shape == v.shape}
        model.load_state_dict(usable, strict=False)
        return [k for k in own if k not in usable]

    print(load_matching(build_mlp(hidden=(256, 64)), {"module." + k: v for k, v in build_mlp().state_dict().items()}))
else:
    print("Solution not executed (RUN_SOLUTIONS = False).")

### 📌 Key takeaways
- A resumable checkpoint = model + optimizer + scheduler + epoch + best metric + RNG states (+ the loader's generator) + config.
- Write checkpoints atomically; load with `weights_only=True` and `map_location`.
- Exact resumption is achievable and testable. Bit-identical results across different hardware are not.
- Report results over several seeds. With `strict=False`, always inspect the missing and unexpected keys.

### ✔️ Check yourself

Commit to an answer (say it out loud or write it down) *before* you expand it.

**1.** After resuming from a checkpoint that contains only the model weights, the training loss jumps up for a few hundred steps, then recovers. What was missing?

<details>
<summary>Answer</summary>

Most likely the optimizer state (Adam's moment estimates restart at zero, so early steps are badly scaled) and possibly the scheduler state (the learning rate restarted from its initial or warm-up value).

</details>

**2.** Why is `torch.load(path, weights_only=False)` on a downloaded file a security risk?

<details>
<summary>Answer</summary>

Checkpoints are pickle files, and unpickling can execute arbitrary code. `weights_only=True` only reconstructs tensors and basic types.

</details>

**3.** Your new method beats the baseline by 0.2% accuracy in a single run each. Is that evidence of an improvement?

<details>
<summary>Answer</summary>

Not yet. Compare it with the seed-to-seed standard deviation, typically 0.1–0.3% for this kind of model. Run several seeds of both and compare means relative to their spread.

</details>

---
# Chapter 6. Weight averaging: EMA and SWA

🎯 **In this chapter you will learn to:** explain why averaging weights along the training trajectory helps; implement an exponential moving average (EMA) of a model by hand; use PyTorch's `AveragedModel` for EMA and stochastic weight averaging (SWA); and recompute BatchNorm statistics for averaged weights.

🔁 **Starting a fresh session here?** Run the two Chapter 0 cells, then the hidden restore-point cell below. It re-creates the tools from earlier chapters that Chapter 6 uses. In a normal top-to-bottom run it does nothing.

In [ ]:
# @title 🔁 Restore point for Chapter 6 (needed only if you start a fresh session here) { display-mode: "form" }
if "device" not in globals():
    raise RuntimeError("Run the Chapter 0 setup cells first, then this cell.")
if "load_checkpoint" not in globals():
    from torch.optim import lr_scheduler as S

    def param_groups_weight_decay(model, weight_decay):
        decay, no_decay = [], []
        for name, p in model.named_parameters():
            if not p.requires_grad:
                continue                                     # frozen parameters: leave them out
            (no_decay if p.ndim <= 1 else decay).append(p)   # biases and norm weights are 1-D
        return [{"params": decay, "weight_decay": weight_decay},
                {"params": no_decay, "weight_decay": 0.0}]

    class EarlyStopping:
        # Stop when `metric` has not improved by more than `min_delta` for `patience` epochs.
        # mode="min" for losses, mode="max" for accuracies. Keeps a copy of the best weights.
        def __init__(self, patience=5, min_delta=0.0, mode="min"):
            self.patience, self.min_delta, self.mode = patience, min_delta, mode
            self.best = math.inf if mode == "min" else -math.inf
            self.best_state, self.best_epoch, self.bad_epochs, self.epoch = None, 0, 0, 0

        def improved(self, value):
            if self.mode == "min":
                return value < self.best - self.min_delta
            return value > self.best + self.min_delta

        def step(self, value, model):
            # Call once per epoch. Returns True when training should stop.
            self.epoch += 1
            if self.improved(value):
                self.best, self.best_epoch, self.bad_epochs = value, self.epoch, 0
                self.best_state = copy.deepcopy(model.state_dict())
            else:
                self.bad_epochs += 1
            return self.bad_epochs >= self.patience

        def restore(self, model):
            model.load_state_dict(self.best_state)

    import contextlib

    def train_one_epoch(model, loader, loss_fn, optimizer, scheduler=None, max_grad_norm=None,
                        averaged_model=None, amp_dtype=None, scaler=None):
        # amp_dtype / scaler: mixed precision, explained in Chapter 11. Leave them as None until then.
        model.train()
        total_loss = torch.zeros((), device=device)                  # running sums stay ON the device:
        correct = torch.zeros((), device=device, dtype=torch.long)   # no CPU-GPU synchronisation per step
        n, grad_norms = 0, []
        for x, y in loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            amp = torch.autocast(device_type=device.type, dtype=amp_dtype) if amp_dtype else contextlib.nullcontext()
            with amp:
                logits = model(x)
                loss = loss_fn(logits, y)
            optimizer.zero_grad(set_to_none=True)
            if scaler is not None:                                   # float16 mixed precision (Chapter 11)
                scaler.scale(loss).backward()
                if max_grad_norm is not None:
                    scaler.unscale_(optimizer)                       # clip the REAL gradients, not the scaled ones
            else:
                loss.backward()
            if max_grad_norm is not None:
                grad_norms.append(nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm))
            if scaler is not None:
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()
            if scheduler is not None:                                # per-batch schedules (OneCycleLR, warm-up)
                scheduler.step()
            if averaged_model is not None:                           # weight averaging (Chapter 6)
                averaged_model.update_parameters(model)
            total_loss += loss.detach().float() * len(y)
            correct += (logits.argmax(dim=1) == y).sum()
            n += len(y)
        mean_norm = torch.stack(grad_norms).mean().item() if grad_norms else float("nan")
        return total_loss.item() / n, correct.item() / n, mean_norm   # one synchronisation per epoch


    def fit(model, train_loader, val_loader, optimizer, epochs, loss_fn=nn.CrossEntropyLoss(),
            scheduler=None, scheduler_interval="batch", early_stopping=None, eval_model=None, verbose=True, **train_kw):
        # scheduler_interval: "batch", "epoch" or "plateau" (ReduceLROnPlateau).
        # eval_model: evaluate (and select) a different model than the one trained, e.g. an EMA copy (Chapter 6).
        history = {k: [] for k in ["train_loss", "train_acc", "val_loss", "val_acc", "lr", "grad_norm", "time"]}
        eval_model = eval_model if eval_model is not None else model
        for epoch in range(1, epochs + 1):
            t0 = time.perf_counter()
            lr = optimizer.param_groups[0]["lr"]
            tr_loss, tr_acc, gnorm = train_one_epoch(model, train_loader, loss_fn, optimizer,
                                                     scheduler if scheduler_interval == "batch" else None, **train_kw)
            va_loss, va_acc = evaluate(eval_model, val_loader, loss_fn)
            if scheduler is not None and scheduler_interval == "epoch":
                scheduler.step()
            elif scheduler is not None and scheduler_interval == "plateau":
                scheduler.step(va_loss)
            elapsed = time.perf_counter() - t0
            for k, v in zip(history, (tr_loss, tr_acc, va_loss, va_acc, lr, gnorm, elapsed)):
                history[k].append(v)
            if verbose:
                print(f"epoch {epoch:2d} | lr {lr:.2e} | train {tr_loss:.4f} / {tr_acc:.4f} | "
                      f"val {va_loss:.4f} / {va_acc:.4f} | {elapsed:4.1f}s")
            if early_stopping is not None and early_stopping.step(va_loss, eval_model):
                print(f"early stopping: no improvement for {early_stopping.patience} epochs")
                break
        if early_stopping is not None:                       # select the epoch with the best validation LOSS...
            early_stopping.restore(eval_model)
            sel_epoch = early_stopping.best_epoch
        else:
            sel_epoch = len(history["val_loss"])            # ...or keep the last epoch
        sel_loss, sel_acc = history["val_loss"][sel_epoch - 1], history["val_acc"][sel_epoch - 1]
        history["selected"] = {"epoch": sel_epoch, "val_loss": sel_loss, "val_acc": sel_acc}
        if verbose:
            print(f"selected model: epoch {sel_epoch}, val loss {sel_loss:.4f}, val acc {sel_acc:.4f}")
        return history


    def plot_history(histories, keys=("val_loss", "val_acc", "lr")):
        fig, axes = plt.subplots(1, len(keys), figsize=(5 * len(keys), 3.2))
        for ax, key in zip(axes, keys):
            for name, h in histories.items():
                ax.plot(range(1, len(h[key]) + 1), h[key], "o-", ms=3, label=name)
            ax.set_title(key); ax.set_xlabel("epoch")
        axes[0].legend(fontsize=8); plt.tight_layout(); plt.show()

    # Small in-memory subsets (normalised exactly like fm_transform) for quick experiments
    def _fm_subset(idx):
        x = ((raw_train.data[idx].float() / 255 - FM_MEAN) / FM_STD).unsqueeze(1)
        return TensorDataset(x, raw_train.targets[idx])

    small_ds = _fm_subset(FM_TRAIN_IDX[:4096])      # 4,096 training images
    small_val = _fm_subset(FM_VAL_IDX[:2000])       # 2,000 validation images

    def save_checkpoint(path, model, optimizer, scheduler=None, epoch=0, best_metric=None,
                        loader_generator=None, config=None):
        np_state = np.random.get_state()
        ckpt = {
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict() if scheduler is not None else None,
            "epoch": epoch,
            "best_metric": best_metric,
            "config": config or {},
            "rng": {
                "torch": torch.get_rng_state(),
                "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [],
                "python": random.getstate(),
                # NumPy's state contains an array; store it as a list so the safe loader accepts it
                "numpy": (np_state[0], np_state[1].tolist(), *np_state[2:]),
            },
            "loader_generator": loader_generator.get_state() if loader_generator is not None else None,
        }
        tmp = Path(str(path) + ".tmp")
        torch.save(ckpt, tmp)             # write to a temporary file first...
        os.replace(tmp, path)             # ...then rename: an interruption can never leave a half-written checkpoint


    def load_checkpoint(path, model, optimizer=None, scheduler=None, loader_generator=None):
        ckpt = torch.load(path, map_location=device, weights_only=True)   # safe loading: tensors and plain data only
        model.load_state_dict(ckpt["model"])
        if optimizer is not None:
            optimizer.load_state_dict(ckpt["optimizer"])
        if scheduler is not None and ckpt["scheduler"] is not None:
            scheduler.load_state_dict(ckpt["scheduler"])
        rng = ckpt["rng"]
        torch.set_rng_state(rng["torch"].cpu())
        if torch.cuda.is_available() and rng["cuda"]:
            torch.cuda.set_rng_state_all([s.cpu() for s in rng["cuda"]])
        random.setstate(rng["python"])
        name, keys, *rest = rng["numpy"]
        np.random.set_state((name, np.array(keys, dtype=np.uint32), *rest))
        if loader_generator is not None and ckpt["loader_generator"] is not None:
            loader_generator.set_state(ckpt["loader_generator"].cpu())
        return ckpt
    print("Restored the tools defined in earlier chapters.")

## 6.1 The idea
With mini-batch gradients, the weights never settle exactly at a minimum. In the late phase of training they **bounce around** it, pushed by the noise of each batch. The average of those bouncing points lies closer to the centre of the low-loss region than any single point. Averaging weights is therefore a cheap way to remove noise from the final model, and the averaged weights tend to sit in **flatter** regions of the loss surface, which often generalise slightly better.

The simulation below makes this concrete with noisy gradient descent on a 2-D bowl:

In [ ]:
set_seed(0)
w = torch.tensor([2.0, 2.0])
H = torch.tensor([1.0, 4.0])                      # curvature of the bowl f(w) = 0.5 * sum(H * w^2)
lr, path, ema, ema_path = 0.2, [], w.clone(), []
for step in range(300):
    grad = H * w + 1.0 * torch.randn(2)            # true gradient + mini-batch noise
    w = w - lr * grad
    ema = 0.95 * ema + 0.05 * w                    # exponential moving average of the iterates
    path.append(w.clone()); ema_path.append(ema.clone())
path, ema_path = torch.stack(path), torch.stack(ema_path)

plt.figure(figsize=(5, 4.2))
plt.scatter(path[100:, 0], path[100:, 1], s=6, alpha=0.5, label="SGD iterates (last 200)")
plt.plot(ema_path[100:, 0], ema_path[100:, 1], color="tab:red", lw=1.5, label="EMA of the iterates")
plt.scatter([0], [0], marker="*", s=200, color="k", label="minimum")
plt.legend(fontsize=8); plt.axis("equal"); plt.title("Averaging removes the noise"); plt.show()
print(f"mean distance to the minimum: iterates {path[100:].norm(dim=1).mean():.3f}, "
      f"EMA {ema_path[100:].norm(dim=1).mean():.3f}")

Two averaging schemes are common:

| | EMA (exponential moving average) | SWA (stochastic weight averaging) |
| --- | --- | --- |
| Formula | after every step: $\bar\theta \leftarrow \beta\,\bar\theta + (1-\beta)\,\theta$ | equal-weight average of snapshots: $\bar\theta = \frac{1}{K}\sum_k \theta_k$ |
| When | throughout training | only in a final phase, typically the last 20–25% of epochs, with a constant or cyclic learning rate |
| Typical setting | $\beta = 0.999$–$0.9999$ | one snapshot per epoch |
| Seen in | diffusion models, object detectors (YOLO), semi-supervised learning, many vision models | classification and regression recipes; `torch.optim.swa_utils` |

The EMA's decay $\beta$ sets its **memory**: it averages over roughly the last $1/(1-\beta)$ steps. With $\beta = 0.999$ that is about 1,000 steps.

## 6.2 EMA from scratch
The EMA model is a separate copy of the network whose weights are updated with the formula above after every optimizer step. It is never trained directly, so its parameters do not need gradients.

In [ ]:
class EMA:
    def __init__(self, model, decay=0.999):
        self.module = copy.deepcopy(model).eval()        # a separate copy, used only for evaluation
        for p in self.module.parameters():
            p.requires_grad_(False)
        self.decay = decay

    @torch.no_grad()
    def update_parameters(self, model):                  # same method name as PyTorch's AveragedModel
        for ema_p, p in zip(self.module.parameters(), model.parameters()):
            ema_p.lerp_(p, 1 - self.decay)               # ema = ema + (1 - decay) * (p - ema)
        for ema_b, b in zip(self.module.buffers(), model.buffers()):
            ema_b.copy_(b)                               # BatchNorm statistics: copy the current ones

# Check the formula on a tiny model
lin = nn.Linear(2, 1)
ema = EMA(lin, decay=0.9)
with torch.no_grad():
    lin.weight.add_(1.0)                                 # pretend an optimizer step moved the weights by +1
old = ema.module.weight.clone()
ema.update_parameters(lin)
print("EMA moved 10% of the way to the new weights:", torch.allclose(ema.module.weight, old + 0.1, atol=1e-6))

`lerp_(target, weight)` ("linear interpolation") computes `self + weight · (target − self)` in place, which is the EMA formula rearranged. You will see exactly this line in many repositories.

**The warm-up problem.** The EMA starts from the random initial weights. With $\beta = 0.999$ it takes thousands of steps to forget them, so early in training the EMA model is worse than the trained one. Implementations therefore often ramp the decay up, for example $\beta_t = \min\big(\beta, \frac{1 + t}{10 + t}\big)$, or start averaging only after some number of steps.

## 6.3 PyTorch's `AveragedModel`
`torch.optim.swa_utils.AveragedModel` implements both schemes. For EMA, pass an averaging function; with `use_buffers=True` it also averages the BatchNorm buffers:

```python
from torch.optim.swa_utils import AveragedModel, get_ema_multi_avg_fn
ema_model = AveragedModel(model, multi_avg_fn=get_ema_multi_avg_fn(0.999), use_buffers=True)
...
optimizer.step()
ema_model.update_parameters(model)           # after every optimizer step
```

⚠️ `AveragedModel` **wraps** the model: the averaged network is `ema_model.module`, and `ema_model.state_dict()` has every key prefixed with `"module."` plus an extra `n_averaged` entry. To deploy the averaged weights, save `ema_model.module.state_dict()`. (Or use the `load_matching` function from Exercise 5.2.)

Our `train_one_epoch` already calls `averaged_model.update_parameters(model)` after each step when you pass `averaged_model=...`, so both our `EMA` class and `AveragedModel` plug straight in.

## 6.4 SWA and the BatchNorm problem
SWA averages snapshots from the end of training: for the last few epochs, the learning rate is held at a moderate constant value (so the weights keep exploring the low-loss region instead of freezing), and the weights are added to the running average once per epoch. `SWALR` provides the constant-learning-rate phase.

There is one catch. **BatchNorm's running statistics were computed for the individual snapshots, not for the averaged weights.** An averaged network produces different activations, so its stored statistics are wrong, sometimes badly. The fix is one pass over the training data in training mode to recompute them, which `update_bn` does:

```python
from torch.optim.swa_utils import AveragedModel, SWALR, update_bn
swa_model = AveragedModel(model)                      # default: equal-weight average
swa_scheduler = SWALR(optimizer, swa_lr=5e-4)
for epoch in range(epochs):
    train_one_epoch(...)
    if epoch >= swa_start:
        swa_model.update_parameters(model)            # once per epoch
        swa_scheduler.step()
update_bn(train_loader, swa_model, device=device)     # recompute BatchNorm statistics
```

## 6.5 Experiment: last weights vs EMA vs SWA
We train an MLP **with BatchNorm** for 15 epochs with AdamW. An EMA copy is updated every step, and SWA averages the last 5 epochs. We evaluate the three models after every epoch.

In [ ]:
from torch.optim.swa_utils import AveragedModel, SWALR, update_bn, get_ema_multi_avg_fn

set_seed(0)
EPOCHS, SWA_START = n_epochs(15), n_epochs(15) - n_epochs(5)
model = build_mlp(batchnorm=True).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3)
ema_model = AveragedModel(model, multi_avg_fn=get_ema_multi_avg_fn(0.999), use_buffers=True)
swa_model = AveragedModel(model)
swa_scheduler = SWALR(optimizer, swa_lr=1e-3, anneal_epochs=1)

log = {"last": [], "EMA": [], "SWA": []}
for epoch in range(EPOCHS):
    train_one_epoch(model, train_loader, nn.CrossEntropyLoss(), optimizer, averaged_model=ema_model)
    log["last"].append(evaluate(model, val_loader)[1])
    log["EMA"].append(evaluate(ema_model, val_loader)[1])
    if epoch >= SWA_START:
        swa_model.update_parameters(model)
        swa_scheduler.step()
        update_bn(train_loader, swa_model, device=device)      # needed before evaluating SWA
        log["SWA"].append(evaluate(swa_model, val_loader)[1])
    print(f"epoch {epoch + 1:2d} | last {log['last'][-1]:.4f} | EMA {log['EMA'][-1]:.4f}"
          + (f" | SWA {log['SWA'][-1]:.4f}" if epoch >= SWA_START else ""))

plt.figure(figsize=(6.5, 3.5))
plt.plot(range(1, EPOCHS + 1), log["last"], "o-", label="last weights")
plt.plot(range(1, EPOCHS + 1), log["EMA"], "o-", label="EMA (0.999)")
plt.plot(range(SWA_START + 1, EPOCHS + 1), log["SWA"], "o-", label="SWA (last 5 epochs)")
plt.xlabel("epoch"); plt.ylabel("validation accuracy"); plt.legend(); plt.show()
for name in ("last", "EMA", "SWA"):
    record(f"Ch6 {name} weights, final val acc", log[name][-1])

**What to expect.** The EMA curve is visibly **smoother** than the curve of the trained weights, and by the end it is usually equal or a few tenths of a percent better. SWA typically lands in the same range. On a small, easy problem like this the gains are modest. They are larger with noisier training: small batches, high learning rates, or long training runs where the last checkpoint is a lottery. The cost is small too: one extra copy of the weights, and an `update_bn` pass for SWA. That is why EMA is standard in many modern training recipes.

Calling `update_bn` every epoch here is only for the plot. In practice you call it once, at the end.

## 🧪 Exercises

**Exercise 6.1: forget `update_bn`.** `★☆☆ · ~10 min`
Create a fresh `AveragedModel(model)`, average the current `model` together with a slightly perturbed copy (add `0.05 * torch.randn_like(p)` to every parameter of the copy) using `update_parameters` twice, and evaluate it on `val_loader` **without** calling `update_bn`. Then call `update_bn` and evaluate again. Explain the difference.

In [ ]:
# ✍️ Your attempt here.

In [ ]:
# @title ✅ Solution (double-click to reveal) { display-mode: "form" }
# Solutions only execute when RUN_SOLUTIONS = True (setup cell), so "Run all" stays fast.
if RUN_SOLUTIONS:
    avg = AveragedModel(model)                     # its first update copies the weights
    avg.update_parameters(model)
    perturbed = copy.deepcopy(model)
    with torch.no_grad():
        for p in perturbed.parameters():
            p.add_(0.05 * torch.randn_like(p))
    avg.update_parameters(perturbed)               # now the average of two different networks
    print("without update_bn:", evaluate(avg, val_loader)[1])
    update_bn(train_loader, avg, device=device)
    print("with update_bn:   ", evaluate(avg, val_loader)[1])
    # AveragedModel(model) copies the BatchNorm buffers of the FIRST model and never averages them
    # (use_buffers=False is the default). The averaged weights produce different activations, so the
    # stored means and variances no longer match: each BatchNorm layer shifts and rescales its inputs
    # wrongly, and the errors compound through the layers. update_bn recomputes the statistics for
    # the weights that are actually used. (How large the drop is depends on how far the weights moved.)
else:
    print("Solution not executed (RUN_SOLUTIONS = False).")

**Exercise 6.2: choosing the decay.** `★★☆ · ~15 min`
One epoch here is about 430 optimizer steps. Train `build_mlp()` for 5 epochs while updating three EMA models with decays 0.99, 0.999 and 0.9999 (use our `EMA` class; call `update_parameters` on all three after every step, which you can do by writing the loop yourself). Compare their validation accuracy after each epoch. Which decay is too large for a 5-epoch run, and why?

In [ ]:
# ✍️ Your attempt here.
# Hint: Write the batch loop yourself: forward, loss, zero_grad, backward, step, then update the three EMAs.

In [ ]:
# @title ✅ Solution (double-click to reveal) { display-mode: "form" }
# Solutions only execute when RUN_SOLUTIONS = True (setup cell), so "Run all" stays fast.
if RUN_SOLUTIONS:
    set_seed(0)
    m = build_mlp().to(device)
    opt = torch.optim.AdamW(m.parameters(), lr=1e-3)
    emas = {d: EMA(m, d) for d in (0.99, 0.999, 0.9999)}
    for epoch in range(n_epochs(5)):
        m.train()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            loss = F.cross_entropy(m(x), y)
            opt.zero_grad(); loss.backward(); opt.step()
            for e in emas.values():
                e.update_parameters(m)
        print(f"epoch {epoch + 1}: model {evaluate(m, val_loader)[1]:.4f} | " +
              " | ".join(f"EMA {d}: {evaluate(e.module, val_loader)[1]:.4f}" for d, e in emas.items()))
    # 0.9999 averages over about 10,000 steps, i.e. more than the whole run (about 2,150 steps):
    # it is still dominated by the random initial weights and performs badly. Rule of thumb:
    # 1 / (1 - decay) should be well below the total number of steps (or use a decay warm-up).
else:
    print("Solution not executed (RUN_SOLUTIONS = False).")

### 📌 Key takeaways
- Averaging weights along the trajectory removes mini-batch noise; it is cheap and often gives a small, reliable gain.
- EMA: update after every step with `lerp_`; memory ≈ `1/(1−β)` steps; mind the warm-up.
- SWA: average the last epochs at a constant learning rate; always run `update_bn` before evaluating or saving.
- `AveragedModel` adds a `"module."` prefix; save `averaged.module.state_dict()`.

### ✔️ Check yourself

Commit to an answer (say it out loud or write it down) *before* you expand it.

**1.** Why does an SWA model with BatchNorm need `update_bn`, while a model averaged without any normalisation layers does not?

<details>
<summary>Answer</summary>

BatchNorm stores running statistics of activations that were computed with other weights. Averaged weights produce different activations, so the statistics must be recomputed. Without normalisation layers there is no such stored state.

</details>

**2.** You evaluate an EMA model with decay 0.9999 after 500 training steps and it is much worse than the trained model. Is EMA broken?

<details>
<summary>Answer</summary>

No. Its memory is about 10,000 steps, so it is still mostly the random initialisation. Use a smaller decay, a decay warm-up, or start averaging later.

</details>

**3.** Does the EMA model influence the training of the main model?

<details>
<summary>Answer</summary>

No. It is updated from the trained weights but never fed back. It is only used for evaluation and deployment.

</details>

---
# Chapter 7. Convolutional networks, from first principles

🎯 **In this chapter you will learn to:** explain why fully connected layers are a poor fit for images; build convolution yourself from sliding windows and `einsum`, and check it against PyTorch; predict output shapes and parameter counts of `nn.Conv2d`; use pooling, receptive fields and `BatchNorm2d`; train a CNN that clearly beats the MLP on Fashion-MNIST; and apply 1-D convolutions to signals of different lengths using padding and masks.

🔁 **Starting a fresh session here?** Run the two Chapter 0 cells, then the hidden restore-point cell below. It re-creates the tools from earlier chapters that Chapter 7 uses. In a normal top-to-bottom run it does nothing.

In [ ]:
# @title 🔁 Restore point for Chapter 7 (needed only if you start a fresh session here) { display-mode: "form" }
if "device" not in globals():
    raise RuntimeError("Run the Chapter 0 setup cells first, then this cell.")
if "load_checkpoint" not in globals():
    from torch.optim import lr_scheduler as S

    def param_groups_weight_decay(model, weight_decay):
        decay, no_decay = [], []
        for name, p in model.named_parameters():
            if not p.requires_grad:
                continue                                     # frozen parameters: leave them out
            (no_decay if p.ndim <= 1 else decay).append(p)   # biases and norm weights are 1-D
        return [{"params": decay, "weight_decay": weight_decay},
                {"params": no_decay, "weight_decay": 0.0}]

    class EarlyStopping:
        # Stop when `metric` has not improved by more than `min_delta` for `patience` epochs.
        # mode="min" for losses, mode="max" for accuracies. Keeps a copy of the best weights.
        def __init__(self, patience=5, min_delta=0.0, mode="min"):
            self.patience, self.min_delta, self.mode = patience, min_delta, mode
            self.best = math.inf if mode == "min" else -math.inf
            self.best_state, self.best_epoch, self.bad_epochs, self.epoch = None, 0, 0, 0

        def improved(self, value):
            if self.mode == "min":
                return value < self.best - self.min_delta
            return value > self.best + self.min_delta

        def step(self, value, model):
            # Call once per epoch. Returns True when training should stop.
            self.epoch += 1
            if self.improved(value):
                self.best, self.best_epoch, self.bad_epochs = value, self.epoch, 0
                self.best_state = copy.deepcopy(model.state_dict())
            else:
                self.bad_epochs += 1
            return self.bad_epochs >= self.patience

        def restore(self, model):
            model.load_state_dict(self.best_state)

    import contextlib

    def train_one_epoch(model, loader, loss_fn, optimizer, scheduler=None, max_grad_norm=None,
                        averaged_model=None, amp_dtype=None, scaler=None):
        # amp_dtype / scaler: mixed precision, explained in Chapter 11. Leave them as None until then.
        model.train()
        total_loss = torch.zeros((), device=device)                  # running sums stay ON the device:
        correct = torch.zeros((), device=device, dtype=torch.long)   # no CPU-GPU synchronisation per step
        n, grad_norms = 0, []
        for x, y in loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            amp = torch.autocast(device_type=device.type, dtype=amp_dtype) if amp_dtype else contextlib.nullcontext()
            with amp:
                logits = model(x)
                loss = loss_fn(logits, y)
            optimizer.zero_grad(set_to_none=True)
            if scaler is not None:                                   # float16 mixed precision (Chapter 11)
                scaler.scale(loss).backward()
                if max_grad_norm is not None:
                    scaler.unscale_(optimizer)                       # clip the REAL gradients, not the scaled ones
            else:
                loss.backward()
            if max_grad_norm is not None:
                grad_norms.append(nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm))
            if scaler is not None:
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()
            if scheduler is not None:                                # per-batch schedules (OneCycleLR, warm-up)
                scheduler.step()
            if averaged_model is not None:                           # weight averaging (Chapter 6)
                averaged_model.update_parameters(model)
            total_loss += loss.detach().float() * len(y)
            correct += (logits.argmax(dim=1) == y).sum()
            n += len(y)
        mean_norm = torch.stack(grad_norms).mean().item() if grad_norms else float("nan")
        return total_loss.item() / n, correct.item() / n, mean_norm   # one synchronisation per epoch


    def fit(model, train_loader, val_loader, optimizer, epochs, loss_fn=nn.CrossEntropyLoss(),
            scheduler=None, scheduler_interval="batch", early_stopping=None, eval_model=None, verbose=True, **train_kw):
        # scheduler_interval: "batch", "epoch" or "plateau" (ReduceLROnPlateau).
        # eval_model: evaluate (and select) a different model than the one trained, e.g. an EMA copy (Chapter 6).
        history = {k: [] for k in ["train_loss", "train_acc", "val_loss", "val_acc", "lr", "grad_norm", "time"]}
        eval_model = eval_model if eval_model is not None else model
        for epoch in range(1, epochs + 1):
            t0 = time.perf_counter()
            lr = optimizer.param_groups[0]["lr"]
            tr_loss, tr_acc, gnorm = train_one_epoch(model, train_loader, loss_fn, optimizer,
                                                     scheduler if scheduler_interval == "batch" else None, **train_kw)
            va_loss, va_acc = evaluate(eval_model, val_loader, loss_fn)
            if scheduler is not None and scheduler_interval == "epoch":
                scheduler.step()
            elif scheduler is not None and scheduler_interval == "plateau":
                scheduler.step(va_loss)
            elapsed = time.perf_counter() - t0
            for k, v in zip(history, (tr_loss, tr_acc, va_loss, va_acc, lr, gnorm, elapsed)):
                history[k].append(v)
            if verbose:
                print(f"epoch {epoch:2d} | lr {lr:.2e} | train {tr_loss:.4f} / {tr_acc:.4f} | "
                      f"val {va_loss:.4f} / {va_acc:.4f} | {elapsed:4.1f}s")
            if early_stopping is not None and early_stopping.step(va_loss, eval_model):
                print(f"early stopping: no improvement for {early_stopping.patience} epochs")
                break
        if early_stopping is not None:                       # select the epoch with the best validation LOSS...
            early_stopping.restore(eval_model)
            sel_epoch = early_stopping.best_epoch
        else:
            sel_epoch = len(history["val_loss"])            # ...or keep the last epoch
        sel_loss, sel_acc = history["val_loss"][sel_epoch - 1], history["val_acc"][sel_epoch - 1]
        history["selected"] = {"epoch": sel_epoch, "val_loss": sel_loss, "val_acc": sel_acc}
        if verbose:
            print(f"selected model: epoch {sel_epoch}, val loss {sel_loss:.4f}, val acc {sel_acc:.4f}")
        return history


    def plot_history(histories, keys=("val_loss", "val_acc", "lr")):
        fig, axes = plt.subplots(1, len(keys), figsize=(5 * len(keys), 3.2))
        for ax, key in zip(axes, keys):
            for name, h in histories.items():
                ax.plot(range(1, len(h[key]) + 1), h[key], "o-", ms=3, label=name)
            ax.set_title(key); ax.set_xlabel("epoch")
        axes[0].legend(fontsize=8); plt.tight_layout(); plt.show()

    # Small in-memory subsets (normalised exactly like fm_transform) for quick experiments
    def _fm_subset(idx):
        x = ((raw_train.data[idx].float() / 255 - FM_MEAN) / FM_STD).unsqueeze(1)
        return TensorDataset(x, raw_train.targets[idx])

    small_ds = _fm_subset(FM_TRAIN_IDX[:4096])      # 4,096 training images
    small_val = _fm_subset(FM_VAL_IDX[:2000])       # 2,000 validation images

    def save_checkpoint(path, model, optimizer, scheduler=None, epoch=0, best_metric=None,
                        loader_generator=None, config=None):
        np_state = np.random.get_state()
        ckpt = {
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict() if scheduler is not None else None,
            "epoch": epoch,
            "best_metric": best_metric,
            "config": config or {},
            "rng": {
                "torch": torch.get_rng_state(),
                "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [],
                "python": random.getstate(),
                # NumPy's state contains an array; store it as a list so the safe loader accepts it
                "numpy": (np_state[0], np_state[1].tolist(), *np_state[2:]),
            },
            "loader_generator": loader_generator.get_state() if loader_generator is not None else None,
        }
        tmp = Path(str(path) + ".tmp")
        torch.save(ckpt, tmp)             # write to a temporary file first...
        os.replace(tmp, path)             # ...then rename: an interruption can never leave a half-written checkpoint


    def load_checkpoint(path, model, optimizer=None, scheduler=None, loader_generator=None):
        ckpt = torch.load(path, map_location=device, weights_only=True)   # safe loading: tensors and plain data only
        model.load_state_dict(ckpt["model"])
        if optimizer is not None:
            optimizer.load_state_dict(ckpt["optimizer"])
        if scheduler is not None and ckpt["scheduler"] is not None:
            scheduler.load_state_dict(ckpt["scheduler"])
        rng = ckpt["rng"]
        torch.set_rng_state(rng["torch"].cpu())
        if torch.cuda.is_available() and rng["cuda"]:
            torch.cuda.set_rng_state_all([s.cpu() for s in rng["cuda"]])
        random.setstate(rng["python"])
        name, keys, *rest = rng["numpy"]
        np.random.set_state((name, np.array(keys, dtype=np.uint32), *rest))
        if loader_generator is not None and ckpt["loader_generator"] is not None:
            loader_generator.set_state(ckpt["loader_generator"].cpu())
        return ckpt
    print("Restored the tools defined in earlier chapters.")

## 7.1 Why an MLP is the wrong tool for images
1. **Too many parameters.** A 224×224 colour image has 150,528 values. A first fully connected layer with 1,000 units would need 150 million weights, for one layer.
2. **No notion of "nearby".** An MLP treats pixels as an unordered list. Shuffle every image's pixels with the same fixed permutation and an MLP trains exactly as well, which shows that it ignores the spatial structure that makes an image an image.
3. **No reuse of what it learns.** A pattern (say, the edge of a collar) learned in one place must be learned again, with separate weights, in every other place it can appear.

A **convolutional layer** fixes all three: it slides one small filter over the image, so it looks only at local neighbourhoods, uses the same weights everywhere (**weight sharing**), and has a number of parameters that does not depend on the image size.

## 7.2 Convolution in 1-D: a sliding dot product
Chapter 1 built sliding windows from strides. A 1-D convolution is just **a dot product of every window with a small filter** (also called a kernel):

$$y_i = \sum_{j=0}^{k-1} w_j\, x_{i+j} + b.$$

The filter `[-1, 0, 1]` responds to *changes* in the signal, so it is a simple edge (step) detector:

In [ ]:
signal = torch.tensor([0., 0., 0., 0., 1., 1., 1., 1., 0., 0., 0.])   # a rectangular pulse
kernel = torch.tensor([-1., 0., 1.])

windows = signal.unfold(0, 3, 1)                     # (9, 3): every window of length 3 (Chapter 1)
by_hand = windows @ kernel                           # dot product of each window with the filter
by_torch = F.conv1d(signal.view(1, 1, -1), kernel.view(1, 1, -1)).flatten()   # (N, C, L) layout
print("windows @ kernel:", by_hand)
print("F.conv1d:        ", by_torch)

The output is +1 where the signal steps up and −1 where it steps down: the filter **detects edges**.

⚠️ **Terminology.** In mathematics and signal processing, convolution flips the kernel before sliding it. Deep-learning "convolution" does **not** flip it; strictly, it computes a *cross-correlation*. Since the filters are learned, the difference does not matter for training, but it matters when you compare with a signals textbook or port a hand-designed filter.

## 7.3 Convolution in 2-D, built from patches
For images, the filter is a small 2-D grid (typically 3×3) that slides over both height and width. With several input channels (for example RGB), each filter spans **all** input channels; with several output channels, there is one filter per output channel. A `Conv2d` weight therefore has shape

$$(C_\text{out},\; C_\text{in},\; k_H,\; k_W),$$

and output channel $o$ at position $(i, j)$ is

$$y_{o,i,j} = b_o + \sum_{c} \sum_{u,v} w_{o,c,u,v}\; x_{c,\,i+u,\,j+v}.$$

We can implement exactly this with two tools you already know. `F.unfold` cuts the input into all its $k \times k$ patches (the "im2col" trick, which real libraries also use), and each output is then the dot product of a flattened patch with a flattened filter, an `einsum`:

In [ ]:
torch.manual_seed(0)
x = torch.randn(2, 3, 6, 6)                  # (N, C_in, H, W)
w = torch.randn(4, 3, 3, 3)                  # (C_out, C_in, kH, kW): 4 filters, each 3x3 over 3 channels
b = torch.randn(4)

patches = F.unfold(x, kernel_size=3)         # (N, C_in*3*3, L): every 3x3x3 patch as a column; L = 4*4 positions
print("patches:", tuple(patches.shape))
out = torch.einsum("ok,nkl->nol", w.reshape(4, -1), patches) + b.view(1, 4, 1)   # dot product per patch & filter
out = out.reshape(2, 4, 4, 4)                # back to (N, C_out, H_out, W_out)

print("matches F.conv2d:", torch.allclose(out, F.conv2d(x, w, b), atol=1e-5))

That is all a convolutional layer computes. GPUs make it fast, but there is no other magic. (Exercise 7.2 extends this to stride and padding.)

## 7.4 `nn.Conv2d`: shapes and parameter counts
`nn.Conv2d(in_channels, out_channels, kernel_size, stride=1, padding=0, dilation=1)` has four settings that determine the output size:
- **`kernel_size`**: the filter size $k$;
- **`padding`**: zeros added around the border, $p$ on each side, so the filter can be centred on edge pixels;
- **`stride`**: the step between filter positions, $s$; `stride=2` halves the resolution;
- **`dilation`**: spacing between filter taps, $d$, which enlarges the field of view without more weights.

The output height (and likewise width) is

$$H_\text{out} = \left\lfloor \frac{H + 2p - d\,(k - 1) - 1}{s} \right\rfloor + 1.$$

| Layer, input 28×28 | Output | Why |
| --- | --- | --- |
| `Conv2d(1, 32, 3)` | 26×26 | no padding: lose $k-1 = 2$ pixels |
| `Conv2d(1, 32, 3, padding=1)` | 28×28 | "same" size: $p = (k-1)/2$ for odd $k$ |
| `Conv2d(1, 32, 3, stride=2, padding=1)` | 14×14 | stride 2 halves the size |
| `Conv2d(1, 32, 5, padding=2)` | 28×28 | same rule with $k = 5$ |

**Parameters:** $C_\text{out} \cdot C_\text{in} \cdot k_H \cdot k_W + C_\text{out}$, which is **independent of the image size**.

In [ ]:
x = torch.randn(8, 1, 28, 28)
for layer in [nn.Conv2d(1, 32, 3), nn.Conv2d(1, 32, 3, padding=1), nn.Conv2d(1, 32, 3, stride=2, padding=1),
              nn.Conv2d(1, 32, 5, padding="same"), nn.Conv2d(32, 64, 3, padding=1)]:
    inp = x if layer.in_channels == 1 else torch.randn(8, 32, 28, 28)
    print(f"{str(layer):62s} -> {str(tuple(layer(inp).shape)):17s} params {count_params(layer):,}")
print(f"\nfor comparison, a Linear layer from 784 pixels to 32*28*28 outputs would have {784 * 32 * 28 * 28 + 32 * 28 * 28:,} parameters")

## 7.5 What a filter detects, and translation equivariance
Hand-designed filters show what convolutions can do. The Sobel filters below respond to vertical and horizontal edges. A trained CNN's first layer learns filters much like these by itself.

In [ ]:
img = fm_transform(raw_train[0][0]).unsqueeze(0)            # (1, 1, 28, 28), an ankle boot
sobel_x = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]])
filters = torch.stack([sobel_x, sobel_x.T]).unsqueeze(1)     # (2, 1, 3, 3): vertical- and horizontal-edge filters
edges = F.conv2d(img, filters, padding=1)                    # (1, 2, 28, 28)

fig, axes = plt.subplots(1, 3, figsize=(8, 2.8))
for ax, im, title in zip(axes, [img[0, 0], edges[0, 0], edges[0, 1]], ["input", "vertical edges", "horizontal edges"]):
    ax.imshow(im, cmap="gray"); ax.set_title(title); ax.axis("off")
plt.tight_layout(); plt.show()

**Translation equivariance.** Because the same filter is applied everywhere, shifting the input shifts the output by the same amount: $\text{conv}(\text{shift}(x)) = \text{shift}(\text{conv}(x))$. The check below uses circular padding, so that pixels shifted off one edge re-enter on the other and the equality is exact.

In [ ]:
conv = nn.Conv2d(1, 4, 3, padding=1, padding_mode="circular", bias=False)
shift = lambda t: torch.roll(t, shifts=(3, 5), dims=(2, 3))      # move 3 pixels down, 5 right
with torch.no_grad():
    print("conv(shift(x)) == shift(conv(x)):", torch.allclose(conv(shift(img)), shift(conv(img)), atol=1e-5))

A detector for "collar" therefore works wherever the collar is. Pooling and the final global average pooling (next section) then turn this equivariance into approximate **invariance**: the class prediction barely changes when the object moves.

## 7.6 Pooling, receptive fields, and BatchNorm for images
**Pooling** reduces resolution: `nn.MaxPool2d(2)` keeps the maximum of every 2×2 block, halving height and width. A strided convolution (`stride=2`) is a learnable alternative that many modern networks prefer.

**Global average pooling** (`nn.AdaptiveAvgPool2d(1)`) averages each channel over all positions, turning `(N, C, H, W)` into `(N, C, 1, 1)`. It is the standard bridge between the convolutional part and the final linear classifier, and it works for any input size.

**Receptive field.** One 3×3 convolution "sees" a 3×3 region of the input. Stacking layers grows the region: each new $k \times k$ layer adds $(k-1)\cdot J$ pixels, where $J$ is the product of the strides of the previous layers (the "jump" between neighbouring positions measured in input pixels). Downsampling therefore makes later layers see much larger regions cheaply.

In [ ]:
def receptive_field(layers):
    # layers: list of (kernel_size, stride). Returns the receptive field after each layer.
    r, jump, out = 1, 1, []
    for k, s in layers:
        r += (k - 1) * jump
        jump *= s
        out.append(r)
    return out

stack = [(3, 1), (3, 1), (2, 2), (3, 1), (3, 1), (2, 2), (3, 1)]      # the CNN we build next
names = ["conv", "conv", "pool", "conv", "conv", "pool", "conv"]
for name, rf in zip(names, receptive_field(stack)):
    print(f"{name:5s} receptive field {rf:2d} x {rf:2d}")

By the last convolution each output position sees 24×24 of the 28×28 input, so it can respond to whole garment parts; global average pooling then combines the evidence from all positions.

**`BatchNorm2d(C)`** normalises each **channel** with a mean and variance computed over the batch **and all spatial positions** (dimensions N, H, W), and has a learnable scale and shift per channel (2C parameters). Convolutions followed by BatchNorm are usually created with `bias=False`, because BatchNorm's shift makes the bias redundant.

## 7.7 A CNN for Fashion-MNIST
The design follows a pattern you will see everywhere: blocks of *convolution → BatchNorm → ReLU*, with the resolution halved and the number of channels doubled at each stage, then global average pooling and a linear classifier.

In [ ]:
def conv_bn_relu(c_in, c_out):
    return nn.Sequential(nn.Conv2d(c_in, c_out, 3, padding=1, bias=False), nn.BatchNorm2d(c_out), nn.ReLU(inplace=True))


class SmallCNN(nn.Module):
    def __init__(self, num_classes=10, width=32, p_drop=0.3):
        super().__init__()
        w = width
        self.features = nn.Sequential(
            conv_bn_relu(1, w), conv_bn_relu(w, w), nn.MaxPool2d(2),              # 28x28 -> 14x14
            conv_bn_relu(w, 2 * w), conv_bn_relu(2 * w, 2 * w), nn.MaxPool2d(2),  # 14x14 -> 7x7
            conv_bn_relu(2 * w, 4 * w),                                           # 7x7, 128 channels
            nn.AdaptiveAvgPool2d(1),                                              # -> 1x1
        )
        self.head = nn.Sequential(nn.Flatten(), nn.Dropout(p_drop), nn.Linear(4 * w, num_classes))

    def forward(self, x):
        return self.head(self.features(x))

In [ ]:
cnn = SmallCNN()
x = torch.randn(2, 1, 28, 28)
print(f"{'input':38s} {tuple(x.shape)}")
for layer in list(cnn.features) + list(cnn.head):          # trace the shape through every stage
    x = layer(x)
    print(f"{layer.__class__.__name__ if not isinstance(layer, nn.Sequential) else 'Conv-BN-ReLU (' + str(layer[0].out_channels) + ')':38s} {tuple(x.shape)}")
print(f"\nparameters: {count_params(cnn):,}  (the MLP of Notebook 1 had 235,146)")

`ReLU(inplace=True)` overwrites its input instead of allocating a new tensor. This is safe here because the BatchNorm output it overwrites is not needed for the backward pass, and you will see it often in vision code.

Now train it with the tools from Chapters 3–4: AdamW with weight-decay groups and a OneCycle schedule. The training recipe is wrapped in a function, `train_small_cnn`, because later chapters need this trained network: it saves the weights to `CKPT_DIR`, and `get_trained_cnn()` loads them again (or retrains if the file is missing, for example in a fresh session). Expect a few seconds per epoch on a T4.

In [ ]:
CNN_PATH = CKPT_DIR / "smallcnn_fashion.pt"

def train_small_cnn(epochs=12, seed=0):
    # The recipe of Section 7.7: AdamW with weight-decay groups and a OneCycle schedule.
    set_seed(seed)
    model = SmallCNN().to(device)
    E = n_epochs(epochs)
    opt = torch.optim.AdamW(param_groups_weight_decay(model, 0.05), lr=1e-3)
    sched = S.OneCycleLR(opt, max_lr=3e-3, total_steps=E * len(train_loader))
    hist = fit(model, train_loader, val_loader, opt, E, scheduler=sched,
               early_stopping=EarlyStopping(patience=100))        # patience=100: just keep the best epoch
    torch.save({"state_dict": model.state_dict()}, CNN_PATH)
    return model, hist

def get_trained_cnn():
    # Load the CNN trained in Chapter 7 if it was saved; otherwise train it now.
    model = SmallCNN().to(device)
    if CNN_PATH.exists():
        model.load_state_dict(torch.load(CNN_PATH, map_location=device, weights_only=True)["state_dict"])
        print(f"loaded the trained SmallCNN from {CNN_PATH}")
    else:
        print("no saved SmallCNN found: training one with the recipe of Section 7.7 ...")
        model, _ = train_small_cnn()
    return model.eval()

In [ ]:
cnn, hist_cnn = train_small_cnn()
record("Ch7 SmallCNN, test accuracy", evaluate(cnn, test_loader)[1])

In typical runs the CNN reaches around 92–93% test accuracy, compared with about 88–89% for the MLP, with **fewer** parameters (compare with your recorded numbers). That gap is the value of building the right assumptions (locality, weight sharing) into the architecture.

Let us check where the improvement came from, using the confusion-matrix trick from Notebook 1:

In [ ]:
@torch.no_grad()
def per_class_accuracy(model, loader):
    model.eval()
    preds, ys = [], []
    for x, y in loader:
        preds.append(model(x.to(device)).argmax(dim=1).cpu()); ys.append(y)
    p, t = torch.cat(preds), torch.cat(ys)
    cm = torch.bincount(t * 10 + p, minlength=100).reshape(10, 10)
    return cm.diag() / cm.sum(dim=1).clamp(min=1)

set_seed(0)
mlp_ref = build_mlp().to(device)
opt_ref = torch.optim.AdamW(mlp_ref.parameters(), lr=1e-3)
print("training the reference MLP for comparison...")
fit(mlp_ref, train_loader, val_loader, opt_ref, n_epochs(12), verbose=False)
acc_mlp, acc_cnn = per_class_accuracy(mlp_ref, test_loader), per_class_accuracy(cnn, test_loader)
for i in (acc_cnn - acc_mlp).argsort(descending=True):
    print(f"{CLASSES[i]:12s} MLP {acc_mlp[i]:.3f} -> CNN {acc_cnn[i]:.3f}  ({acc_cnn[i] - acc_mlp[i]:+.3f})")

The largest gains are usually on the hard upper-body garments, especially **Shirt**, where the details (collars, buttons, sleeve shapes) are local patterns that convolutions are built to detect.

### Looking inside: learned filters and feature maps

In [ ]:
first = cnn.features[0][0].weight.detach().cpu()           # (32, 1, 3, 3)
fig, axes = plt.subplots(2, 16, figsize=(14, 2))
for i, ax in enumerate(axes.flat):
    ax.imshow(first[i, 0], cmap="RdBu"); ax.axis("off")
plt.suptitle("First-layer filters (red/blue = positive/negative weights)"); plt.show()

cnn.eval()
with torch.no_grad():
    fmap = cnn.features[:2](img.to(device))[0].cpu()        # output of the first two blocks: (32, 28, 28)
fig, axes = plt.subplots(2, 8, figsize=(12, 3.2))
for i, ax in enumerate(axes.flat):
    ax.imshow(fmap[i], cmap="viridis"); ax.axis("off")
plt.suptitle("16 feature maps after two conv blocks (input: the ankle boot above)"); plt.show()

## 7.8 1-D convolutions for signals, and inputs of different lengths
Convolutions are not only for images. For **time series** (vibration, audio, ECG, process sensors) `nn.Conv1d` slides a filter along time. The layout is `(N, C, L)`: batch, channels (sensors), length. Everything from this chapter carries over, with one dimension instead of two.

Real recordings rarely have equal lengths. A `DataLoader` builds a batch with `torch.stack`, which needs equal shapes, so we need two new tools:
1. a **collate function** that **pads** every signal in a batch to the longest one and returns a **mask** marking the real samples;
2. **masked pooling**, so that the padded zeros cannot influence the prediction.

As a predictive-maintenance example, we classify synthetic machine vibration signals into three conditions: **healthy** (shaft rotation plus noise), **misalignment** (a strong component at twice the shaft frequency) and **bearing fault** (periodic, decaying impacts). The signals are 0.2–0.8 s long, sampled at 1 kHz, and the fault signatures are deliberately weak compared with the noise, as in real early-stage faults.

In [ ]:
FS = 1000                                                     # sampling rate, Hz
VIB_CLASSES = ["healthy", "misalignment", "bearing fault"]

def make_vibration(label, length, g):
    t = torch.arange(length) / FS
    f = 20 + 10 * torch.rand(1, generator=g)                  # shaft speed 20-30 Hz
    phase = 2 * math.pi * torch.rand(1, generator=g)
    x = torch.sin(2 * math.pi * f * t + phase) + 0.8 * torch.randn(length, generator=g)   # sensor noise
    if label == 1:                                            # misalignment: strong 2x harmonic
        x += 0.45 * torch.sin(2 * math.pi * 2 * f * t + phase)
    elif label == 2:                                          # bearing fault: decaying impacts at ~4.3x shaft speed
        period = int(FS / (4.3 * f.item()))
        k = torch.arange(30)
        burst = torch.exp(-k / 5.0) * torch.sin(2 * math.pi * 0.3 * k)
        start = int(torch.randint(0, period, (1,), generator=g))
        for s0 in range(start, length - 30, period):
            x[s0:s0 + 30] += 0.9 * burst
    return x

def make_vibration_set(n, seed):
    g = torch.Generator().manual_seed(seed)
    labels = torch.randint(0, 3, (n,), generator=g)
    lengths = torch.randint(200, 801, (n,), generator=g)
    return [make_vibration(int(c), int(L), g) for c, L in zip(labels, lengths)], labels

n_sig = 300 if FAST_DEV_RUN else 3000
train_signals, train_sig_labels = make_vibration_set(n_sig, seed=0)
val_signals, val_sig_labels = make_vibration_set(n_sig // 5, seed=1)

fig, axes = plt.subplots(3, 1, figsize=(10, 5), sharex=True)
for c, ax in enumerate(axes):
    i = int((train_sig_labels == c).nonzero()[0])
    ax.plot(torch.arange(len(train_signals[i])) / FS, train_signals[i], lw=0.7)
    ax.set_ylabel(VIB_CLASSES[c], fontsize=8)
axes[-1].set_xlabel("time (s)"); plt.suptitle("One signal per class (note the different lengths)"); plt.tight_layout(); plt.show()

**Padding collate.** `collate_fn` receives the list of examples the `DataLoader` picked for one batch and must return the batch. `pad_sequence` pads to the longest signal, and the mask is a broadcasting comparison (Notebook 1, §1.4) of each position with each signal's length.

In [ ]:
from torch.nn.utils.rnn import pad_sequence

class SignalDataset(Dataset):
    def __init__(self, signals, labels):
        self.signals, self.labels = signals, labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, i):
        return self.signals[i], self.labels[i]

def pad_collate(batch):
    signals, labels = zip(*batch)
    lengths = torch.tensor([len(s) for s in signals])
    x = pad_sequence(list(signals), batch_first=True)                 # (B, L_max): zeros after each signal ends
    mask = torch.arange(x.shape[1])[None, :] < lengths[:, None]       # (B, L_max): True on real samples
    return x.unsqueeze(1), mask, torch.stack(labels)                   # x: (B, 1, L_max), one input channel

sig_train_loader = DataLoader(SignalDataset(train_signals, train_sig_labels), batch_size=64, shuffle=True, collate_fn=pad_collate)
sig_val_loader = DataLoader(SignalDataset(val_signals, val_sig_labels), batch_size=256, collate_fn=pad_collate)
xb, mb, yb = next(iter(sig_train_loader))
print("x:", tuple(xb.shape), "| mask:", tuple(mb.shape), "| real samples in the first 4 signals:", mb.sum(dim=1)[:4].tolist())

try:
    next(iter(DataLoader(SignalDataset(train_signals, train_sig_labels), batch_size=4)))    # default collate
except RuntimeError as e:
    print("default collate fails:", str(e)[:75], "...")

**A 1-D CNN with masked pooling.** Three convolutions with growing **dilation** (1, 2, 4) give each output position a receptive field of 57 samples (about 57 ms) cheaply, and `padding` keeps the length unchanged, so the mask still lines up with the feature maps. Pooling then uses only the real positions: a masked **mean** (sum over real positions divided by their number) and a masked **max** (padded positions are set to $-\infty$ with `masked_fill` first).

⚠️ **The padding must be re-zeroed after every layer.** The padded input is zero, but after one convolution the padded positions are not: they contain the bias plus contributions from the last real samples, which the filter reaches across the boundary. The next layer would then mix these values back into the real positions near the end of the signal, and the output of a short signal would depend on how much padding its batch happened to need. `masked_fill(~m, 0.0)` after each layer restores exactly the zeros that the convolution's own padding would provide for the signal on its own.

There is deliberately no BatchNorm: its batch statistics would include the padded zeros. Normalisation layers that work per example (or masked versions) are used instead in sequence models.

In [ ]:
class SignalCNN(nn.Module):
    def __init__(self, num_classes=3, c=32):
        super().__init__()
        self.convs = nn.ModuleList([
            nn.Conv1d(1, c, 9, padding=4),
            nn.Conv1d(c, c, 9, padding=8, dilation=2),
            nn.Conv1d(c, c, 9, padding=16, dilation=4),
        ])
        self.head = nn.Linear(2 * c, num_classes)

    def forward(self, x, mask):                         # x: (B, 1, L), mask: (B, L)
        m = mask[:, None, :]                            # (B, 1, L): broadcasts over the channels
        h = x
        for conv in self.convs:                         # (B, C, L): padding keeps the length
            h = F.relu(conv(h)).masked_fill(~m, 0.0)    # re-zero the padding after EVERY layer
        mean = (h * m).sum(dim=-1) / m.sum(dim=-1)      # average over real positions only
        mx = h.masked_fill(~m, float("-inf")).amax(dim=-1)   # maximum over real positions only
        return self.head(torch.cat([mean, mx], dim=1))

def run_signal_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total_loss, correct, n = 0.0, 0, 0
    with torch.set_grad_enabled(training):              # no graph during evaluation
        for x, mask, y in loader:
            x, mask, y = x.to(device), mask.to(device), y.to(device)
            logits = model(x, mask)
            loss = F.cross_entropy(logits, y)
            if training:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
            total_loss += loss.item() * len(y)
            correct += (logits.argmax(dim=1) == y).sum().item()
            n += len(y)
    return total_loss / n, correct / n

set_seed(0)
sig_model = SignalCNN().to(device)
opt = torch.optim.AdamW(sig_model.parameters(), lr=3e-3)
for epoch in range(n_epochs(8)):
    tr = run_signal_epoch(sig_model, sig_train_loader, opt)
    va = run_signal_epoch(sig_model, sig_val_loader)
    print(f"epoch {epoch + 1} | train loss {tr[0]:.4f} acc {tr[1]:.3f} | val loss {va[0]:.4f} acc {va[1]:.3f}")
record("Ch7 1-D CNN on vibration signals, val accuracy", va[1])

When this notebook was written, a CPU run climbed from about 65% validation accuracy after the first epoch to about 90% after eight; the curve is still rising, so more epochs help.

Our generic `fit()` does not fit this model, because each batch now has three parts (signal, mask, label). Writing a small dedicated loop is normal; frameworks solve it by letting you override the "training step".

**Does padding change the prediction?** It must not: a signal should get the same output alone or padded inside a batch. With the padding re-zeroed after every layer and masked pooling, the computation at the real positions is identical. Without the mask, the padded positions leak into the convolutions and the averages:

In [ ]:
sig_model.eval()
s0 = val_signals[0].to(device)
with torch.no_grad():
    alone = sig_model(s0.view(1, 1, -1), torch.ones(1, len(s0), dtype=torch.bool, device=device))
    padded = F.pad(s0, (0, 300)).view(1, 1, -1)                       # 300 zeros appended
    pad_mask = (torch.arange(len(s0) + 300, device=device) < len(s0))[None]
    with_mask = sig_model(padded, pad_mask)
    no_mask = sig_model(padded, torch.ones_like(pad_mask))             # pretend the zeros are signal
print("same logits with the mask:   ", torch.allclose(alone, with_mask, atol=1e-5))
print("difference without the mask: ", (alone - no_mask).abs().max().item())

## 🧪 Exercises

**Exercise 7.1: output sizes.** `★☆☆ · ~10 min`
Write `conv_out_size(h, k, s=1, p=0, d=1)` implementing the formula of §7.4. Then use it to compute, by hand, the spatial size after this stack applied to a 224×224 image: `Conv(k=7, s=2, p=3)` → `MaxPool(k=3, s=2, p=1)` → `Conv(k=3, s=1, p=1)` → `Conv(k=3, s=2, p=1)`. (Pooling uses the same formula.) This is the start of a ResNet.

In [ ]:
# ✍️ Your attempt here.

In [ ]:
# 🔎 Self-check for Exercise 7.1
def _test(fn):
    for h, k, s, p, d in [(28, 3, 1, 0, 1), (28, 3, 2, 1, 1), (224, 7, 2, 3, 1), (32, 3, 1, 2, 2), (15, 4, 3, 0, 1)]:
        real = nn.Conv2d(1, 1, k, stride=s, padding=p, dilation=d)(torch.zeros(1, 1, h, h)).shape[-1]
        assert fn(h, k, s, p, d) == real, f"wrong for h={h}, k={k}, s={s}, p={p}, d={d}: expected {real}"
self_check("conv_out_size", _test)

In [ ]:
# @title ✅ Solution (double-click to reveal) { display-mode: "form" }
# Solutions only execute when RUN_SOLUTIONS = True (setup cell), so "Run all" stays fast.
if RUN_SOLUTIONS:
    def conv_out_size(h, k, s=1, p=0, d=1):
        return (h + 2 * p - d * (k - 1) - 1) // s + 1

    h = 224
    for k, s, p in [(7, 2, 3), (3, 2, 1), (3, 1, 1), (3, 2, 1)]:
        h = conv_out_size(h, k, s, p)
        print(h)        # 112, 56, 56, 28
else:
    print("Solution not executed (RUN_SOLUTIONS = False).")

**Exercise 7.2: your own `conv2d`.** `★★★ · ~30 min`
Extend §7.3 into `my_conv2d(x, weight, bias, stride=1, padding=0)` for inputs `(N, C_in, H, W)` and weights `(C_out, C_in, k, k)`. `F.unfold` accepts `stride` and `padding` arguments; use `conv_out_size` (or the formula) for the output shape. It must match `F.conv2d`.

In [ ]:
# ✍️ Your attempt here.
# Hint: F.unfold(x, k, padding=padding, stride=stride) -> (N, C_in*k*k, L). Then einsum, add the bias, reshape to (N, C_out, H_out, W_out).

In [ ]:
# 🔎 Self-check for Exercise 7.2
def _test(fn):
    torch.manual_seed(0)
    for (n, c, h, w_), (o, k), s, p in [((2, 3, 9, 9), (5, 3), 1, 0), ((1, 2, 10, 7), (4, 3), 2, 1), ((3, 1, 8, 8), (2, 5), 1, 2)]:
        x, wt, b = torch.randn(n, c, h, w_), torch.randn(o, c, k, k), torch.randn(o)
        ref = F.conv2d(x, wt, b, stride=s, padding=p)
        out = fn(x, wt, b, stride=s, padding=p)
        assert out.shape == ref.shape, f"shape {tuple(out.shape)}, expected {tuple(ref.shape)}"
        assert torch.allclose(out, ref, atol=1e-4), "values differ from F.conv2d"
self_check("my_conv2d", _test)

In [ ]:
# @title ✅ Solution (double-click to reveal) { display-mode: "form" }
# Solutions only execute when RUN_SOLUTIONS = True (setup cell), so "Run all" stays fast.
if RUN_SOLUTIONS:
    def my_conv2d(x, weight, bias, stride=1, padding=0):
        n, _, h, w = x.shape
        c_out, _, k, _ = weight.shape
        h_out = (h + 2 * padding - k) // stride + 1
        w_out = (w + 2 * padding - k) // stride + 1
        cols = F.unfold(x, k, padding=padding, stride=stride)                  # (N, C_in*k*k, h_out*w_out)
        out = torch.einsum("ok,nkl->nol", weight.reshape(c_out, -1), cols)     # (N, C_out, h_out*w_out)
        return (out + bias.view(1, -1, 1)).reshape(n, c_out, h_out, w_out)
else:
    print("Solution not executed (RUN_SOLUTIONS = False).")

**Exercise 7.3: masked mean.** `★★☆ · ~15 min`
Write `masked_mean(h, mask)` for features `h` of shape `(B, C, L)` and a boolean `mask` of shape `(B, L)`, returning the `(B, C)` mean over the positions where `mask` is `True`. Use `torch.where` (Notebook 1, §1.6) instead of multiplying by the mask, so that even `inf` or `nan` values at padded positions cannot leak into the result.

In [ ]:
# ✍️ Your attempt here.
# Hint: torch.where(mask[:, None, :], h, 0.0).sum(-1), divided by the number of True values per row (shape (B, 1)).

In [ ]:
# 🔎 Self-check for Exercise 7.3
def _test(fn):
    h = torch.randn(3, 4, 6)
    mask = torch.tensor([[1, 1, 1, 0, 0, 0], [1, 1, 1, 1, 1, 1], [1, 0, 0, 0, 0, 0]], dtype=torch.bool)
    h_bad = h.clone(); h_bad[~mask[:, None, :].expand_as(h)] = float("nan")   # garbage in the padding
    out = fn(h_bad, mask)
    assert out.shape == (3, 4), f"shape {tuple(out.shape)}, expected (3, 4)"
    ref = torch.stack([h[b][:, mask[b]].mean(dim=1) for b in range(3)])
    assert torch.allclose(out, ref, atol=1e-6), "wrong values (or padded values leaked in)"
self_check("masked_mean", _test)

In [ ]:
# @title ✅ Solution (double-click to reveal) { display-mode: "form" }
# Solutions only execute when RUN_SOLUTIONS = True (setup cell), so "Run all" stays fast.
if RUN_SOLUTIONS:
    def masked_mean(h, mask):
        m = mask[:, None, :]                                   # (B, 1, L)
        total = torch.where(m, h, torch.zeros((), dtype=h.dtype)).sum(dim=-1)
        return total / m.sum(dim=-1)                           # (B, C) / (B, 1)

    print(masked_mean(torch.ones(2, 3, 4), torch.tensor([[1, 1, 0, 0], [1, 1, 1, 1]], dtype=torch.bool)))
    # Why not (h * mask).sum()? Because nan * 0 = nan: multiplying does not remove garbage, where() does.
else:
    print("Solution not executed (RUN_SOLUTIONS = False).")

### 📌 Key takeaways
- A convolution is a dot product of every local patch with every filter: `unfold` + `einsum` reproduces `F.conv2d` exactly.
- Weight shape `(C_out, C_in, k, k)`; parameters do not depend on image size; output size follows one formula.
- Conv → BatchNorm → ReLU blocks; halve the resolution and double the channels; finish with global average pooling and a linear layer.
- Built-in locality and weight sharing let a CNN beat a larger MLP.
- Variable-length inputs: pad in a `collate_fn`, carry a mask, and pool only over real positions (`masked_fill`, `where`).

### ✔️ Check yourself

Commit to an answer (say it out loud or write it down) *before* you expand it.

**1.** How many parameters does `nn.Conv2d(64, 128, 3)` have? And with `bias=False`?

<details>
<summary>Answer</summary>

`128·64·3·3 + 128 = 73,856`; without bias, 73,728.

</details>

**2.** An input of 32×32 goes through `Conv2d(k=3, s=2, p=1)`. What is the output size?

<details>
<summary>Answer</summary>

`floor((32 + 2 − 2 − 1)/2) + 1 = 16`.

</details>

**3.** Why do convolutions followed by BatchNorm usually use `bias=False`?

<details>
<summary>Answer</summary>

BatchNorm subtracts the per-channel mean, which removes any constant bias, and then adds its own learnable shift. A conv bias would be redundant parameters.

</details>

**4.** What does `nn.AdaptiveAvgPool2d(1)` do, and why is it useful before the classifier?

<details>
<summary>Answer</summary>

It averages each channel over all spatial positions, giving `(N, C, 1, 1)`. The classifier then has a fixed input size `C` regardless of the image resolution, with very few parameters.

</details>

---
# Chapter 8. Transfer learning: freezing layers and fine-tuning

🎯 **In this chapter you will learn to:** load pretrained models from `torchvision.models` with the matching preprocessing; replace a model's classification head; freeze and unfreeze layers correctly (including the BatchNorm trap); choose label-preserving data augmentation; train a linear probe on cached features; fine-tune all or part of a network with different learning rates per part; and judge which strategy fits your data.

🔁 **Starting a fresh session here?** Run the two Chapter 0 cells, then the hidden restore-point cell below. It re-creates the tools from earlier chapters that Chapter 8 uses. In a normal top-to-bottom run it does nothing.

In [ ]:
# @title 🔁 Restore point for Chapter 8 (needed only if you start a fresh session here) { display-mode: "form" }
if "device" not in globals():
    raise RuntimeError("Run the Chapter 0 setup cells first, then this cell.")
if "get_trained_cnn" not in globals():
    from torch.optim import lr_scheduler as S

    def param_groups_weight_decay(model, weight_decay):
        decay, no_decay = [], []
        for name, p in model.named_parameters():
            if not p.requires_grad:
                continue                                     # frozen parameters: leave them out
            (no_decay if p.ndim <= 1 else decay).append(p)   # biases and norm weights are 1-D
        return [{"params": decay, "weight_decay": weight_decay},
                {"params": no_decay, "weight_decay": 0.0}]

    class EarlyStopping:
        # Stop when `metric` has not improved by more than `min_delta` for `patience` epochs.
        # mode="min" for losses, mode="max" for accuracies. Keeps a copy of the best weights.
        def __init__(self, patience=5, min_delta=0.0, mode="min"):
            self.patience, self.min_delta, self.mode = patience, min_delta, mode
            self.best = math.inf if mode == "min" else -math.inf
            self.best_state, self.best_epoch, self.bad_epochs, self.epoch = None, 0, 0, 0

        def improved(self, value):
            if self.mode == "min":
                return value < self.best - self.min_delta
            return value > self.best + self.min_delta

        def step(self, value, model):
            # Call once per epoch. Returns True when training should stop.
            self.epoch += 1
            if self.improved(value):
                self.best, self.best_epoch, self.bad_epochs = value, self.epoch, 0
                self.best_state = copy.deepcopy(model.state_dict())
            else:
                self.bad_epochs += 1
            return self.bad_epochs >= self.patience

        def restore(self, model):
            model.load_state_dict(self.best_state)

    import contextlib

    def train_one_epoch(model, loader, loss_fn, optimizer, scheduler=None, max_grad_norm=None,
                        averaged_model=None, amp_dtype=None, scaler=None):
        # amp_dtype / scaler: mixed precision, explained in Chapter 11. Leave them as None until then.
        model.train()
        total_loss = torch.zeros((), device=device)                  # running sums stay ON the device:
        correct = torch.zeros((), device=device, dtype=torch.long)   # no CPU-GPU synchronisation per step
        n, grad_norms = 0, []
        for x, y in loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            amp = torch.autocast(device_type=device.type, dtype=amp_dtype) if amp_dtype else contextlib.nullcontext()
            with amp:
                logits = model(x)
                loss = loss_fn(logits, y)
            optimizer.zero_grad(set_to_none=True)
            if scaler is not None:                                   # float16 mixed precision (Chapter 11)
                scaler.scale(loss).backward()
                if max_grad_norm is not None:
                    scaler.unscale_(optimizer)                       # clip the REAL gradients, not the scaled ones
            else:
                loss.backward()
            if max_grad_norm is not None:
                grad_norms.append(nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm))
            if scaler is not None:
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()
            if scheduler is not None:                                # per-batch schedules (OneCycleLR, warm-up)
                scheduler.step()
            if averaged_model is not None:                           # weight averaging (Chapter 6)
                averaged_model.update_parameters(model)
            total_loss += loss.detach().float() * len(y)
            correct += (logits.argmax(dim=1) == y).sum()
            n += len(y)
        mean_norm = torch.stack(grad_norms).mean().item() if grad_norms else float("nan")
        return total_loss.item() / n, correct.item() / n, mean_norm   # one synchronisation per epoch


    def fit(model, train_loader, val_loader, optimizer, epochs, loss_fn=nn.CrossEntropyLoss(),
            scheduler=None, scheduler_interval="batch", early_stopping=None, eval_model=None, verbose=True, **train_kw):
        # scheduler_interval: "batch", "epoch" or "plateau" (ReduceLROnPlateau).
        # eval_model: evaluate (and select) a different model than the one trained, e.g. an EMA copy (Chapter 6).
        history = {k: [] for k in ["train_loss", "train_acc", "val_loss", "val_acc", "lr", "grad_norm", "time"]}
        eval_model = eval_model if eval_model is not None else model
        for epoch in range(1, epochs + 1):
            t0 = time.perf_counter()
            lr = optimizer.param_groups[0]["lr"]
            tr_loss, tr_acc, gnorm = train_one_epoch(model, train_loader, loss_fn, optimizer,
                                                     scheduler if scheduler_interval == "batch" else None, **train_kw)
            va_loss, va_acc = evaluate(eval_model, val_loader, loss_fn)
            if scheduler is not None and scheduler_interval == "epoch":
                scheduler.step()
            elif scheduler is not None and scheduler_interval == "plateau":
                scheduler.step(va_loss)
            elapsed = time.perf_counter() - t0
            for k, v in zip(history, (tr_loss, tr_acc, va_loss, va_acc, lr, gnorm, elapsed)):
                history[k].append(v)
            if verbose:
                print(f"epoch {epoch:2d} | lr {lr:.2e} | train {tr_loss:.4f} / {tr_acc:.4f} | "
                      f"val {va_loss:.4f} / {va_acc:.4f} | {elapsed:4.1f}s")
            if early_stopping is not None and early_stopping.step(va_loss, eval_model):
                print(f"early stopping: no improvement for {early_stopping.patience} epochs")
                break
        if early_stopping is not None:                       # select the epoch with the best validation LOSS...
            early_stopping.restore(eval_model)
            sel_epoch = early_stopping.best_epoch
        else:
            sel_epoch = len(history["val_loss"])            # ...or keep the last epoch
        sel_loss, sel_acc = history["val_loss"][sel_epoch - 1], history["val_acc"][sel_epoch - 1]
        history["selected"] = {"epoch": sel_epoch, "val_loss": sel_loss, "val_acc": sel_acc}
        if verbose:
            print(f"selected model: epoch {sel_epoch}, val loss {sel_loss:.4f}, val acc {sel_acc:.4f}")
        return history


    def plot_history(histories, keys=("val_loss", "val_acc", "lr")):
        fig, axes = plt.subplots(1, len(keys), figsize=(5 * len(keys), 3.2))
        for ax, key in zip(axes, keys):
            for name, h in histories.items():
                ax.plot(range(1, len(h[key]) + 1), h[key], "o-", ms=3, label=name)
            ax.set_title(key); ax.set_xlabel("epoch")
        axes[0].legend(fontsize=8); plt.tight_layout(); plt.show()

    # Small in-memory subsets (normalised exactly like fm_transform) for quick experiments
    def _fm_subset(idx):
        x = ((raw_train.data[idx].float() / 255 - FM_MEAN) / FM_STD).unsqueeze(1)
        return TensorDataset(x, raw_train.targets[idx])

    small_ds = _fm_subset(FM_TRAIN_IDX[:4096])      # 4,096 training images
    small_val = _fm_subset(FM_VAL_IDX[:2000])       # 2,000 validation images

    def save_checkpoint(path, model, optimizer, scheduler=None, epoch=0, best_metric=None,
                        loader_generator=None, config=None):
        np_state = np.random.get_state()
        ckpt = {
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict() if scheduler is not None else None,
            "epoch": epoch,
            "best_metric": best_metric,
            "config": config or {},
            "rng": {
                "torch": torch.get_rng_state(),
                "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [],
                "python": random.getstate(),
                # NumPy's state contains an array; store it as a list so the safe loader accepts it
                "numpy": (np_state[0], np_state[1].tolist(), *np_state[2:]),
            },
            "loader_generator": loader_generator.get_state() if loader_generator is not None else None,
        }
        tmp = Path(str(path) + ".tmp")
        torch.save(ckpt, tmp)             # write to a temporary file first...
        os.replace(tmp, path)             # ...then rename: an interruption can never leave a half-written checkpoint


    def load_checkpoint(path, model, optimizer=None, scheduler=None, loader_generator=None):
        ckpt = torch.load(path, map_location=device, weights_only=True)   # safe loading: tensors and plain data only
        model.load_state_dict(ckpt["model"])
        if optimizer is not None:
            optimizer.load_state_dict(ckpt["optimizer"])
        if scheduler is not None and ckpt["scheduler"] is not None:
            scheduler.load_state_dict(ckpt["scheduler"])
        rng = ckpt["rng"]
        torch.set_rng_state(rng["torch"].cpu())
        if torch.cuda.is_available() and rng["cuda"]:
            torch.cuda.set_rng_state_all([s.cpu() for s in rng["cuda"]])
        random.setstate(rng["python"])
        name, keys, *rest = rng["numpy"]
        np.random.set_state((name, np.array(keys, dtype=np.uint32), *rest))
        if loader_generator is not None and ckpt["loader_generator"] is not None:
            loader_generator.set_state(ckpt["loader_generator"].cpu())
        return ckpt

    def conv_bn_relu(c_in, c_out):
        return nn.Sequential(nn.Conv2d(c_in, c_out, 3, padding=1, bias=False), nn.BatchNorm2d(c_out), nn.ReLU(inplace=True))


    class SmallCNN(nn.Module):
        def __init__(self, num_classes=10, width=32, p_drop=0.3):
            super().__init__()
            w = width
            self.features = nn.Sequential(
                conv_bn_relu(1, w), conv_bn_relu(w, w), nn.MaxPool2d(2),              # 28x28 -> 14x14
                conv_bn_relu(w, 2 * w), conv_bn_relu(2 * w, 2 * w), nn.MaxPool2d(2),  # 14x14 -> 7x7
                conv_bn_relu(2 * w, 4 * w),                                           # 7x7, 128 channels
                nn.AdaptiveAvgPool2d(1),                                              # -> 1x1
            )
            self.head = nn.Sequential(nn.Flatten(), nn.Dropout(p_drop), nn.Linear(4 * w, num_classes))

        def forward(self, x):
            return self.head(self.features(x))

    CNN_PATH = CKPT_DIR / "smallcnn_fashion.pt"

    def train_small_cnn(epochs=12, seed=0):
        # The recipe of Section 7.7: AdamW with weight-decay groups and a OneCycle schedule.
        set_seed(seed)
        model = SmallCNN().to(device)
        E = n_epochs(epochs)
        opt = torch.optim.AdamW(param_groups_weight_decay(model, 0.05), lr=1e-3)
        sched = S.OneCycleLR(opt, max_lr=3e-3, total_steps=E * len(train_loader))
        hist = fit(model, train_loader, val_loader, opt, E, scheduler=sched,
                   early_stopping=EarlyStopping(patience=100))        # patience=100: just keep the best epoch
        torch.save({"state_dict": model.state_dict()}, CNN_PATH)
        return model, hist

    def get_trained_cnn():
        # Load the CNN trained in Chapter 7 if it was saved; otherwise train it now.
        model = SmallCNN().to(device)
        if CNN_PATH.exists():
            model.load_state_dict(torch.load(CNN_PATH, map_location=device, weights_only=True)["state_dict"])
            print(f"loaded the trained SmallCNN from {CNN_PATH}")
        else:
            print("no saved SmallCNN found: training one with the recipe of Section 7.7 ...")
            model, _ = train_small_cnn()
        return model.eval()
    print("Restored the tools defined in earlier chapters.")

## 8.1 Why it works, and the strategies
A network trained on ImageNet (1.3 million photos, 1,000 classes) has learned a hierarchy of features: edges and colours in the first layers, textures and parts in the middle, object-like patterns at the end. Most of these features are useful for **other** image tasks too. Starting from them instead of from random weights lets you reach high accuracy with a few hundred or thousand labelled images, which is the normal situation in industry.

| Strategy | What trains | When to use it |
| --- | --- | --- |
| **A. Linear probe** (feature extraction) | Only a new final layer; the backbone is frozen | Very little data; the new images resemble the pretraining images; fastest |
| **B. Full fine-tuning** | Everything, with a small learning rate for the backbone | More data, or images that differ from the pretraining domain; usually the most accurate |
| **Partial fine-tuning** | The last block(s) and the head | In between; limits overfitting and memory |
| **C. From scratch** | Everything, from random weights | Lots of data in a domain very unlike the pretraining data (e.g. medical scans, spectrograms) |

We use **Flowers-102**: 102 flower species, with only **10 training images per class** (1,020 in total), 1,020 for validation and 6,149 for testing. It is a realistic small-data problem.

## 8.2 Pretrained models in `torchvision.models`
Each architecture comes with a **weights enum** that bundles the weights, the class names and, crucially, the **preprocessing** they were trained with. Using different preprocessing (say, a different normalisation) silently degrades a pretrained model.

In [ ]:
from torchvision.models import resnet18, ResNet18_Weights

weights = ResNet18_Weights.DEFAULT                  # the best available ImageNet weights for this architecture
backbone = resnet18(weights=weights)                # downloads about 45 MB the first time
print("preprocessing these weights expect:\n", weights.transforms())
print("first ImageNet classes:", weights.meta["categories"][:5])
print("\ntop-level children:", [name for name, _ in backbone.named_children()])
print("the classification head:", backbone.fc)

ResNet-18 is a stem (`conv1`, `bn1`, `relu`, `maxpool`), four stages of residual blocks (`layer1` to `layer4`), global average pooling (`avgpool`) and a linear head `fc` from 512 features to 1,000 ImageNet classes. To use it for our task, we **replace the head** with a new layer that has 102 outputs. Other architectures name their head differently (`classifier` for EfficientNet and MobileNet, `heads` for vision transformers), so always print the model first.

🏭 You will also see the generic API `torchvision.models.get_model("resnet18", weights="DEFAULT")`, and the separate `timm` library, which offers far more pretrained vision models and is common in research code.

## 8.3 The data
The images have different sizes, so the transforms resize them to 224×224, the resolution the weights were trained at. The training transform adds **augmentation** (random crops and flips), which creates new variations of the few training images. The evaluation transform is deterministic. Both use the **ImageNet normalisation** statistics, because that is what the pretrained weights expect.

In [ ]:
IMAGENET_MEAN, IMAGENET_STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
flower_train_tf = T.Compose([
    T.RandomResizedCrop(224, scale=(0.5, 1.0)),       # random crop covering 50-100% of the image, resized to 224
    T.RandomHorizontalFlip(),
    T.ToImage(), T.ToDtype(torch.float32, scale=True), T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
flower_eval_tf = T.Compose([
    T.Resize(256), T.CenterCrop(224),                  # the standard ImageNet evaluation recipe
    T.ToImage(), T.ToDtype(torch.float32, scale=True), T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

def flowers(split, tf):
    return maybe_subset(datasets.Flowers102(DATA_DIR, split=split, transform=tf, download=True), 128)

fl_train, fl_train_eval = flowers("train", flower_train_tf), flowers("train", flower_eval_tf)   # same images, two transforms
fl_val, fl_test = flowers("val", flower_eval_tf), flowers("test", flower_eval_tf)
kw = dict(num_workers=NUM_WORKERS, pin_memory=PIN)
fl_train_loader = DataLoader(fl_train, batch_size=32, shuffle=True, drop_last=True, **kw)
fl_val_loader = DataLoader(fl_val, batch_size=128, **kw)
fl_test_loader = DataLoader(fl_test, batch_size=128, **kw)
print(f"train {len(fl_train)} | val {len(fl_val)} | test {len(fl_test)}  (the download is about 330 MB the first time)")

raw_flowers = datasets.Flowers102(DATA_DIR, split="train")
fig, axes = plt.subplots(1, 6, figsize=(12, 2.4))
for ax, i in zip(axes, range(0, 60, 10)):
    im, lab = raw_flowers[i]
    ax.imshow(im); ax.set_title(f"class {lab}", fontsize=8); ax.axis("off")
plt.tight_layout(); plt.show()

## 8.4 Data augmentation
**Augmentation** applies random, label-preserving changes to each training image every time it is loaded, so every epoch sees different versions of the same few photos. It effectively enlarges a small dataset and teaches the model invariances (a flower is still the same flower when shifted, zoomed or mirrored). Four rules:

1. **Training split only.** Validation and test use the deterministic transform; otherwise their scores become noisy and biased.
2. **The label must survive the change.** Horizontal flips are fine for flowers but wrong for handwritten digits (a mirrored 2 is not a 2) or text. Vertical flips are wrong for natural scenes but fine for microscopy or satellite tiles. In industrial inspection, a flip can turn a correctly assembled part into an incorrect one.
3. **Imitate the variation expected in deployment**: lighting, position, scale, sensor noise. Augmentations that never occur in practice mostly waste capacity.
4. **Geometry moves annotations too.** For detection or segmentation, boxes and masks must be transformed together with the image; the `torchvision.transforms.v2` API does this when you pass them along.

Here is what our training transform produces from one photo, compared with the evaluation transform:

In [ ]:
img0, lab0 = raw_flowers[0]
m_t, s_t = torch.tensor(IMAGENET_MEAN).view(3, 1, 1), torch.tensor(IMAGENET_STD).view(3, 1, 1)
show = lambda t: (t * s_t + m_t).clamp(0, 1).permute(1, 2, 0)        # undo the normalisation for display

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
axes[0, 0].imshow(img0); axes[0, 0].set_title("original", fontsize=8)
axes[1, 0].imshow(show(flower_eval_tf(img0))); axes[1, 0].set_title("evaluation transform", fontsize=8)
for ax in list(axes[0, 1:]) + list(axes[1, 1:]):
    ax.imshow(show(flower_train_tf(img0))); ax.set_title("training transform", fontsize=8)
for ax in axes.flat:
    ax.axis("off")
plt.tight_layout(); plt.show()

Stronger policies exist as ready-made transforms: `T.TrivialAugmentWide()` and `T.RandAugment()` apply randomly chosen colour and geometry operations. **MixUp** and **CutMix** work on whole batches: they blend pairs of images (or paste a patch of one into another) and mix their labels in the same proportion, so the targets become class probabilities. `CrossEntropyLoss` accepts such soft targets directly (Notebook 1, §5.4).

In [ ]:
xb, yb = next(iter(fl_train_loader))
mixup = T.MixUp(alpha=0.2, num_classes=102)
xm, ym = mixup(xb, yb)
print("images:", tuple(xm.shape), "| targets became soft labels:", tuple(ym.shape))
print("each row of the targets sums to 1:", torch.allclose(ym.sum(dim=1), torch.ones(len(ym))))
print("loss with soft targets:", F.cross_entropy(torch.randn(len(ym), 102), ym).item())

## 8.5 Freezing, and the BatchNorm trap
**Freezing** a parameter means excluding it from training. Set `requires_grad_(False)`: autograd then computes no gradient for it, which also saves memory and time. Two rules go with it:

1. **Give the optimizer only the trainable parameters.** (Frozen parameters have no gradient, so they would be skipped anyway, but filtering makes the intent explicit and avoids surprises with weight decay and some optimizers.)
2. **A newly assigned layer is trainable**, because it is freshly created. So freeze first, then replace the head.

In [ ]:
def freeze(module):
    for p in module.parameters():
        p.requires_grad_(False)

def make_resnet(num_classes=102, pretrained=True):
    m = resnet18(weights=ResNet18_Weights.DEFAULT if pretrained else None)
    m.fc = nn.Linear(m.fc.in_features, num_classes)       # new head: 512 -> 102, trainable
    return m

model = make_resnet()
freeze(model)                      # freeze EVERYTHING...
for p in model.fc.parameters():    # ...then unfreeze the head (or: freeze first, assign the new head afterwards)
    p.requires_grad_(True)
print(f"trainable: {count_params(model):,} of {count_params(model, trainable_only=False):,} parameters")

model = model.to(device)
opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-3)

# Does a training step leave the frozen weights alone?
conv1_before = model.conv1.weight.clone()
bn_mean_before = model.bn1.running_mean.clone()
model.train()
x, y = next(iter(fl_train_loader))
loss = F.cross_entropy(model(x.to(device)), y.to(device))
opt.zero_grad(); loss.backward(); opt.step()
print("frozen conv1 weights unchanged:   ", torch.equal(conv1_before, model.conv1.weight))
print("frozen bn1 running_mean unchanged:", torch.equal(bn_mean_before, model.bn1.running_mean))

⚠️ **The BatchNorm trap.** The convolution weights stayed fixed, but BatchNorm's **running statistics changed**, even though every parameter was frozen. Running statistics are buffers, not parameters: `requires_grad` does not affect them, and in `train()` mode BatchNorm updates them on every forward pass. With small batches of a new dataset, this slowly overwrites the carefully estimated ImageNet statistics and can noticeably hurt a "frozen" model.

**Fix:** after every `model.train()`, put the frozen part back into `eval()` mode. Since our `train_one_epoch` calls `model.train()` itself, the cleanest solution is a small wrapper module whose `train()` method keeps the frozen BatchNorm layers in evaluation mode:

In [ ]:
BATCHNORMS = (nn.BatchNorm1d, nn.BatchNorm2d, nn.BatchNorm3d, nn.SyncBatchNorm)

class FrozenBNWrapper(nn.Module):
    # Wraps a model; whenever .train() is called, BatchNorm layers whose parameters are frozen stay in eval mode.
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x):
        return self.model(x)

    def train(self, mode=True):
        super().train(mode)
        for m in self.model.modules():
            if isinstance(m, BATCHNORMS) and not any(p.requires_grad for p in m.parameters()):
                m.eval()
        return self

wrapped = FrozenBNWrapper(model)
bn_mean_before = model.bn1.running_mean.clone()
wrapped.train()
with torch.no_grad():
    wrapped(x.to(device))
print("bn1 running_mean unchanged with the wrapper:", torch.equal(bn_mean_before, model.bn1.running_mean))
print("head is still in training mode:", model.fc.training, "| bn1 in training mode:", model.bn1.training)

⚠️ **Wrapping changes the names.** Because the network now lives in the attribute `model`, every key of `wrapped.state_dict()` starts with `"model."`. A file saved from the wrapper will not load into a plain ResNet (the same problem as the `"module."` prefix of Chapter 5). Save the inner network instead:

In [ ]:
print("wrapper keys:", list(wrapped.state_dict())[:2])
print("inner keys:  ", list(wrapped.model.state_dict())[:2])
torch.save(wrapped.model.state_dict(), CKPT_DIR / "resnet18_flowers_head.pt")   # loads into make_resnet() directly

## 8.6 Strategy A: a linear probe on cached features
When the backbone is completely frozen and there is no augmentation, its output for each image never changes. So compute the 512-dimensional features **once**, store them, and train only the linear head on those stored features: hundreds of epochs then take seconds. Replacing `fc` with `nn.Identity()` turns the network into a feature extractor.

In [ ]:
@torch.no_grad()
def extract_features(backbone, dataset, batch_size=128):
    backbone.eval()
    feats, labels = [], []
    for x, y in DataLoader(dataset, batch_size=batch_size, num_workers=NUM_WORKERS):
        feats.append(backbone(x.to(device)).cpu()); labels.append(y)
    return torch.cat(feats), torch.cat(labels)

extractor = resnet18(weights=ResNet18_Weights.DEFAULT)
extractor.fc = nn.Identity()                                    # output: the 512-d pooled features
extractor = extractor.to(device)
t0 = time.perf_counter()
f_train, y_train = extract_features(extractor, fl_train_eval)   # no augmentation: features are fixed
f_val, y_val = extract_features(extractor, fl_val)
print(f"features: {tuple(f_train.shape)} train, {tuple(f_val.shape)} val, extracted in {time.perf_counter() - t0:.0f}s")

set_seed(0)
probe = nn.Linear(512, 102).to(device)
opt = torch.optim.AdamW(probe.parameters(), lr=1e-3, weight_decay=1e-2)
feat_loader = DataLoader(TensorDataset(f_train, y_train), batch_size=128, shuffle=True)
feat_val_loader = DataLoader(TensorDataset(f_val, y_val), batch_size=512)
hist_probe = fit(probe, feat_loader, feat_val_loader, opt, n_epochs(100), verbose=False,
                 early_stopping=EarlyStopping(patience=15))
probe_acc = evaluate(probe, feat_val_loader)[1]
print(f"linear probe validation accuracy: {probe_acc:.4f}")

With only 10 images per class, a linear layer on frozen ImageNet features already reaches a high validation accuracy; values roughly in the 75–85% range are typical for this setup (record yours). That is the power of pretrained features.

## 8.7 Strategy B: full fine-tuning with discriminative learning rates
Now train the whole network, with augmentation. The new head is random, while the backbone is already good, so we use **two parameter groups**: a small learning rate for the backbone (we only want to adapt it gently) and a 10× larger one for the head. A short warm-up protects the pretrained weights from the large, random gradients of the untrained head in the first steps.

In [ ]:
set_seed(0)
ft_model = make_resnet().to(device)
head_params = list(ft_model.fc.parameters())
backbone_params = [p for n, p in ft_model.named_parameters() if not n.startswith("fc.")]
opt = torch.optim.AdamW([
    {"params": backbone_params, "lr": 1e-4},     # gentle: adapt the pretrained features
    {"params": head_params, "lr": 1e-3},         # 10x larger: the head starts from scratch
], weight_decay=1e-2)
FT_EPOCHS = n_epochs(15)
total = FT_EPOCHS * len(fl_train_loader)
sched = S.LambdaLR(opt, lambda s: min(1.0, (s + 1) / 30) * 0.5 * (1 + math.cos(math.pi * s / total)))  # warm-up 30 steps + cosine
hist_ft = fit(ft_model, fl_train_loader, fl_val_loader, opt, FT_EPOCHS, scheduler=sched,
              early_stopping=EarlyStopping(patience=100))

`LambdaLR` multiplies **each group's own** base learning rate by the same factor, so the 10× ratio between head and backbone is kept throughout the schedule.

Full fine-tuning usually beats the linear probe by several points (typical values are roughly 85–90%), because the backbone adapts its features to flowers. On a problem this small, it can also overfit; the augmentation, weight decay and small backbone learning rate are what keep it in check.

## 8.8 Partial fine-tuning
In between the two: freeze the early stages, which hold generic edge and texture detectors, and train only the last stage and the head. Fewer trainable parameters means less overfitting, less memory and faster steps.

In [ ]:
partial = make_resnet()
freeze(partial)
for module in (partial.layer4, partial.fc):          # unfreeze only the last stage and the head
    for p in module.parameters():
        p.requires_grad_(True)
partial = FrozenBNWrapper(partial).to(device)        # keep frozen BatchNorm layers in eval mode
print(f"trainable: {count_params(partial):,} of {count_params(partial, trainable_only=False):,} parameters")
for name, child in partial.model.named_children():
    n_train = sum(p.numel() for p in child.parameters() if p.requires_grad)
    print(f"  {name:8s} {'trainable' if n_train else 'frozen':9s}")

Train it with the same schedule and number of epochs as full fine-tuning, using the same two learning rates (small for the unfrozen stage, larger for the head), so the comparison changes only *what is trainable*:

In [ ]:
set_seed(0)
opt = torch.optim.AdamW([
    {"params": partial.model.layer4.parameters(), "lr": 1e-4},
    {"params": partial.model.fc.parameters(), "lr": 1e-3},
], weight_decay=1e-2)
sched = S.LambdaLR(opt, lambda s: min(1.0, (s + 1) / 30) * 0.5 * (1 + math.cos(math.pi * s / total)))
t0 = time.perf_counter()
hist_partial = fit(partial, fl_train_loader, fl_val_loader, opt, FT_EPOCHS, scheduler=sched,
                   early_stopping=EarlyStopping(patience=100))
print(f"partial fine-tuning took {time.perf_counter() - t0:.0f} s; full fine-tuning took {sum(hist_ft['time']):.0f} s")

A related schedule, **gradual unfreezing**, trains the head first, then unfreezes one more stage every few epochs, from the top down. Whatever you unfreeze, the BatchNorm question (§8.5) must be answered for the frozen part.

## 8.9 Comparing the strategies on the test set
We chose the strategies' settings on the validation set; now evaluate each once on the test set.

In [ ]:
f_test, y_test = extract_features(extractor, fl_test)
results = {
    "A. linear probe (frozen)": evaluate(probe, DataLoader(TensorDataset(f_test, y_test), batch_size=512))[1],
    "partial fine-tuning (layer4 + head)": evaluate(partial, fl_test_loader)[1],
    "B. full fine-tuning": evaluate(ft_model, fl_test_loader)[1],
}
for name, acc in results.items():
    print(f"{name:36s} test accuracy {acc:.4f}")
    record(f"Ch8 {name}, test accuracy", acc)

## 🧪 Exercises

**Exercise 8.1: Strategy C, from scratch.** `★★☆ · ~15 min (mostly waiting)`
Train `make_resnet(pretrained=False)` with exactly the recipe of §8.7 (use one learning rate, `1e-3`, for everything) and compare its test accuracy with the two strategies above. What does the result tell you about how much the pretrained features were worth?

In [ ]:
# ✍️ Your attempt here.

In [ ]:
# @title ✅ Solution (double-click to reveal) { display-mode: "form" }
# Solutions only execute when RUN_SOLUTIONS = True (setup cell), so "Run all" stays fast.
if RUN_SOLUTIONS:
    set_seed(0)
    scratch = make_resnet(pretrained=False).to(device)
    opt = torch.optim.AdamW(scratch.parameters(), lr=1e-3, weight_decay=1e-2)
    total = FT_EPOCHS * len(fl_train_loader)
    sched = S.LambdaLR(opt, lambda s: min(1.0, (s + 1) / 30) * 0.5 * (1 + math.cos(math.pi * s / total)))
    fit(scratch, fl_train_loader, fl_val_loader, opt, FT_EPOCHS, scheduler=sched, early_stopping=EarlyStopping(patience=100))
    results["C. from scratch"] = evaluate(scratch, fl_test_loader)[1]
    for name, acc in results.items():
        print(f"{name:28s} test accuracy {acc:.4f}")
    # Typically far lower (often well below 50%): 1,020 images cannot teach an 11M-parameter network
    # good visual features from nothing. With small data, pretraining is worth tens of percentage points.
else:
    print("Solution not executed (RUN_SOLUTIONS = False).")

**Exercise 8.2: a head replacer for any architecture.** `★★☆ · ~20 min`
Write `replace_head(model, num_classes)` that finds the **last `nn.Linear` layer** anywhere in the model, replaces it with a new `nn.Linear(in_features, num_classes)`, and returns the model. It must work for `resnet18` (`model.fc`), `mobilenet_v3_small` and `efficientnet_b0` (inside `model.classifier`). Use `weights=None` to test without downloads.

In [ ]:
# ✍️ Your attempt here.
# Hint: Loop over model.named_modules() to find the name of the last Linear; split the name at the last '.' to get the parent module, then setattr(parent, child_name, new_layer). For Sequential parents the child name is a digit, and setattr still works.

In [ ]:
# 🔎 Self-check for Exercise 8.2
def _test(fn):
    from torchvision import models
    for builder in (models.resnet18, models.mobilenet_v3_small, models.efficientnet_b0):
        m = fn(builder(weights=None), 7)
        out = m.eval()(torch.randn(1, 3, 64, 64))
        assert out.shape == (1, 7), f"{builder.__name__}: output shape {tuple(out.shape)}, expected (1, 7)"
self_check("replace_head", _test)

In [ ]:
# @title ✅ Solution (double-click to reveal) { display-mode: "form" }
# Solutions only execute when RUN_SOLUTIONS = True (setup cell), so "Run all" stays fast.
if RUN_SOLUTIONS:
    def replace_head(model, num_classes):
        last_name = None
        for name, module in model.named_modules():
            if isinstance(module, nn.Linear):
                last_name = name
        parent_name, _, child_name = last_name.rpartition(".")
        parent = model.get_submodule(parent_name) if parent_name else model
        old = getattr(parent, child_name)
        setattr(parent, child_name, nn.Linear(old.in_features, num_classes))
        return model

    from torchvision import models
    print(replace_head(models.mobilenet_v3_small(weights=None), 7).classifier)
else:
    print("Solution not executed (RUN_SOLUTIONS = False).")

### 📌 Key takeaways
- Use the weights' own preprocessing (`weights.transforms()`, ImageNet mean/std).
- Replace the head; freeze with `requires_grad_(False)`; pass only trainable parameters to the optimizer.
- Frozen BatchNorm layers must be kept in `eval()` mode, or their running statistics drift.
- Linear probe on cached features: fast and strong with tiny data. Full fine-tuning: smaller backbone learning rate, warm-up, augmentation.

### ✔️ Check yourself

Commit to an answer (say it out loud or write it down) *before* you expand it.

**1.** You freeze a ResNet with `requires_grad_(False)`, train a new head, and find the backbone's predictions on the old task have changed. The weights are identical. What changed?

<details>
<summary>Answer</summary>

BatchNorm running statistics (buffers), which are updated in `train()` mode regardless of `requires_grad`. Keep frozen BatchNorm layers in `eval()` mode.

</details>

**2.** Why use a smaller learning rate for the pretrained backbone than for the new head?

<details>
<summary>Answer</summary>

The backbone is already near a good solution and should only be adjusted gently; a large learning rate (especially early, when the random head sends large, noisy gradients) can destroy the pretrained features. The head starts from scratch and needs larger steps.

</details>

**3.** Why can you cache features for the linear probe, but not when fine-tuning the whole network with augmentation?

<details>
<summary>Answer</summary>

Caching requires the backbone output for each image to be fixed. That holds only when the backbone is frozen (and in eval mode) and the inputs are not randomly augmented.

</details>

**4.** A colleague normalises Flowers images with their own dataset mean/std instead of the ImageNet statistics before using a pretrained ResNet. Is that wrong?

<details>
<summary>Answer</summary>

It is a mismatch: the pretrained filters and BatchNorm statistics expect ImageNet-normalised inputs. Fine-tuning can partly adapt, but the linear-probe features in particular degrade. Use the statistics the weights were trained with.

</details>

---
# Chapter 9. Evaluation beyond accuracy, and regression

🎯 **In this chapter you will learn to:** see why accuracy misleads on imbalanced data; compute precision, recall, F1, PR and ROC curves from scratch; choose a decision threshold on the validation set; compare remedies for class imbalance; report per-class and macro metrics; and build a regression surrogate model with input and target transformations, evaluated in physical units.

So far every task was balanced classification, judged by accuracy. Engineering problems are often different: defects are rare, missing a fault costs far more than a false alarm, and many targets are continuous quantities with units.

🔁 **Starting a fresh session here?** Run the two Chapter 0 cells, then the hidden restore-point cell below. It re-creates the tools from earlier chapters that Chapter 9 uses. In a normal top-to-bottom run it does nothing.

In [ ]:
# @title 🔁 Restore point for Chapter 9 (needed only if you start a fresh session here) { display-mode: "form" }
if "device" not in globals():
    raise RuntimeError("Run the Chapter 0 setup cells first, then this cell.")
if "get_trained_cnn" not in globals():
    from torch.optim import lr_scheduler as S

    def param_groups_weight_decay(model, weight_decay):
        decay, no_decay = [], []
        for name, p in model.named_parameters():
            if not p.requires_grad:
                continue                                     # frozen parameters: leave them out
            (no_decay if p.ndim <= 1 else decay).append(p)   # biases and norm weights are 1-D
        return [{"params": decay, "weight_decay": weight_decay},
                {"params": no_decay, "weight_decay": 0.0}]

    class EarlyStopping:
        # Stop when `metric` has not improved by more than `min_delta` for `patience` epochs.
        # mode="min" for losses, mode="max" for accuracies. Keeps a copy of the best weights.
        def __init__(self, patience=5, min_delta=0.0, mode="min"):
            self.patience, self.min_delta, self.mode = patience, min_delta, mode
            self.best = math.inf if mode == "min" else -math.inf
            self.best_state, self.best_epoch, self.bad_epochs, self.epoch = None, 0, 0, 0

        def improved(self, value):
            if self.mode == "min":
                return value < self.best - self.min_delta
            return value > self.best + self.min_delta

        def step(self, value, model):
            # Call once per epoch. Returns True when training should stop.
            self.epoch += 1
            if self.improved(value):
                self.best, self.best_epoch, self.bad_epochs = value, self.epoch, 0
                self.best_state = copy.deepcopy(model.state_dict())
            else:
                self.bad_epochs += 1
            return self.bad_epochs >= self.patience

        def restore(self, model):
            model.load_state_dict(self.best_state)

    import contextlib

    def train_one_epoch(model, loader, loss_fn, optimizer, scheduler=None, max_grad_norm=None,
                        averaged_model=None, amp_dtype=None, scaler=None):
        # amp_dtype / scaler: mixed precision, explained in Chapter 11. Leave them as None until then.
        model.train()
        total_loss = torch.zeros((), device=device)                  # running sums stay ON the device:
        correct = torch.zeros((), device=device, dtype=torch.long)   # no CPU-GPU synchronisation per step
        n, grad_norms = 0, []
        for x, y in loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            amp = torch.autocast(device_type=device.type, dtype=amp_dtype) if amp_dtype else contextlib.nullcontext()
            with amp:
                logits = model(x)
                loss = loss_fn(logits, y)
            optimizer.zero_grad(set_to_none=True)
            if scaler is not None:                                   # float16 mixed precision (Chapter 11)
                scaler.scale(loss).backward()
                if max_grad_norm is not None:
                    scaler.unscale_(optimizer)                       # clip the REAL gradients, not the scaled ones
            else:
                loss.backward()
            if max_grad_norm is not None:
                grad_norms.append(nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm))
            if scaler is not None:
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()
            if scheduler is not None:                                # per-batch schedules (OneCycleLR, warm-up)
                scheduler.step()
            if averaged_model is not None:                           # weight averaging (Chapter 6)
                averaged_model.update_parameters(model)
            total_loss += loss.detach().float() * len(y)
            correct += (logits.argmax(dim=1) == y).sum()
            n += len(y)
        mean_norm = torch.stack(grad_norms).mean().item() if grad_norms else float("nan")
        return total_loss.item() / n, correct.item() / n, mean_norm   # one synchronisation per epoch


    def fit(model, train_loader, val_loader, optimizer, epochs, loss_fn=nn.CrossEntropyLoss(),
            scheduler=None, scheduler_interval="batch", early_stopping=None, eval_model=None, verbose=True, **train_kw):
        # scheduler_interval: "batch", "epoch" or "plateau" (ReduceLROnPlateau).
        # eval_model: evaluate (and select) a different model than the one trained, e.g. an EMA copy (Chapter 6).
        history = {k: [] for k in ["train_loss", "train_acc", "val_loss", "val_acc", "lr", "grad_norm", "time"]}
        eval_model = eval_model if eval_model is not None else model
        for epoch in range(1, epochs + 1):
            t0 = time.perf_counter()
            lr = optimizer.param_groups[0]["lr"]
            tr_loss, tr_acc, gnorm = train_one_epoch(model, train_loader, loss_fn, optimizer,
                                                     scheduler if scheduler_interval == "batch" else None, **train_kw)
            va_loss, va_acc = evaluate(eval_model, val_loader, loss_fn)
            if scheduler is not None and scheduler_interval == "epoch":
                scheduler.step()
            elif scheduler is not None and scheduler_interval == "plateau":
                scheduler.step(va_loss)
            elapsed = time.perf_counter() - t0
            for k, v in zip(history, (tr_loss, tr_acc, va_loss, va_acc, lr, gnorm, elapsed)):
                history[k].append(v)
            if verbose:
                print(f"epoch {epoch:2d} | lr {lr:.2e} | train {tr_loss:.4f} / {tr_acc:.4f} | "
                      f"val {va_loss:.4f} / {va_acc:.4f} | {elapsed:4.1f}s")
            if early_stopping is not None and early_stopping.step(va_loss, eval_model):
                print(f"early stopping: no improvement for {early_stopping.patience} epochs")
                break
        if early_stopping is not None:                       # select the epoch with the best validation LOSS...
            early_stopping.restore(eval_model)
            sel_epoch = early_stopping.best_epoch
        else:
            sel_epoch = len(history["val_loss"])            # ...or keep the last epoch
        sel_loss, sel_acc = history["val_loss"][sel_epoch - 1], history["val_acc"][sel_epoch - 1]
        history["selected"] = {"epoch": sel_epoch, "val_loss": sel_loss, "val_acc": sel_acc}
        if verbose:
            print(f"selected model: epoch {sel_epoch}, val loss {sel_loss:.4f}, val acc {sel_acc:.4f}")
        return history


    def plot_history(histories, keys=("val_loss", "val_acc", "lr")):
        fig, axes = plt.subplots(1, len(keys), figsize=(5 * len(keys), 3.2))
        for ax, key in zip(axes, keys):
            for name, h in histories.items():
                ax.plot(range(1, len(h[key]) + 1), h[key], "o-", ms=3, label=name)
            ax.set_title(key); ax.set_xlabel("epoch")
        axes[0].legend(fontsize=8); plt.tight_layout(); plt.show()

    # Small in-memory subsets (normalised exactly like fm_transform) for quick experiments
    def _fm_subset(idx):
        x = ((raw_train.data[idx].float() / 255 - FM_MEAN) / FM_STD).unsqueeze(1)
        return TensorDataset(x, raw_train.targets[idx])

    small_ds = _fm_subset(FM_TRAIN_IDX[:4096])      # 4,096 training images
    small_val = _fm_subset(FM_VAL_IDX[:2000])       # 2,000 validation images

    def save_checkpoint(path, model, optimizer, scheduler=None, epoch=0, best_metric=None,
                        loader_generator=None, config=None):
        np_state = np.random.get_state()
        ckpt = {
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict() if scheduler is not None else None,
            "epoch": epoch,
            "best_metric": best_metric,
            "config": config or {},
            "rng": {
                "torch": torch.get_rng_state(),
                "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [],
                "python": random.getstate(),
                # NumPy's state contains an array; store it as a list so the safe loader accepts it
                "numpy": (np_state[0], np_state[1].tolist(), *np_state[2:]),
            },
            "loader_generator": loader_generator.get_state() if loader_generator is not None else None,
        }
        tmp = Path(str(path) + ".tmp")
        torch.save(ckpt, tmp)             # write to a temporary file first...
        os.replace(tmp, path)             # ...then rename: an interruption can never leave a half-written checkpoint


    def load_checkpoint(path, model, optimizer=None, scheduler=None, loader_generator=None):
        ckpt = torch.load(path, map_location=device, weights_only=True)   # safe loading: tensors and plain data only
        model.load_state_dict(ckpt["model"])
        if optimizer is not None:
            optimizer.load_state_dict(ckpt["optimizer"])
        if scheduler is not None and ckpt["scheduler"] is not None:
            scheduler.load_state_dict(ckpt["scheduler"])
        rng = ckpt["rng"]
        torch.set_rng_state(rng["torch"].cpu())
        if torch.cuda.is_available() and rng["cuda"]:
            torch.cuda.set_rng_state_all([s.cpu() for s in rng["cuda"]])
        random.setstate(rng["python"])
        name, keys, *rest = rng["numpy"]
        np.random.set_state((name, np.array(keys, dtype=np.uint32), *rest))
        if loader_generator is not None and ckpt["loader_generator"] is not None:
            loader_generator.set_state(ckpt["loader_generator"].cpu())
        return ckpt

    def conv_bn_relu(c_in, c_out):
        return nn.Sequential(nn.Conv2d(c_in, c_out, 3, padding=1, bias=False), nn.BatchNorm2d(c_out), nn.ReLU(inplace=True))


    class SmallCNN(nn.Module):
        def __init__(self, num_classes=10, width=32, p_drop=0.3):
            super().__init__()
            w = width
            self.features = nn.Sequential(
                conv_bn_relu(1, w), conv_bn_relu(w, w), nn.MaxPool2d(2),              # 28x28 -> 14x14
                conv_bn_relu(w, 2 * w), conv_bn_relu(2 * w, 2 * w), nn.MaxPool2d(2),  # 14x14 -> 7x7
                conv_bn_relu(2 * w, 4 * w),                                           # 7x7, 128 channels
                nn.AdaptiveAvgPool2d(1),                                              # -> 1x1
            )
            self.head = nn.Sequential(nn.Flatten(), nn.Dropout(p_drop), nn.Linear(4 * w, num_classes))

        def forward(self, x):
            return self.head(self.features(x))

    CNN_PATH = CKPT_DIR / "smallcnn_fashion.pt"

    def train_small_cnn(epochs=12, seed=0):
        # The recipe of Section 7.7: AdamW with weight-decay groups and a OneCycle schedule.
        set_seed(seed)
        model = SmallCNN().to(device)
        E = n_epochs(epochs)
        opt = torch.optim.AdamW(param_groups_weight_decay(model, 0.05), lr=1e-3)
        sched = S.OneCycleLR(opt, max_lr=3e-3, total_steps=E * len(train_loader))
        hist = fit(model, train_loader, val_loader, opt, E, scheduler=sched,
                   early_stopping=EarlyStopping(patience=100))        # patience=100: just keep the best epoch
        torch.save({"state_dict": model.state_dict()}, CNN_PATH)
        return model, hist

    def get_trained_cnn():
        # Load the CNN trained in Chapter 7 if it was saved; otherwise train it now.
        model = SmallCNN().to(device)
        if CNN_PATH.exists():
            model.load_state_dict(torch.load(CNN_PATH, map_location=device, weights_only=True)["state_dict"])
            print(f"loaded the trained SmallCNN from {CNN_PATH}")
        else:
            print("no saved SmallCNN found: training one with the recipe of Section 7.7 ...")
            model, _ = train_small_cnn()
        return model.eval()
    print("Restored the tools defined in earlier chapters.")

## 9.1 When accuracy lies: rare defects
A synthetic quality-control dataset: 8 standardised sensor features per part, and about **4% defective parts**. Defects shift a few of the features, but the classes overlap, as in real inspection data.

In [ ]:
def make_defect_data(n, seed, defect_rate=0.04):
    g = torch.Generator().manual_seed(seed)
    y = (torch.rand(n, generator=g) < defect_rate).long()
    X = torch.randn(n, 8, generator=g)
    shift = torch.tensor([1.2, -0.8, 0.6, 0.0, 0.0, 0.9, 0.0, -0.5])     # how a defect changes the readings
    return X + y[:, None] * shift, y

n_def = 4000 if FAST_DEV_RUN else 20000
Xd_tr, yd_tr = make_defect_data(n_def, seed=0)
Xd_va, yd_va = make_defect_data(n_def // 4, seed=1)
Xd_te, yd_te = make_defect_data(n_def // 4, seed=2)
print(f"defect rate: train {yd_tr.float().mean():.3f} | val {yd_va.float().mean():.3f} | test {yd_te.float().mean():.3f}")
print(f"accuracy of a 'model' that always says OK: {1 - yd_va.float().mean():.3f}")

A model that never finds a single defect scores about 96% accuracy. Accuracy rewards the majority class, so on imbalanced data it says almost nothing about what we care about. Let us train a real classifier (one logit, `BCEWithLogitsLoss`, Notebook 1 §5.5) and look at better metrics.

In [ ]:
def train_defect_model(pos_weight=None, sampler=None, epochs=15, seed=0):
    set_seed(seed)
    model = nn.Sequential(nn.Linear(8, 32), nn.ReLU(), nn.Linear(32, 32), nn.ReLU(), nn.Linear(32, 1)).to(device)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=None if pos_weight is None else torch.tensor([pos_weight], device=device))
    opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
    loader = DataLoader(TensorDataset(Xd_tr, yd_tr.float()), batch_size=256,
                        shuffle=sampler is None, sampler=sampler)
    for _ in range(n_epochs(epochs)):
        model.train()
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            loss = loss_fn(model(x).squeeze(1), y)
            opt.zero_grad(); loss.backward(); opt.step()
    return model.eval()

@torch.no_grad()
def defect_scores(model, X):
    return torch.sigmoid(model(X.to(device)).squeeze(1)).cpu()     # estimated P(defect) per part

defect_model = train_defect_model()
scores_va = defect_scores(defect_model, Xd_va)

## 9.2 The confusion matrix and the metrics built from it
For a binary decision, every prediction falls into one of four cells. Treat "defect" as **positive**:

| | predicted defect | predicted OK |
| --- | --- | --- |
| **actual defect** | TP (true positive) | FN (false negative: a **missed** defect) |
| **actual OK** | FP (false positive: a **false alarm**) | TN (true negative) |

| Metric | Formula | Question it answers |
| --- | --- | --- |
| Precision | TP / (TP + FP) | Of the parts we flagged, how many were really defective? |
| Recall (sensitivity) | TP / (TP + FN) | Of the defective parts, how many did we catch? |
| F1 | 2 · P · R / (P + R) | One number that is high only if both are high |
| Specificity | TN / (TN + FP) | Of the good parts, how many did we pass? |

In [ ]:
def binary_metrics(y_true, y_pred):
    tp = ((y_pred == 1) & (y_true == 1)).sum().item()
    fp = ((y_pred == 1) & (y_true == 0)).sum().item()
    fn = ((y_pred == 0) & (y_true == 1)).sum().item()
    tn = ((y_pred == 0) & (y_true == 0)).sum().item()
    precision = tp / max(tp + fp, 1)                 # define 0/0 as 0
    recall = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-12)
    return {"accuracy": (tp + tn) / len(y_true), "precision": precision, "recall": recall, "f1": f1,
            "tp": tp, "fp": fp, "fn": fn, "tn": tn}

always_ok = binary_metrics(yd_va, torch.zeros_like(yd_va))
model_05 = binary_metrics(yd_va, (scores_va > 0.5).long())
for name, m in [("always OK", always_ok), ("model, threshold 0.5", model_05)]:
    print(f"{name:22s} " + " | ".join(f"{k} {v:.3f}" if isinstance(v, float) else f"{k} {v}" for k, v in m.items()))

try:                                                  # cross-check with scikit-learn, if installed
    from sklearn.metrics import precision_recall_fscore_support
    p, r, f, _ = precision_recall_fscore_support(yd_va, (scores_va > 0.5).long(), average="binary", zero_division=0)
    print(f"scikit-learn agrees: precision {p:.3f}, recall {r:.3f}, f1 {f:.3f}")
except ImportError:
    pass

The model's accuracy is barely above the trivial baseline, but precision and recall reveal what really happens: at the default threshold of 0.5 it catches only part of the defects. Whether that is acceptable depends on costs, which the threshold controls.

## 9.3 Thresholds, PR curves and ROC curves
The model outputs a **score**; the decision "defect if score > t" depends on a threshold `t` that **you** choose. Lowering `t` catches more defects (higher recall) at the price of more false alarms (lower precision). Two curves summarise all thresholds at once:
- the **precision–recall (PR) curve**, summarised by the **average precision (AP)**;
- the **ROC curve** (recall versus false-positive rate FP / (FP + TN)), summarised by the **area under it (ROC AUC)**.

Both are easy to compute: sort the examples by score, then sweep the threshold down one example at a time with cumulative sums.

In [ ]:
def pr_roc(scores, y):
    order = scores.argsort(descending=True)                # threshold sweeps from high to low
    ys = y[order].float()
    tp, fp = ys.cumsum(0), (1 - ys).cumsum(0)              # counts when the top-k examples are flagged
    precision = tp / (tp + fp)
    recall = tp / ys.sum()
    fpr = fp / (1 - ys).sum()
    return precision, recall, fpr, scores[order]

prec, rec, fpr, thr = pr_roc(scores_va, yd_va)
ap = (prec * torch.diff(rec, prepend=torch.zeros(1))).sum().item()     # average precision: sum of P * (step in R)
auc = torch.trapz(torch.cat([torch.zeros(1), rec]), torch.cat([torch.zeros(1), fpr])).item()
print(f"average precision {ap:.3f} (a random model scores ≈ the defect rate, {yd_va.float().mean():.3f}) | ROC AUC {auc:.3f} (random: 0.5)")

fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.8))
a1.plot(rec, prec); a1.set_xlabel("recall"); a1.set_ylabel("precision"); a1.set_title(f"PR curve (AP = {ap:.3f})")
a2.plot(fpr, rec); a2.plot([0, 1], [0, 1], "k--", lw=0.8)
a2.set_xlabel("false-positive rate"); a2.set_ylabel("recall"); a2.set_title(f"ROC curve (AUC = {auc:.3f})")
plt.tight_layout(); plt.show()

⚠️ **ROC AUC can look reassuring on imbalanced data.** The false-positive rate divides by the huge number of good parts, so even many false alarms produce a small FPR. The PR curve divides by the number of *flagged* parts, so it shows the false alarms directly. For rare positives, report AP (or precision at the recall you need) rather than ROC AUC alone.

(Our simple sweep treats tied scores one at a time; libraries such as scikit-learn group ties, which changes the numbers only slightly when there are many ties.)

**Choosing the threshold.** Suppose the requirement is to catch at least 90% of defects. Choose the threshold on the **validation** set (the highest threshold that still reaches 90% recall, which maximises precision under the constraint), then measure the result **once** on the test set:

In [ ]:
TARGET_RECALL = 0.90
ok = (rec >= TARGET_RECALL).nonzero()
threshold = thr[ok[0]].item()                               # first index in the sweep = highest such threshold
print(f"chosen threshold on validation: {threshold:.3f}")

test_m = binary_metrics(yd_te, (defect_scores(defect_model, Xd_te) >= threshold).long())
print("test set at this threshold:", {k: round(v, 3) if isinstance(v, float) else v for k, v in test_m.items()})
record("Ch9 defect model, test recall at the chosen threshold", test_m["recall"])
record("Ch9 defect model, test precision at the chosen threshold", test_m["precision"])

The test recall lands near the 90% target (not exactly, because thresholds chosen on one sample transfer imperfectly to another), and the precision tells the plant how many flagged parts will turn out to be fine. That is the conversation to have with the people who use the model: a pair of numbers with a meaning, not "96% accuracy".

## 9.4 Remedies for class imbalance
Three common remedies:
1. **Reweight the loss**: `BCEWithLogitsLoss(pos_weight=n_neg / n_pos)` makes each defect count as much as all good parts together.
2. **Resample**: `WeightedRandomSampler` draws defects more often, so batches are roughly balanced.
3. **Move the threshold** (§9.3), leaving training unchanged.

Compare them with **AP**, which does not depend on a threshold, so we measure the ranking quality itself:

In [ ]:
from torch.utils.data import WeightedRandomSampler

def average_precision_of(model, X, y):
    p, r, _, _ = pr_roc(defect_scores(model, X), y)
    return (p * torch.diff(r, prepend=torch.zeros(1))).sum().item()

n_pos, n_neg = yd_tr.sum().item(), (1 - yd_tr).sum().item()
class_weight = torch.where(yd_tr == 1, 1.0 / n_pos, 1.0 / n_neg)          # each class gets half the draws
sampler = WeightedRandomSampler(class_weight, num_samples=len(yd_tr), replacement=True)
variants = {"plain": train_defect_model(),
            "pos_weight": train_defect_model(pos_weight=n_neg / n_pos),
            "balanced sampler": train_defect_model(sampler=sampler)}
for name, m in variants.items():
    s = defect_scores(m, Xd_va)
    f05 = binary_metrics(yd_va, (s > 0.5).long())
    print(f"{name:17s} AP {average_precision_of(m, Xd_va, yd_va):.3f} | at threshold 0.5: "
          f"precision {f05['precision']:.3f}, recall {f05['recall']:.3f} | mean score {s.mean():.3f}")

Typically all three reach a similar AP: reweighting and resampling mostly **shift the scores upwards** (look at the mean score), which raises recall at the fixed threshold 0.5 and lowers precision, while the ranking of parts barely changes. Practical consequences:
- Reweighting or resampling is useful when you must use a fixed threshold, or when the minority class is so rare that the model otherwise ignores it.
- The scores of a reweighted model are **no longer probabilities** (they overestimate the defect rate), which matters if anyone uses them as probabilities. Chapter 14 shows how to recalibrate scores.
- Threshold tuning on the validation set is cheap, preserves calibrated scores, and is often enough.

## 9.5 Multi-class: per-class metrics, macro and micro averages
For several classes, compute precision, recall and F1 **per class** (each class in turn is "positive"). Two averages summarise them:
- **macro** average: the plain mean over classes, so every class counts equally (rare classes matter);
- **micro** average: pool all TP/FP/FN first. For single-label classification, micro-F1 equals accuracy.

Here are the Fashion-MNIST CNN's numbers (the network from Chapter 7):

In [ ]:
cnn = get_trained_cnn()
preds, ys = [], []
with torch.no_grad():
    for x, y in test_loader:
        preds.append(cnn(x.to(device)).argmax(dim=1).cpu()); ys.append(y.cpu())
pred_t, true_t = torch.cat(preds), torch.cat(ys)
cm = torch.bincount(true_t * 10 + pred_t, minlength=100).reshape(10, 10).float()   # rows: true, columns: predicted
tp = cm.diag()
precision_c = tp / cm.sum(dim=0).clamp(min=1)
recall_c = tp / cm.sum(dim=1).clamp(min=1)
f1_c = 2 * precision_c * recall_c / (precision_c + recall_c).clamp(min=1e-12)
for i in f1_c.argsort():
    print(f"{CLASSES[i]:12s} precision {precision_c[i]:.3f} | recall {recall_c[i]:.3f} | F1 {f1_c[i]:.3f}")
print(f"\nmacro-F1 {f1_c.mean():.4f} | micro-F1 = accuracy {(tp.sum() / cm.sum()):.4f}")

`torchmetrics` (and scikit-learn's `classification_report`) compute all of these, including for batches accumulated over an epoch; knowing the definitions lets you check them and pick the right average.

## 9.6 Regression: a surrogate model for a vibrating system
Engineers often replace an expensive simulation with a fast neural **surrogate**. Our "simulation" is the steady-state amplitude of a forced spring–mass–damper:

$$A = \frac{F_0}{\sqrt{(k - m\,\omega^2)^2 + (c\,\omega)^2}}$$

with mass $m$ (0.5–5 kg), stiffness $k$ (100–5000 N/m), damping $c$ (0.5–50 N·s/m), force amplitude $F_0$ (1–100 N) and forcing frequency $\omega$ (1–100 rad/s). This is a realistic surrogate problem: the inputs span decades in different units, and the target spans more than five orders of magnitude, with sharp peaks at resonance ($\omega^2 \approx k/m$).

In [ ]:
def make_spring_data(n, seed):
    g = torch.Generator().manual_seed(seed)
    lo = torch.tensor([0.5, 100.0, 0.5, 1.0, 1.0])            # m, k, c, F0, omega
    hi = torch.tensor([5.0, 5000.0, 50.0, 100.0, 100.0])
    u = torch.rand(n, 5, generator=g)
    X = torch.exp(lo.log() + u * (hi.log() - lo.log()))       # log-uniform: every decade equally represented
    m, k, c, F0, w = X.unbind(dim=1)
    A = F0 / torch.sqrt((k - m * w ** 2) ** 2 + (c * w) ** 2)
    return X, A

n_spr = 4000 if FAST_DEV_RUN else 20000
Xs_tr, As_tr = make_spring_data(n_spr, seed=0)
Xs_va, As_va = make_spring_data(4000, seed=1)
Xs_te, As_te = make_spring_data(4000, seed=2)
print(f"amplitude range: {As_tr.min():.2e} to {As_tr.max():.2e} m")

Three decisions make or break such a model:
1. **Input transform.** The inputs are positive and span decades, so use $\log x$, then standardise with **training** statistics.
2. **Target transform.** Train on $\log A$, standardised. An MSE on $\log A$ penalises **relative** errors (being 10% off counts the same for a tiny and a huge amplitude), and predictions are automatically positive after $\exp$. Store the statistics: they are part of the model.
3. **Evaluate in physical units**, after undoing the transforms: here the relative error $|\hat A - A| / A$.

In [ ]:
class Standardizer:
    # Fit on the training data only; apply the same frozen statistics everywhere (Notebook 1, §6.2).
    def __init__(self, z):
        self.mean, self.std = z.mean(dim=0), z.std(dim=0)
    def __call__(self, z):
        return (z - self.mean) / self.std
    def inverse(self, z):
        return z * self.std + self.mean

x_scaler = Standardizer(Xs_tr.log())
y_scaler = Standardizer(As_tr.log())

def spring_tensors(X, A):
    return x_scaler(X.log()), y_scaler(A.log())[:, None]

def train_surrogate(target="log", epochs=60, seed=0, width=128, lr=3e-3, Xtr=None, Atr=None):
    Xtr = Xs_tr if Xtr is None else Xtr
    Atr = As_tr if Atr is None else Atr
    if target == "log":
        xt, yt = spring_tensors(Xtr, Atr)
    else:                                                     # the naive alternative, for comparison
        xt, yt = x_scaler(Xtr.log()), ((Atr - As_tr.mean()) / As_tr.std())[:, None]
    set_seed(seed)
    model = nn.Sequential(nn.Linear(5, width), nn.SiLU(), nn.Linear(width, width), nn.SiLU(),
                          nn.Linear(width, width), nn.SiLU(), nn.Linear(width, 1)).to(device)
    E = n_epochs(epochs)
    loader = TensorLoader(xt, yt, batch_size=256, shuffle=True, drop_last=True)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)   # OneCycleLR sets the actual lr
    sched = S.OneCycleLR(opt, max_lr=lr, total_steps=E * len(loader))
    for _ in range(E):
        model.train()
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            loss = F.mse_loss(model(x), y)
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
    return model.eval()

@torch.no_grad()
def predict_amplitude(model, X, target="log"):
    z = model(x_scaler(X.log()).to(device)).squeeze(1).cpu()
    return y_scaler.inverse(z).exp() if target == "log" else z * As_tr.std() + As_tr.mean()

surrogate = train_surrogate()
rel_err = (predict_amplitude(surrogate, Xs_va) - As_va).abs() / As_va
print(f"validation relative error: median {rel_err.median():.4f} | 90th percentile {rel_err.quantile(0.9):.4f} | max {rel_err.max():.3f}")
record("Ch9 surrogate, median relative error (val)", rel_err.median().item())

When this notebook was written, a CPU run gave a median relative error of about 0.5% and a 90th percentile of about 2.5%; yours should be in the same range. A median alone hides the worst cases, so always look at the tail too, and at **where** the errors occur:

In [ ]:
A_hat = predict_amplitude(surrogate, Xs_va)
m, k, c, F0, w = Xs_va.unbind(dim=1)
ratio = w / torch.sqrt(k / m)                                   # forcing frequency / natural frequency

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4))
a1.loglog(As_va, A_hat, ".", ms=2, alpha=0.5); a1.loglog([1e-5, 10], [1e-5, 10], "k--", lw=0.8)
a1.set_xlabel("true amplitude (m)"); a1.set_ylabel("predicted amplitude (m)"); a1.set_title("Parity plot")
a2.semilogx(ratio, (A_hat - As_va).abs() / As_va, ".", ms=2, alpha=0.5)
a2.axvline(1.0, color="tab:red", ls="--", lw=0.8, label="resonance")
a2.set_xlabel("ω / ω_n"); a2.set_ylabel("relative error"); a2.set_yscale("log"); a2.legend(); a2.set_title("Where are the errors?")
plt.tight_layout(); plt.show()

The parity plot hugs the diagonal across five decades, and the error plot shows that the largest errors cluster around **resonance** ($\omega/\omega_n \approx 1$), where the amplitude changes fastest with the inputs. That is typical: surrogates are least reliable exactly where the physics is sharpest. In practice you would add training samples there, or report the uncertainty near resonance, and **never** use the surrogate outside the input ranges it was trained on.

## 9.7 ★ Choosing hyper-parameters honestly: random search and k-fold cross-validation
**Random search.** Sample hyper-parameter combinations at random (log-uniformly for learning rates), train each briefly, and keep the best by **validation** score. For the same budget it usually beats a grid, because it tries more distinct values of the parameters that matter most.

In [ ]:
gen = torch.Generator().manual_seed(0)
trials = []
for t in range(2 if FAST_DEV_RUN else 6):
    lr = 10 ** (torch.empty(1).uniform_(-3.5, -2, generator=gen).item())      # 3e-4 ... 1e-2, log-uniform
    width = [32, 64, 128][torch.randint(0, 3, (1,), generator=gen).item()]
    m = train_surrogate(epochs=15, lr=lr, width=width, Xtr=Xs_tr[:5000], Atr=As_tr[:5000])    # short, cheap trials
    med = ((predict_amplitude(m, Xs_va) - As_va).abs() / As_va).median().item()
    trials.append((med, lr, width))
    print(f"trial {t}: lr {lr:.1e}, width {width:3d} -> validation median relative error {med:.4f}")
best = min(trials)
print(f"best: lr {best[1]:.1e}, width {best[2]} ({best[0]:.4f})")

The trials are far less accurate than the full model of §9.6 because they are deliberately short and use a quarter of the data: their job is to **rank** settings cheaply, and the winner is then trained properly. Short-budget rankings are not always the same as full-budget rankings, so confirm the final choice with a full run.

**k-fold cross-validation.** With small datasets a single validation split is noisy. Split the training data into k folds; train k times, each time validating on a different fold; report the mean and standard deviation. It costs k trainings, so it is used for small data and final comparisons, not for every quick experiment.

In [ ]:
def kfold_indices(n, k, seed=0):
    perm = torch.randperm(n, generator=torch.Generator().manual_seed(seed))
    return perm.chunk(k)                                     # k nearly equal folds

n_small, K = 600, (2 if FAST_DEV_RUN else 5)                 # pretend only 600 simulations are affordable
folds = kfold_indices(n_small, K)
fold_errors = []
for i in range(K):
    val_idx = folds[i]
    tr_idx = torch.cat([f for j, f in enumerate(folds) if j != i])
    m = train_surrogate(epochs=40, Xtr=Xs_tr[tr_idx], Atr=As_tr[tr_idx])
    err = ((predict_amplitude(m, Xs_tr[val_idx]) - As_tr[val_idx]).abs() / As_tr[val_idx]).median().item()
    fold_errors.append(err)
    print(f"fold {i}: median relative error {err:.4f}")
fe = torch.tensor(fold_errors)
print(f"{K}-fold estimate: {fe.mean():.4f} ± {fe.std():.4f}")

With only 600 simulations the surrogate is far less accurate than with 20,000 (when this notebook was written: a median error around 46% versus 0.5%), and the k-fold spread tells you how much that estimate itself can be trusted. Small data is where this matters most.

(For simplicity the scalers here still use the statistics of the full training set; in a strict k-fold protocol you refit them on each fold's training part, exactly as §6.2 of Notebook 1 requires for any split.)

## 🧪 Exercises

**Exercise 9.1: average precision.** `★★☆ · ~15 min`
Write `average_precision(scores, y)` that returns AP, the sum over all positives of the precision at the rank where each positive is found, divided by the number of positives. (This is exactly what the cumulative-sum code of §9.3 computes; write it your own way to check your understanding.)

In [ ]:
# ✍️ Your attempt here.
# Hint: Sort by score; for the k-th example in that order, precision@k = (#positives among the first k) / k. Average this over the positions of the positives.

In [ ]:
# 🔎 Self-check for Exercise 9.1
def _test(fn):
    s = torch.tensor([0.9, 0.8, 0.7, 0.6, 0.5, 0.4])
    y = torch.tensor([1, 0, 1, 0, 0, 1])
    expected = (1 / 1 + 2 / 3 + 3 / 6) / 3
    assert abs(fn(s, y) - expected) < 1e-6, f"got {fn(s, y)}, expected {expected:.6f}"
    g = torch.Generator().manual_seed(0)
    s2, y2 = torch.rand(200, generator=g), (torch.rand(200, generator=g) < 0.2).long()
    p, r, _, _ = pr_roc(s2, y2)
    ref = (p * torch.diff(r, prepend=torch.zeros(1))).sum().item()
    assert abs(fn(s2, y2) - ref) < 1e-5, "does not match the §9.3 computation"
self_check("average_precision", _test)

In [ ]:
# @title ✅ Solution (double-click to reveal) { display-mode: "form" }
# Solutions only execute when RUN_SOLUTIONS = True (setup cell), so "Run all" stays fast.
if RUN_SOLUTIONS:
    def average_precision(scores, y):
        ys = y[scores.argsort(descending=True)].float()
        hits = ys.cumsum(0)                                   # positives among the first k
        precision_at_k = hits / torch.arange(1, len(ys) + 1)
        return float((precision_at_k * ys).sum() / ys.sum())  # average over the positions of the positives

    print(average_precision(torch.tensor([0.9, 0.8, 0.7]), torch.tensor([1, 0, 1])))
else:
    print("Solution not executed (RUN_SOLUTIONS = False).")

**Exercise 9.2: what if you skip the target transform?** `★★☆ · ~15 min`
Train `train_surrogate(target="raw")`, which standardises $A$ itself instead of $\log A$, and compute the same relative-error statistics with `predict_amplitude(model, Xs_va, target="raw")`. Also count how many predicted amplitudes are **negative**. Explain the result.

In [ ]:
# ✍️ Your attempt here.

In [ ]:
# @title ✅ Solution (double-click to reveal) { display-mode: "form" }
# Solutions only execute when RUN_SOLUTIONS = True (setup cell), so "Run all" stays fast.
if RUN_SOLUTIONS:
    raw = train_surrogate(target="raw")
    A_raw = predict_amplitude(raw, Xs_va, target="raw")
    rel = (A_raw - As_va).abs() / As_va
    print(f"raw target: median {rel.median():.4f} | 90th percentile {rel.quantile(0.9):.4f} | negative predictions {(A_raw <= 0).float().mean():.1%}")
    # On the raw scale the MSE is dominated by the few large amplitudes near resonance; errors of 1e-4 m
    # are invisible to it, although they are 100% errors for parts that only move 1e-4 m. Nothing
    # keeps predictions positive either. When this notebook was written, the raw target gave a median
    # relative error around 15% (versus about 0.5% with log A) and about one in ten predictions was negative.
else:
    print("Solution not executed (RUN_SOLUTIONS = False).")

### 📌 Key takeaways
- On imbalanced data, report precision, recall, F1 and AP (or precision at a required recall), not accuracy alone.
- The threshold is a design decision: choose it on validation from the costs of misses and false alarms; measure once on test.
- Reweighting and resampling mainly shift scores; threshold tuning is often enough and keeps scores calibrated.
- Multi-class: per-class metrics; macro-average when rare classes matter; micro-F1 = accuracy for single-label tasks.
- Regression: transform inputs and targets to sensible scales (often logs), keep the statistics with the model, and evaluate in physical units, including the error tail and where errors occur.

### ✔️ Check yourself

Commit to an answer (say it out loud or write it down) *before* you expand it.

**1.** A fault detector has 99.5% accuracy on data with 0.5% faults. What do you ask before believing it is good?

<details>
<summary>Answer</summary>

Its recall and precision on the fault class (or AP): a detector that never reports a fault also has 99.5% accuracy.

</details>

**2.** Why choose the threshold on the validation set rather than on the test set?

<details>
<summary>Answer</summary>

Choosing it is a modelling decision. If the test set is used for it, the test result is no longer an unbiased estimate.

</details>

**3.** Why is an MSE on log A a good loss for a quantity that spans several orders of magnitude?

<details>
<summary>Answer</summary>

A difference in logs is a ratio, so the loss penalises relative errors equally for small and large values; on the raw scale the largest values dominate the loss and small values are effectively ignored. It also guarantees positive predictions after exp.

</details>

**4.** Is macro-F1 or micro-F1 more informative when one of ten classes is rare and important?

<details>
<summary>Answer</summary>

Macro-F1: every class counts equally, so poor performance on the rare class lowers it visibly. Micro-F1 is dominated by the frequent classes (and equals accuracy for single-label tasks).

</details>

---
# Chapter 10. Higher-order autograd and `torch.func`

🎯 **In this chapter you will learn to:** compute gradients with respect to *inputs* (sensitivities) with `torch.autograd.grad`; differentiate through a derivative with `create_graph=True`; train a small physics-informed network; and use `torch.func` (`grad`, `vmap`, `jacrev`, `functional_call`) for per-sample gradients and batched Jacobians.

So far `backward()` gave us one thing: the gradient of a scalar loss with respect to the parameters. Many engineering and research uses need more: derivatives of a model's **output with respect to its inputs** (sensitivity analysis), derivatives of derivatives (physics-informed networks, gradient penalties, meta-learning), or gradients **per example** instead of averaged over the batch.

## 10.1 `torch.autograd.grad`: gradients you get back
`torch.autograd.grad(outputs, inputs)` computes $\partial\,\text{outputs} / \partial\,\text{inputs}$ and **returns** it, instead of accumulating into `.grad`. The inputs can be anything that requires gradients, including data.

**Example: sensitivity of the vibrating system.** For the amplitude $A = F_0 / \sqrt{(k - m\omega^2)^2 + (c\omega)^2}$ of Chapter 9, the derivative with respect to the stiffness is, by hand,
$$\frac{\partial A}{\partial k} = -\frac{F_0\,(k - m\omega^2)}{\big((k - m\omega^2)^2 + (c\omega)^2\big)^{3/2}}.$$
Autograd gives the derivatives with respect to **all five** inputs at once, for a whole batch of designs:

In [ ]:
def amplitude(X):
    m, k, c, F0, w = X.unbind(dim=-1)
    return F0 / torch.sqrt((k - m * w ** 2) ** 2 + (c * w) ** 2)

X = torch.tensor([[1.0, 1000.0, 5.0, 10.0, 20.0],        # m, k, c, F0, omega: below resonance (omega_n ≈ 31.6)
                  [1.0, 1000.0, 5.0, 10.0, 31.0]],        # close to resonance
                 dtype=torch.float64, requires_grad=True)
A = amplitude(X)
(dA_dX,) = torch.autograd.grad(A.sum(), X)                # sum: each A depends only on its own row
m, k, c, F0, w = X.detach().unbind(dim=1)
D = (k - m * w ** 2) ** 2 + (c * w) ** 2
print("dA/dk by autograd:", dA_dX[:, 1])
print("dA/dk by hand:    ", -F0 * (k - m * w ** 2) / D ** 1.5)
print("relative sensitivities (x/A)·dA/dx  [m, k, c, F0, omega]:\n", (X.detach() / A.detach()[:, None] * dA_dX))

The **relative sensitivity** $(x/A)\,\partial A/\partial x$ answers "if this input changes by 1%, by how many % does the output change?". Below resonance (first row) the stiffness has the largest effect (about −1.6) and the damping almost none (about −0.03). Close to resonance (second row) the damping suddenly matters (about −0.94) and the frequency's influence grows (from about 1.3 to 2.0): physically sensible, since damping is what limits the amplitude at resonance. The same three lines work for a trained surrogate model, which is one way engineers use neural surrogates: fast gradients for sensitivity studies and design optimisation.

`A.sum()` is a standard trick: autograd needs a scalar, and because each $A_i$ depends only on row $i$, the gradient of the sum with respect to row $i$ is exactly $\partial A_i / \partial X_i$.

## 10.2 Derivatives of derivatives: `create_graph=True`
Normally the backward pass is not recorded, so its result cannot be differentiated again. With `create_graph=True`, autograd records the backward computation too, and the returned gradient is itself a tensor in a graph.

In [ ]:
x = torch.linspace(0, 2 * math.pi, 5, requires_grad=True)
y = torch.sin(x)
(dy,) = torch.autograd.grad(y.sum(), x, create_graph=True)     # cos(x), and still differentiable
(d2y,) = torch.autograd.grad(dy.sum(), x)                      # -sin(x)
print("dy/dx  :", dy.detach().round(decimals=4), "\ncos(x) :", torch.cos(x).detach().round(decimals=4))
print("d2y/dx2:", d2y.round(decimals=4), "\n-sin(x):", (-torch.sin(x)).detach().round(decimals=4))

Uses you will meet: physics-informed neural networks (next section), gradient penalties in GANs, Hessian-vector products for analysing training, and meta-learning. Second derivatives cost roughly another backward pass of memory and time.

## 10.3 A physics-informed neural network (PINN)
Instead of fitting data, a PINN fits an **equation**. We solve the decay equation

$$\frac{du}{dt} = -\lambda u, \qquad u(0) = 1, \qquad t \in [0, 3],$$

whose exact solution is $u = e^{-\lambda t}$. A small network $u_\theta(t)$ is trained so that the **residual** $\frac{du_\theta}{dt} + \lambda u_\theta$ is zero at many "collocation" points, and the initial condition holds. The derivative $du_\theta/dt$ comes from autograd with `create_graph=True`, because the loss contains it and must be differentiated again with respect to $\theta$.

In [ ]:
LAM = 1.5
set_seed(0)
pinn = nn.Sequential(nn.Linear(1, 32), nn.Tanh(), nn.Linear(32, 32), nn.Tanh(), nn.Linear(32, 1)).to(device)
opt = torch.optim.Adam(pinn.parameters(), lr=1e-2)
t_col = torch.linspace(0, 3, 100, device=device)[:, None].requires_grad_(True)    # collocation points
t0 = torch.zeros(1, 1, device=device)

for step in range(200 if FAST_DEV_RUN else 3000):
    u = pinn(t_col)
    (du_dt,) = torch.autograd.grad(u.sum(), t_col, create_graph=True)   # du/dt at every collocation point
    residual = du_dt + LAM * u
    loss = (residual ** 2).mean() + (pinn(t0) - 1.0).pow(2).mean()      # equation + initial condition
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 1000 == 0:
        print(f"step {step:4d}  loss {loss.item():.2e}")

t_test = torch.linspace(0, 3, 200, device=device)[:, None]
with torch.no_grad():
    u_pred = pinn(t_test).squeeze(1).cpu()
u_true = torch.exp(-LAM * t_test.squeeze(1).cpu())
print(f"max |u_pinn - u_exact| = {(u_pred - u_true).abs().max():.2e}")
record("Ch10 PINN, max abs error", (u_pred - u_true).abs().max().item())

plt.figure(figsize=(6, 3.2))
plt.plot(t_test.cpu(), u_true, lw=3, alpha=0.4, label="exact e^{-λt}")
plt.plot(t_test.cpu(), u_pred, "--", label="PINN")
plt.xlabel("t"); plt.ylabel("u"); plt.legend(); plt.title("A network trained only on the equation"); plt.show()

No data point of the solution was used: the network learned it from the equation. PINNs are used for inverse problems (estimating an unknown parameter such as $\lambda$ from a few measurements, by making it a learnable `nn.Parameter`) and for combining sparse data with known physics. For simple forward problems like this one, classical solvers are faster and more accurate, which is worth remembering.

## 10.4 `torch.func`: functions instead of objects
`torch.func` (inspired by JAX) treats models as **pure functions** of their parameters and inputs. Its transforms compose freely:

| Transform | What it does |
| --- | --- |
| `grad(f)` | A function returning the gradient of scalar `f` with respect to its first argument |
| `vmap(f)` | Runs `f`, written for **one** example, over a batch, without a Python loop |
| `jacrev(f)` / `jacfwd(f)` | The full Jacobian (reverse or forward mode) |
| `functional_call(model, params, args)` | Calls an `nn.Module` with a given dictionary of parameters |

**Per-sample gradients.** `loss.backward()` gives the gradient of the batch **average**. Some techniques need the gradient of every example separately: differential privacy (clip each example's gradient), finding mislabelled or influential examples, or measuring gradient noise. Write the loss for one example, take `grad` of it, and `vmap` it over the batch:

In [ ]:
from torch.func import functional_call, grad, vmap, jacrev

set_seed(0)
net = nn.Sequential(nn.Linear(4, 16), nn.Tanh(), nn.Linear(16, 3))
params = {k: v.detach() for k, v in net.named_parameters()}
Xb, yb = torch.randn(8, 4), torch.randint(0, 3, (8,))

def loss_one(params, x, y):                                  # written for ONE example
    logits = functional_call(net, params, (x[None],))         # add a batch dimension of 1
    return F.cross_entropy(logits, y[None])

per_sample = vmap(grad(loss_one), in_dims=(None, 0, 0))(params, Xb, yb)   # params shared, data batched
print({k: tuple(v.shape) for k, v in per_sample.items()})              # each: (8, *parameter shape)

# Check against the slow way: one backward pass per example
loop = []
for i in range(8):
    net.zero_grad()
    F.cross_entropy(net(Xb[i:i + 1]), yb[i:i + 1]).backward()
    loop.append(net[0].weight.grad.clone())
print("matches a loop of backward passes:", torch.allclose(per_sample["0.weight"], torch.stack(loop), atol=1e-6))
print("their mean is the usual batch gradient:",
      torch.allclose(per_sample["0.weight"].mean(dim=0), torch.autograd.grad(F.cross_entropy(net(Xb), yb), net[0].weight)[0], atol=1e-6))

**Jacobians.** `jacrev(f)(x)` returns the matrix of all partial derivatives of the outputs of `f` with respect to `x`. For a classifier, the Jacobian of the logits with respect to the input shows which input features move which class score (a basic "saliency" analysis):

In [ ]:
def logits_one(x):                                            # one example in, (3,) logits out
    return net(x[None])[0]

J = jacrev(logits_one)(Xb[0])
print("Jacobian of 3 logits w.r.t. 4 inputs:", tuple(J.shape))
x0 = Xb[0].clone().requires_grad_(True)
row0 = torch.autograd.grad(logits_one(x0)[0], x0)[0]          # gradient of logit 0 = first row of J
print("first row matches autograd.grad:", torch.allclose(J[0], row0, atol=1e-6))

`vmap(jacrev(f))` gives the Jacobian of every example in a batch at once (Exercise 10.2), and `torch.func.hessian(f)` the matrix of second derivatives. Two cautions: functions passed to these transforms should not modify tensors in place or rely on randomness without care, and BatchNorm in training mode does not fit the "one example at a time" model (use the model in `eval()` mode).

## 🧪 Exercises

**Exercise 10.1: a second-order equation.** `★★★ · ~25 min`
Train a PINN for the undamped oscillator $\frac{d^2u}{dt^2} + \omega^2 u = 0$ with $u(0) = 1$, $\frac{du}{dt}(0) = 0$ and $\omega = 2$ on $t \in [0, \pi]$ (exact solution: $\cos 2t$). You need the **second** derivative: call `torch.autograd.grad` twice, both times with `create_graph=True`. Report the maximum error.

In [ ]:
# ✍️ Your attempt here.
# Hint: The initial-velocity condition also needs a derivative: evaluate the network at t = 0 with requires_grad and differentiate it.

In [ ]:
# @title ✅ Solution (double-click to reveal) { display-mode: "form" }
# Solutions only execute when RUN_SOLUTIONS = True (setup cell), so "Run all" stays fast.
if RUN_SOLUTIONS:
    OMEGA = 2.0
    set_seed(0)
    osc = nn.Sequential(nn.Linear(1, 64), nn.Tanh(), nn.Linear(64, 64), nn.Tanh(), nn.Linear(64, 1)).to(device)
    opt = torch.optim.Adam(osc.parameters(), lr=3e-3)
    t_col = torch.linspace(0, math.pi, 200, device=device)[:, None].requires_grad_(True)
    t0 = torch.zeros(1, 1, device=device, requires_grad=True)
    for step in range(200 if FAST_DEV_RUN else 6000):
        u = osc(t_col)
        (du,) = torch.autograd.grad(u.sum(), t_col, create_graph=True)
        (d2u,) = torch.autograd.grad(du.sum(), t_col, create_graph=True)
        u0 = osc(t0)
        (du0,) = torch.autograd.grad(u0.sum(), t0, create_graph=True)
        loss = ((d2u + OMEGA ** 2 * u) ** 2).mean() + (u0 - 1).pow(2).mean() + du0.pow(2).mean()
        opt.zero_grad(); loss.backward(); opt.step()
    tt = torch.linspace(0, math.pi, 300, device=device)[:, None]
    with torch.no_grad():
        err = (osc(tt) - torch.cos(OMEGA * tt)).abs().max().item()
    print(f"final loss {loss.item():.2e} | max |u - cos(2t)| = {err:.2e}")
    # Oscillatory solutions are harder for PINNs than decays: they need more steps, a wider network,
    # and a smaller learning rate. If the error is still large, train longer.
else:
    print("Solution not executed (RUN_SOLUTIONS = False).")

**Exercise 10.2: batched input Jacobians.** `★★☆ · ~15 min`
Write `batch_input_jacobian(model, X)` that returns, for a model mapping `(N, D)` inputs to `(N, K)` outputs, the tensor of shape `(N, K, D)` of every example's Jacobian, using `vmap` and `jacrev` (no Python loop over examples).

In [ ]:
# ✍️ Your attempt here.
# Hint: Define f(x) = model(x[None])[0] for one example; then vmap(jacrev(f))(X).

In [ ]:
# 🔎 Self-check for Exercise 10.2
def _test(fn):
    torch.manual_seed(0)
    mdl = nn.Sequential(nn.Linear(5, 7), nn.Tanh(), nn.Linear(7, 3))
    X = torch.randn(6, 5)
    J = fn(mdl, X)
    assert J.shape == (6, 3, 5), f"shape {tuple(J.shape)}, expected (6, 3, 5)"
    ref = torch.stack([torch.autograd.functional.jacobian(lambda x: mdl(x[None])[0], X[i]) for i in range(6)])
    assert torch.allclose(J, ref, atol=1e-6), "values differ from a per-example reference"
self_check("batch_input_jacobian", _test)

In [ ]:
# @title ✅ Solution (double-click to reveal) { display-mode: "form" }
# Solutions only execute when RUN_SOLUTIONS = True (setup cell), so "Run all" stays fast.
if RUN_SOLUTIONS:
    def batch_input_jacobian(model, X):
        f = lambda x: model(x[None])[0]
        return vmap(jacrev(f))(X)

    print(batch_input_jacobian(net, Xb).shape)
else:
    print("Solution not executed (RUN_SOLUTIONS = False).")

### 📌 Key takeaways
- `torch.autograd.grad(out, inp)` returns derivatives (also with respect to inputs) instead of accumulating them.
- `create_graph=True` makes a gradient differentiable again: needed for PINNs, gradient penalties and higher derivatives.
- `torch.func`: write the computation for one example, then `grad`, `vmap` and `jacrev` it; `functional_call` runs a module with explicit parameters.

### ✔️ Check yourself

Commit to an answer (say it out loud or write it down) *before* you expand it.

**1.** Why does the PINN loss need `create_graph=True` when computing du/dt?

<details>
<summary>Answer</summary>

The loss contains du/dt, and training differentiates the loss with respect to the parameters. Without `create_graph=True` the derivative is computed but not recorded, so the optimizer would get no gradient through the residual term.

</details>

**2.** What is the difference between the gradient from `loss.backward()` and per-sample gradients?

<details>
<summary>Answer</summary>

`backward()` on a mean loss gives the average of the per-example gradients (one tensor per parameter). Per-sample gradients keep one gradient per example (an extra batch dimension); their mean equals the usual gradient.

</details>

**3.** When would you prefer `jacfwd` over `jacrev`?

<details>
<summary>Answer</summary>

When the function has few inputs and many outputs: forward mode costs one pass per input, reverse mode one pass per output.

</details>

---
# Chapter 11. Performance: making training fast

🎯 **In this chapter you will learn to:** measure where training time goes; remove input-pipeline bottlenecks; train with automatic mixed precision (AMP) and a gradient scaler; speed up models with `torch.compile`; and reason about GPU memory.

**Rule zero of performance work: measure first.** Optimising the wrong part wastes your time. A training step has two halves that can each be the bottleneck: **producing the batch** (loading, decoding, augmenting) and **computing with it** (forward, backward, update).

🔁 **Starting a fresh session here?** Run the two Chapter 0 cells, then the hidden restore-point cell below. It re-creates the tools from earlier chapters that Chapter 11 uses. In a normal top-to-bottom run it does nothing.

In [ ]:
# @title 🔁 Restore point for Chapter 11 (needed only if you start a fresh session here) { display-mode: "form" }
if "device" not in globals():
    raise RuntimeError("Run the Chapter 0 setup cells first, then this cell.")
if "get_trained_cnn" not in globals():
    from torch.optim import lr_scheduler as S

    def param_groups_weight_decay(model, weight_decay):
        decay, no_decay = [], []
        for name, p in model.named_parameters():
            if not p.requires_grad:
                continue                                     # frozen parameters: leave them out
            (no_decay if p.ndim <= 1 else decay).append(p)   # biases and norm weights are 1-D
        return [{"params": decay, "weight_decay": weight_decay},
                {"params": no_decay, "weight_decay": 0.0}]

    class EarlyStopping:
        # Stop when `metric` has not improved by more than `min_delta` for `patience` epochs.
        # mode="min" for losses, mode="max" for accuracies. Keeps a copy of the best weights.
        def __init__(self, patience=5, min_delta=0.0, mode="min"):
            self.patience, self.min_delta, self.mode = patience, min_delta, mode
            self.best = math.inf if mode == "min" else -math.inf
            self.best_state, self.best_epoch, self.bad_epochs, self.epoch = None, 0, 0, 0

        def improved(self, value):
            if self.mode == "min":
                return value < self.best - self.min_delta
            return value > self.best + self.min_delta

        def step(self, value, model):
            # Call once per epoch. Returns True when training should stop.
            self.epoch += 1
            if self.improved(value):
                self.best, self.best_epoch, self.bad_epochs = value, self.epoch, 0
                self.best_state = copy.deepcopy(model.state_dict())
            else:
                self.bad_epochs += 1
            return self.bad_epochs >= self.patience

        def restore(self, model):
            model.load_state_dict(self.best_state)

    import contextlib

    def train_one_epoch(model, loader, loss_fn, optimizer, scheduler=None, max_grad_norm=None,
                        averaged_model=None, amp_dtype=None, scaler=None):
        # amp_dtype / scaler: mixed precision, explained in Chapter 11. Leave them as None until then.
        model.train()
        total_loss = torch.zeros((), device=device)                  # running sums stay ON the device:
        correct = torch.zeros((), device=device, dtype=torch.long)   # no CPU-GPU synchronisation per step
        n, grad_norms = 0, []
        for x, y in loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            amp = torch.autocast(device_type=device.type, dtype=amp_dtype) if amp_dtype else contextlib.nullcontext()
            with amp:
                logits = model(x)
                loss = loss_fn(logits, y)
            optimizer.zero_grad(set_to_none=True)
            if scaler is not None:                                   # float16 mixed precision (Chapter 11)
                scaler.scale(loss).backward()
                if max_grad_norm is not None:
                    scaler.unscale_(optimizer)                       # clip the REAL gradients, not the scaled ones
            else:
                loss.backward()
            if max_grad_norm is not None:
                grad_norms.append(nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm))
            if scaler is not None:
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()
            if scheduler is not None:                                # per-batch schedules (OneCycleLR, warm-up)
                scheduler.step()
            if averaged_model is not None:                           # weight averaging (Chapter 6)
                averaged_model.update_parameters(model)
            total_loss += loss.detach().float() * len(y)
            correct += (logits.argmax(dim=1) == y).sum()
            n += len(y)
        mean_norm = torch.stack(grad_norms).mean().item() if grad_norms else float("nan")
        return total_loss.item() / n, correct.item() / n, mean_norm   # one synchronisation per epoch


    def fit(model, train_loader, val_loader, optimizer, epochs, loss_fn=nn.CrossEntropyLoss(),
            scheduler=None, scheduler_interval="batch", early_stopping=None, eval_model=None, verbose=True, **train_kw):
        # scheduler_interval: "batch", "epoch" or "plateau" (ReduceLROnPlateau).
        # eval_model: evaluate (and select) a different model than the one trained, e.g. an EMA copy (Chapter 6).
        history = {k: [] for k in ["train_loss", "train_acc", "val_loss", "val_acc", "lr", "grad_norm", "time"]}
        eval_model = eval_model if eval_model is not None else model
        for epoch in range(1, epochs + 1):
            t0 = time.perf_counter()
            lr = optimizer.param_groups[0]["lr"]
            tr_loss, tr_acc, gnorm = train_one_epoch(model, train_loader, loss_fn, optimizer,
                                                     scheduler if scheduler_interval == "batch" else None, **train_kw)
            va_loss, va_acc = evaluate(eval_model, val_loader, loss_fn)
            if scheduler is not None and scheduler_interval == "epoch":
                scheduler.step()
            elif scheduler is not None and scheduler_interval == "plateau":
                scheduler.step(va_loss)
            elapsed = time.perf_counter() - t0
            for k, v in zip(history, (tr_loss, tr_acc, va_loss, va_acc, lr, gnorm, elapsed)):
                history[k].append(v)
            if verbose:
                print(f"epoch {epoch:2d} | lr {lr:.2e} | train {tr_loss:.4f} / {tr_acc:.4f} | "
                      f"val {va_loss:.4f} / {va_acc:.4f} | {elapsed:4.1f}s")
            if early_stopping is not None and early_stopping.step(va_loss, eval_model):
                print(f"early stopping: no improvement for {early_stopping.patience} epochs")
                break
        if early_stopping is not None:                       # select the epoch with the best validation LOSS...
            early_stopping.restore(eval_model)
            sel_epoch = early_stopping.best_epoch
        else:
            sel_epoch = len(history["val_loss"])            # ...or keep the last epoch
        sel_loss, sel_acc = history["val_loss"][sel_epoch - 1], history["val_acc"][sel_epoch - 1]
        history["selected"] = {"epoch": sel_epoch, "val_loss": sel_loss, "val_acc": sel_acc}
        if verbose:
            print(f"selected model: epoch {sel_epoch}, val loss {sel_loss:.4f}, val acc {sel_acc:.4f}")
        return history


    def plot_history(histories, keys=("val_loss", "val_acc", "lr")):
        fig, axes = plt.subplots(1, len(keys), figsize=(5 * len(keys), 3.2))
        for ax, key in zip(axes, keys):
            for name, h in histories.items():
                ax.plot(range(1, len(h[key]) + 1), h[key], "o-", ms=3, label=name)
            ax.set_title(key); ax.set_xlabel("epoch")
        axes[0].legend(fontsize=8); plt.tight_layout(); plt.show()

    # Small in-memory subsets (normalised exactly like fm_transform) for quick experiments
    def _fm_subset(idx):
        x = ((raw_train.data[idx].float() / 255 - FM_MEAN) / FM_STD).unsqueeze(1)
        return TensorDataset(x, raw_train.targets[idx])

    small_ds = _fm_subset(FM_TRAIN_IDX[:4096])      # 4,096 training images
    small_val = _fm_subset(FM_VAL_IDX[:2000])       # 2,000 validation images

    def save_checkpoint(path, model, optimizer, scheduler=None, epoch=0, best_metric=None,
                        loader_generator=None, config=None):
        np_state = np.random.get_state()
        ckpt = {
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict() if scheduler is not None else None,
            "epoch": epoch,
            "best_metric": best_metric,
            "config": config or {},
            "rng": {
                "torch": torch.get_rng_state(),
                "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [],
                "python": random.getstate(),
                # NumPy's state contains an array; store it as a list so the safe loader accepts it
                "numpy": (np_state[0], np_state[1].tolist(), *np_state[2:]),
            },
            "loader_generator": loader_generator.get_state() if loader_generator is not None else None,
        }
        tmp = Path(str(path) + ".tmp")
        torch.save(ckpt, tmp)             # write to a temporary file first...
        os.replace(tmp, path)             # ...then rename: an interruption can never leave a half-written checkpoint


    def load_checkpoint(path, model, optimizer=None, scheduler=None, loader_generator=None):
        ckpt = torch.load(path, map_location=device, weights_only=True)   # safe loading: tensors and plain data only
        model.load_state_dict(ckpt["model"])
        if optimizer is not None:
            optimizer.load_state_dict(ckpt["optimizer"])
        if scheduler is not None and ckpt["scheduler"] is not None:
            scheduler.load_state_dict(ckpt["scheduler"])
        rng = ckpt["rng"]
        torch.set_rng_state(rng["torch"].cpu())
        if torch.cuda.is_available() and rng["cuda"]:
            torch.cuda.set_rng_state_all([s.cpu() for s in rng["cuda"]])
        random.setstate(rng["python"])
        name, keys, *rest = rng["numpy"]
        np.random.set_state((name, np.array(keys, dtype=np.uint32), *rest))
        if loader_generator is not None and ckpt["loader_generator"] is not None:
            loader_generator.set_state(ckpt["loader_generator"].cpu())
        return ckpt

    def conv_bn_relu(c_in, c_out):
        return nn.Sequential(nn.Conv2d(c_in, c_out, 3, padding=1, bias=False), nn.BatchNorm2d(c_out), nn.ReLU(inplace=True))


    class SmallCNN(nn.Module):
        def __init__(self, num_classes=10, width=32, p_drop=0.3):
            super().__init__()
            w = width
            self.features = nn.Sequential(
                conv_bn_relu(1, w), conv_bn_relu(w, w), nn.MaxPool2d(2),              # 28x28 -> 14x14
                conv_bn_relu(w, 2 * w), conv_bn_relu(2 * w, 2 * w), nn.MaxPool2d(2),  # 14x14 -> 7x7
                conv_bn_relu(2 * w, 4 * w),                                           # 7x7, 128 channels
                nn.AdaptiveAvgPool2d(1),                                              # -> 1x1
            )
            self.head = nn.Sequential(nn.Flatten(), nn.Dropout(p_drop), nn.Linear(4 * w, num_classes))

        def forward(self, x):
            return self.head(self.features(x))

    CNN_PATH = CKPT_DIR / "smallcnn_fashion.pt"

    def train_small_cnn(epochs=12, seed=0):
        # The recipe of Section 7.7: AdamW with weight-decay groups and a OneCycle schedule.
        set_seed(seed)
        model = SmallCNN().to(device)
        E = n_epochs(epochs)
        opt = torch.optim.AdamW(param_groups_weight_decay(model, 0.05), lr=1e-3)
        sched = S.OneCycleLR(opt, max_lr=3e-3, total_steps=E * len(train_loader))
        hist = fit(model, train_loader, val_loader, opt, E, scheduler=sched,
                   early_stopping=EarlyStopping(patience=100))        # patience=100: just keep the best epoch
        torch.save({"state_dict": model.state_dict()}, CNN_PATH)
        return model, hist

    def get_trained_cnn():
        # Load the CNN trained in Chapter 7 if it was saved; otherwise train it now.
        model = SmallCNN().to(device)
        if CNN_PATH.exists():
            model.load_state_dict(torch.load(CNN_PATH, map_location=device, weights_only=True)["state_dict"])
            print(f"loaded the trained SmallCNN from {CNN_PATH}")
        else:
            print("no saved SmallCNN found: training one with the recipe of Section 7.7 ...")
            model, _ = train_small_cnn()
        return model.eval()
    print("Restored the tools defined in earlier chapters.")

## 11.1 Where does the time go?
We measure the **Notebook 1 pipeline** (a `DataLoader` that decodes and transforms every image in Python) against the computation it feeds.

In [ ]:
def time_data(loader, n_batches=50):
    it = iter(loader)
    next(it)                                   # warm-up: start-up costs (e.g. launching workers) excluded
    t0 = time.perf_counter()
    n = 0
    for x, y in it:
        n += 1
        if n >= n_batches:
            break
    return (time.perf_counter() - t0) / max(n, 1)

def time_compute(model, x, y, n_steps=30, amp_dtype=None):
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
    scaler = torch.amp.GradScaler(device.type) if amp_dtype == torch.float16 else None
    model.train()
    for step in range(n_steps + 5):
        if step == 5:                    # 5 warm-up steps excluded from the timing
            sync(); t0 = time.perf_counter()
        with torch.autocast(device_type=device.type, dtype=amp_dtype or torch.float32, enabled=amp_dtype is not None):
            loss = F.cross_entropy(model(x), y)
        opt.zero_grad(set_to_none=True)
        if scaler:
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        else:
            loss.backward(); opt.step()
    sync()
    return (time.perf_counter() - t0) / n_steps

dl_train, dl_val, _ = fashion_loaders()          # Notebook 1 style: Dataset + per-image transforms + DataLoader
xb, yb = next(iter(dl_train))
xb, yb = xb.to(device), yb.to(device)
t_data = time_data(dl_train)
t_comp = time_compute(SmallCNN().to(device), xb, yb)
print(f"per batch of 128: producing the batch {t_data * 1e3:6.1f} ms | computing with it {t_comp * 1e3:6.1f} ms")
print("bottleneck:", "DATA" if t_data > t_comp else "COMPUTE")

Compare the two numbers on your machine. With a model this small on a GPU, the Notebook 1 pipeline is often the slower half: two CPU cores transform every 28×28 image one at a time in Python, while the GPU finishes the maths quickly and then waits. If that is what you see, making the GPU faster would not help at all until the data problem is solved. (The first batch is excluded from the timing, because it includes one-off costs such as starting the worker processes.)

## 11.2 Feeding the GPU
General remedies for a data bottleneck:
- more worker processes (`num_workers`), as many as the CPU cores allow;
- `pin_memory=True` in the loader and `x.to(device, non_blocking=True)`, so that the copy to the GPU overlaps with computation;
- `persistent_workers=True`, so workers are not restarted every epoch;
- for images, decode and resize **once** and store the results, or do augmentation on the GPU, batched.

**For small datasets, preprocess once.** This is what the toolkit's `TensorLoader` has done since Chapter 0: the whole split is normalised once into a tensor, and batches are sliced from it. If the data also fits in **GPU** memory (Fashion-MNIST is 55,000 × 784 floats ≈ 172 MB), put the tensors on the GPU and even the copy per batch disappears. Many fast-training recipes for CIFAR-10 do exactly this.

In [ ]:
gpu_train_loader = TensorLoader(*[t.to(device) for t in fashion_tensors("train")], batch_size=128, shuffle=True, drop_last=True)
gpu_val_loader = TensorLoader(*[t.to(device) for t in fashion_tensors("val")], batch_size=1024)

x_check, _ = next(iter(DataLoader(Subset(datasets.FashionMNIST(DATA_DIR, train=True, transform=fm_transform),
                                         FM_TRAIN_IDX[:4].tolist()), batch_size=4)))
print("TensorLoader data == fm_transform output:", torch.allclose(x_check, fashion_tensors("train")[0][:4], atol=1e-6))
for name, loader in [("DataLoader + per-image transforms", dl_train),
                     ("TensorLoader, CPU tensors (our default)", train_loader),
                     ("TensorLoader, tensors on the device", gpu_train_loader)]:
    print(f"{name:42s} {time_data(loader) * 1e3:8.3f} ms per batch")

`TensorLoader` behaves like a `DataLoader` as far as our `fit` is concerned (it is iterable and has a length), so it plugs straight in. The trade-off: everything must fit in memory, and per-example random augmentation must be done on whole batches. We compare epoch times in §11.4.

## 11.3 Mixed precision (AMP)
Modern GPUs have special units ("tensor cores") that multiply 16-bit floating-point numbers several times faster than 32-bit ones, and 16-bit tensors halve the memory traffic. **Automatic mixed precision** runs the operations that benefit (matrix multiplications, convolutions) in 16 bits, and keeps the numerically sensitive ones (reductions, softmax, losses, the weight update) in 32 bits. The master copy of the weights stays in `float32`.

| 16-bit format | Range | Precision | Needs a gradient scaler? | GPUs |
| --- | --- | --- | --- | --- |
| `float16` | up to 65,504; tiny values underflow to 0 | about 3 decimal digits | **yes** | all CUDA GPUs with tensor cores, including the T4 |
| `bfloat16` | same as `float32` | about 2 decimal digits | no | Ampere (A100, RTX 30xx) and newer, TPUs |

**Why `float16` needs a scaler.** Many gradients are tiny, say 1e-8, and `float16` cannot represent values below about 6e-8, so they would become exactly zero. The `GradScaler` multiplies the loss by a large factor (for example 65,536) before `backward()`, which scales every gradient up into the representable range. It then divides the gradients by the same factor before the optimizer step. If the scaled gradients overflow to `inf`, it skips that step and lowers the factor. You barely notice it, except for a few skipped steps at the start.

The pattern, with the clipping detail that people often get wrong:

```python
scaler = torch.amp.GradScaler("cuda")                        # only for float16
with torch.autocast(device_type="cuda", dtype=torch.float16):
    loss = loss_fn(model(x), y)                              # forward pass and loss under autocast
optimizer.zero_grad()
scaler.scale(loss).backward()                                # backward OUTSIDE autocast, on the scaled loss
scaler.unscale_(optimizer)                                   # un-scale first if you clip: the norm must be the real one
torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
scaler.step(optimizer)                                       # skips the step if gradients are inf/nan
scaler.update()                                              # adjust the scale factor
```

Our `train_one_epoch` from Chapter 4 already contains exactly this pattern, switched off by default: pass `amp_dtype=AMP_DTYPE` (chosen for your GPU in the setup cell) and, for `float16`, `scaler=torch.amp.GradScaler("cuda")`, and `fit` hands them through (that is what its `**train_kw` is for). Scroll back to Chapter 4 and find the `scaler` branches: they implement the five lines above.

In [ ]:
print("AMP dtype chosen for this machine in the setup cell:", AMP_DTYPE)

`x.to(device, non_blocking=True)` only overlaps the copy with computation when the source is in pinned memory; otherwise it behaves like a normal copy, so it is always safe to write.

You may see a warning that `lr_scheduler.step()` was called before `optimizer.step()`. With `float16`, the scaler deliberately skips the first step or two while it calibrates its scale factor, so the scheduler appears to run first. It is harmless.

## 11.4 The payoff, measured
Same CNN, same number of examples, four configurations. On a CPU the AMP row is skipped.

In [ ]:
def time_epoch(loader, amp_dtype=None, compile_model=False):
    set_seed(0)
    model = SmallCNN().to(device)
    if compile_model:
        model = torch.compile(model)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
    scaler = torch.amp.GradScaler(device.type) if amp_dtype == torch.float16 else None
    train_one_epoch(model, loader, nn.CrossEntropyLoss(), opt, amp_dtype=amp_dtype, scaler=scaler)   # warm-up / compile
    sync(); t0 = time.perf_counter()
    train_one_epoch(model, loader, nn.CrossEntropyLoss(), opt, amp_dtype=amp_dtype, scaler=scaler)
    sync()
    return time.perf_counter() - t0, model

timings = {}
timings["DataLoader + transforms, float32"], _ = time_epoch(dl_train)
timings["TensorLoader (CPU tensors), float32"], _ = time_epoch(train_loader)
timings["TensorLoader (device tensors), float32"], _ = time_epoch(gpu_train_loader)
if AMP_DTYPE is not None:
    timings[f"TensorLoader (device), AMP {str(AMP_DTYPE).split('.')[-1]}"], _ = time_epoch(gpu_train_loader, AMP_DTYPE)
for name, t in timings.items():
    print(f"{name:42s} {t:6.2f} s per epoch")

For a small model like this one, removing the per-image Python work usually gives the biggest win, and AMP adds a further speed-up that grows with model size (it is large for big convolutional networks and modest for a network this small). Your numbers depend on the GPU, so trust the measurement, not the text.

### Hidden synchronisations
GPU work is **asynchronous**: Python queues kernels and runs ahead. Anything that needs a GPU value **on the CPU** forces Python to wait until all queued work is done: `.item()`, `.cpu()`, `.tolist()`, `print(tensor)`, and Python `if` statements on tensor values. One `.item()` per step therefore turns a pipeline into stop-and-go. That is why Chapter 4's `train_one_epoch` accumulates the loss and accuracy **on the device** and calls `.item()` once per epoch, while Notebook 1's version called it on every batch. The cell measures the difference for a small model, where the effect is largest; on a CPU there is nothing to wait for, so both variants take the same time.

In [ ]:
def ms_per_step(item_every_step, n_steps=200):
    set_seed(0)
    model = build_mlp().to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
    x, y = next(iter(gpu_train_loader))
    on_device, on_cpu = torch.zeros((), device=device), 0.0
    for step in range(n_steps + 10):
        if step == 10:                                        # 10 warm-up steps excluded
            sync(); t0 = time.perf_counter()
        loss = F.cross_entropy(model(x), y)
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
        if item_every_step:
            on_cpu += loss.item()                            # waits for the GPU every step
        else:
            on_device += loss.detach()                       # no waiting
    sync()
    return (time.perf_counter() - t0) / n_steps * 1e3

n = 20 if FAST_DEV_RUN else 200
print(f".item() every step:  {ms_per_step(True, n):.3f} ms per step")
print(f"sums on the device: {ms_per_step(False, n):.3f} ms per step")

How large the gap is depends on the model and the GPU: it matters most when each step's GPU work is short compared with Python's overhead. The habit costs nothing, so keep metrics on the device inside loops, and log or print every N steps rather than every step.

## 11.5 `torch.compile`
`torch.compile(model)` captures the model's Python code as a graph the first time it runs, and generates optimised GPU kernels for it, for example **fusing** a convolution's output, BatchNorm and ReLU into fewer passes over memory. The first call takes extra time (from several seconds to a few minutes, depending on the model and the machine); later calls are faster.

In [ ]:
if device.type == "cuda":
    t0 = time.perf_counter()
    t_comp, compiled = time_epoch(gpu_train_loader, AMP_DTYPE, compile_model=True)
    print(f"compiled + AMP: {t_comp:.2f} s per epoch (total time including compilation: {time.perf_counter() - t0:.0f} s)")
    print("state_dict keys of a compiled model:", list(compiled.state_dict())[:2])
else:
    print("Skipping torch.compile timing on this device (it shines on CUDA GPUs).")

Things to know about `torch.compile`:
- **Recompilation.** A new input shape can trigger a new compilation. A smaller final batch is a classic cause, which is another reason to use `drop_last=True` for training.
- **Graph breaks.** Python code the compiler cannot capture (printing, `.item()`, data-dependent `if` statements) splits the graph and reduces the gain. `torch._dynamo.explain(model)(x)` lists them.
- **⚠️ State-dict keys get a prefix.** The compiled wrapper stores the model as `_orig_mod`, so its keys look like `_orig_mod.features.0.0.weight`. Save `compiled._orig_mod.state_dict()` (or the original model object, which shares the same parameters), or strip the prefix when loading.
- Options such as `mode="max-autotune"` try harder (longer compile, sometimes faster kernels).

## 11.6 More switches you will see in training scripts

| Setting | What it does |
| --- | --- |
| `torch.backends.cudnn.benchmark = True` | Tries several convolution algorithms on the first batch and keeps the fastest. Good when input sizes are fixed; hurts with varying sizes |
| `torch.set_float32_matmul_precision("high")` | Lets `float32` matrix multiplications use the faster TF32 format on Ampere+ GPUs |
| `model.to(memory_format=torch.channels_last)` | Stores images as `(N, H, W, C)` in memory while keeping the `(N, C, H, W)` shape, which suits tensor cores. This is purely a **stride** change (Chapter 1): see below |
| `torch.utils.checkpoint.checkpoint(block, x)` | **Activation checkpointing**: do not store a block's activations; recompute them during backward. Trades about 30% more compute for much less memory |

In [ ]:
x = torch.randn(8, 3, 32, 32)
xc = x.to(memory_format=torch.channels_last)
print("shape unchanged:", tuple(xc.shape), "| strides NCHW:", x.stride(), "-> channels_last:", xc.stride())
print("same values:", torch.equal(x, xc), "| contiguous in the usual sense:", xc.is_contiguous())

The strides `(3072, 1, 96, 3)` say: neighbouring **channels** are adjacent in memory (stride 1), then width, then height. Same tensor, different layout, exactly as in Chapter 1.

## 11.7 GPU memory
During training, GPU memory holds:
1. **weights** (4 bytes per parameter in `float32`);
2. **gradients** (the same again);
3. **optimizer state** (2× the weights for Adam);
4. **activations** saved for the backward pass, which grow **linearly with the batch size** and usually dominate for vision models.

When you hit `CUDA out of memory`: lower the batch size (and use gradient accumulation, Chapter 4, to keep the effective batch), use AMP (16-bit activations), or use activation checkpointing. `torch.cuda.max_memory_allocated()` reports the peak.

## 🧪 Exercises

**Exercise 11.1: memory versus batch size.** `★★☆ · ~15 min · GPU only`
For `SmallCNN`, measure the peak GPU memory of one training step (forward, backward, step) for batch sizes 64, 256 and 1024, in `float32` and with AMP. Reset the counter with `torch.cuda.reset_peak_memory_stats()` before each measurement. Is the growth linear in the batch size? How much does AMP save?

In [ ]:
# ✍️ Your attempt here.

In [ ]:
# @title ✅ Solution (double-click to reveal) { display-mode: "form" }
# Solutions only execute when RUN_SOLUTIONS = True (setup cell), so "Run all" stays fast.
if RUN_SOLUTIONS:
    if device.type == "cuda":
        for amp in (None, AMP_DTYPE):
            for bs in (64, 256, 1024):
                model = SmallCNN().to(device)
                opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
                x = torch.randn(bs, 1, 28, 28, device=device); y = torch.randint(0, 10, (bs,), device=device)
                torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
                with torch.autocast("cuda", dtype=amp or torch.float32, enabled=amp is not None):
                    loss = F.cross_entropy(model(x), y)
                loss.backward(); opt.step()
                print(f"{str(amp):15s} batch {bs:5d}: peak {torch.cuda.max_memory_allocated() / 2**20:7.1f} MiB")
                del model, opt, x, y, loss
    else:
        print("This exercise needs a CUDA GPU.")
    # The peak grows roughly linearly with the batch size (activations), on top of a constant part
    # (weights, gradients, optimizer state: a few MiB for this small model). AMP stores many activations
    # in 16 bits, so the batch-dependent part shrinks substantially.
else:
    print("Solution not executed (RUN_SOLUTIONS = False).")

**Exercise 11.2: train fast.** `★☆☆ · ~10 min`
Train `SmallCNN` with `fit` for 12 epochs on `gpu_train_loader` / `gpu_val_loader`, with AMP (pass `amp_dtype=AMP_DTYPE` and, for `float16`, `scaler=torch.amp.GradScaler("cuda")`), AdamW and OneCycle as in §7.7. Compare the accuracy and total time with §7.7.

In [ ]:
# ✍️ Your attempt here.

In [ ]:
# @title ✅ Solution (double-click to reveal) { display-mode: "form" }
# Solutions only execute when RUN_SOLUTIONS = True (setup cell), so "Run all" stays fast.
if RUN_SOLUTIONS:
    set_seed(0)
    fast_cnn = SmallCNN().to(device)
    E = n_epochs(12)
    opt = torch.optim.AdamW(param_groups_weight_decay(fast_cnn, 0.05), lr=1e-3)
    sched = S.OneCycleLR(opt, max_lr=3e-3, total_steps=E * len(gpu_train_loader))
    scaler = torch.amp.GradScaler(device.type) if AMP_DTYPE == torch.float16 else None
    t0 = time.perf_counter()
    h = fit(fast_cnn, gpu_train_loader, gpu_val_loader, opt, E, scheduler=sched, amp_dtype=AMP_DTYPE, scaler=scaler)
    print(f"total {time.perf_counter() - t0:.0f} s | val acc of the kept (last) epoch {h['selected']['val_acc']:.4f}")
    # Same accuracy as in 7.7 (to within seed noise), in a fraction of the time. Mixed precision does
    # not cost accuracy in a normal training run.
else:
    print("Solution not executed (RUN_SOLUTIONS = False).")

### 📌 Key takeaways
- Measure data time and compute time separately; fix the bottleneck you actually have.
- Small datasets: keep them on the GPU. Otherwise: workers, pinned memory, non-blocking copies.
- AMP: `autocast` around forward + loss; for `float16` a `GradScaler` (and `unscale_` before clipping); `bfloat16` needs no scaler.
- `torch.compile`: compile time up front, faster steps after; watch for recompilation and the `_orig_mod.` prefix.
- Activation memory scales with batch size; trade it with accumulation, AMP or checkpointing.

### ✔️ Check yourself

Commit to an answer (say it out loud or write it down) *before* you expand it.

**1.** Why must you call `scaler.unscale_(optimizer)` before `clip_grad_norm_` when using a `GradScaler`?

<details>
<summary>Answer</summary>

The gradients are multiplied by the scale factor (e.g. 65,536). Clipping them before unscaling would compare a hugely inflated norm against `max_norm` and shrink the real gradients by that factor.

</details>

**2.** Why does `bfloat16` not need a gradient scaler while `float16` does?

<details>
<summary>Answer</summary>

`bfloat16` has the same exponent range as `float32`, so small gradients do not underflow; `float16` has a narrow range, and tiny gradients would round to zero without scaling.

</details>

**3.** Your GPU utilisation is around 30% during training. What do you check first?

<details>
<summary>Answer</summary>

Whether the input pipeline is the bottleneck: time the loader alone versus the compute. Then add workers, pin memory, preprocess once, or move the data to the GPU.

</details>

---
# Chapter 12. Debugging training

🎯 **In this chapter you will learn to:** follow a step-by-step protocol that finds most training bugs quickly; inspect activations with forward hooks; check gradient flow layer by layer; hunt down `NaN`s with anomaly detection; and map common symptoms to their causes.

Training bugs rarely crash. The code runs, the loss goes down a little, and the model is just worse than it should be. You need a **systematic** approach, because staring at code is slow.

🔁 **Starting a fresh session here?** Run the two Chapter 0 cells, then the hidden restore-point cell below. It re-creates the tools from earlier chapters that Chapter 12 uses. In a normal top-to-bottom run it does nothing.

In [ ]:
# @title 🔁 Restore point for Chapter 12 (needed only if you start a fresh session here) { display-mode: "form" }
if "device" not in globals():
    raise RuntimeError("Run the Chapter 0 setup cells first, then this cell.")
if "get_trained_cnn" not in globals():
    from torch.optim import lr_scheduler as S

    def param_groups_weight_decay(model, weight_decay):
        decay, no_decay = [], []
        for name, p in model.named_parameters():
            if not p.requires_grad:
                continue                                     # frozen parameters: leave them out
            (no_decay if p.ndim <= 1 else decay).append(p)   # biases and norm weights are 1-D
        return [{"params": decay, "weight_decay": weight_decay},
                {"params": no_decay, "weight_decay": 0.0}]

    class EarlyStopping:
        # Stop when `metric` has not improved by more than `min_delta` for `patience` epochs.
        # mode="min" for losses, mode="max" for accuracies. Keeps a copy of the best weights.
        def __init__(self, patience=5, min_delta=0.0, mode="min"):
            self.patience, self.min_delta, self.mode = patience, min_delta, mode
            self.best = math.inf if mode == "min" else -math.inf
            self.best_state, self.best_epoch, self.bad_epochs, self.epoch = None, 0, 0, 0

        def improved(self, value):
            if self.mode == "min":
                return value < self.best - self.min_delta
            return value > self.best + self.min_delta

        def step(self, value, model):
            # Call once per epoch. Returns True when training should stop.
            self.epoch += 1
            if self.improved(value):
                self.best, self.best_epoch, self.bad_epochs = value, self.epoch, 0
                self.best_state = copy.deepcopy(model.state_dict())
            else:
                self.bad_epochs += 1
            return self.bad_epochs >= self.patience

        def restore(self, model):
            model.load_state_dict(self.best_state)

    import contextlib

    def train_one_epoch(model, loader, loss_fn, optimizer, scheduler=None, max_grad_norm=None,
                        averaged_model=None, amp_dtype=None, scaler=None):
        # amp_dtype / scaler: mixed precision, explained in Chapter 11. Leave them as None until then.
        model.train()
        total_loss = torch.zeros((), device=device)                  # running sums stay ON the device:
        correct = torch.zeros((), device=device, dtype=torch.long)   # no CPU-GPU synchronisation per step
        n, grad_norms = 0, []
        for x, y in loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            amp = torch.autocast(device_type=device.type, dtype=amp_dtype) if amp_dtype else contextlib.nullcontext()
            with amp:
                logits = model(x)
                loss = loss_fn(logits, y)
            optimizer.zero_grad(set_to_none=True)
            if scaler is not None:                                   # float16 mixed precision (Chapter 11)
                scaler.scale(loss).backward()
                if max_grad_norm is not None:
                    scaler.unscale_(optimizer)                       # clip the REAL gradients, not the scaled ones
            else:
                loss.backward()
            if max_grad_norm is not None:
                grad_norms.append(nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm))
            if scaler is not None:
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()
            if scheduler is not None:                                # per-batch schedules (OneCycleLR, warm-up)
                scheduler.step()
            if averaged_model is not None:                           # weight averaging (Chapter 6)
                averaged_model.update_parameters(model)
            total_loss += loss.detach().float() * len(y)
            correct += (logits.argmax(dim=1) == y).sum()
            n += len(y)
        mean_norm = torch.stack(grad_norms).mean().item() if grad_norms else float("nan")
        return total_loss.item() / n, correct.item() / n, mean_norm   # one synchronisation per epoch


    def fit(model, train_loader, val_loader, optimizer, epochs, loss_fn=nn.CrossEntropyLoss(),
            scheduler=None, scheduler_interval="batch", early_stopping=None, eval_model=None, verbose=True, **train_kw):
        # scheduler_interval: "batch", "epoch" or "plateau" (ReduceLROnPlateau).
        # eval_model: evaluate (and select) a different model than the one trained, e.g. an EMA copy (Chapter 6).
        history = {k: [] for k in ["train_loss", "train_acc", "val_loss", "val_acc", "lr", "grad_norm", "time"]}
        eval_model = eval_model if eval_model is not None else model
        for epoch in range(1, epochs + 1):
            t0 = time.perf_counter()
            lr = optimizer.param_groups[0]["lr"]
            tr_loss, tr_acc, gnorm = train_one_epoch(model, train_loader, loss_fn, optimizer,
                                                     scheduler if scheduler_interval == "batch" else None, **train_kw)
            va_loss, va_acc = evaluate(eval_model, val_loader, loss_fn)
            if scheduler is not None and scheduler_interval == "epoch":
                scheduler.step()
            elif scheduler is not None and scheduler_interval == "plateau":
                scheduler.step(va_loss)
            elapsed = time.perf_counter() - t0
            for k, v in zip(history, (tr_loss, tr_acc, va_loss, va_acc, lr, gnorm, elapsed)):
                history[k].append(v)
            if verbose:
                print(f"epoch {epoch:2d} | lr {lr:.2e} | train {tr_loss:.4f} / {tr_acc:.4f} | "
                      f"val {va_loss:.4f} / {va_acc:.4f} | {elapsed:4.1f}s")
            if early_stopping is not None and early_stopping.step(va_loss, eval_model):
                print(f"early stopping: no improvement for {early_stopping.patience} epochs")
                break
        if early_stopping is not None:                       # select the epoch with the best validation LOSS...
            early_stopping.restore(eval_model)
            sel_epoch = early_stopping.best_epoch
        else:
            sel_epoch = len(history["val_loss"])            # ...or keep the last epoch
        sel_loss, sel_acc = history["val_loss"][sel_epoch - 1], history["val_acc"][sel_epoch - 1]
        history["selected"] = {"epoch": sel_epoch, "val_loss": sel_loss, "val_acc": sel_acc}
        if verbose:
            print(f"selected model: epoch {sel_epoch}, val loss {sel_loss:.4f}, val acc {sel_acc:.4f}")
        return history


    def plot_history(histories, keys=("val_loss", "val_acc", "lr")):
        fig, axes = plt.subplots(1, len(keys), figsize=(5 * len(keys), 3.2))
        for ax, key in zip(axes, keys):
            for name, h in histories.items():
                ax.plot(range(1, len(h[key]) + 1), h[key], "o-", ms=3, label=name)
            ax.set_title(key); ax.set_xlabel("epoch")
        axes[0].legend(fontsize=8); plt.tight_layout(); plt.show()

    # Small in-memory subsets (normalised exactly like fm_transform) for quick experiments
    def _fm_subset(idx):
        x = ((raw_train.data[idx].float() / 255 - FM_MEAN) / FM_STD).unsqueeze(1)
        return TensorDataset(x, raw_train.targets[idx])

    small_ds = _fm_subset(FM_TRAIN_IDX[:4096])      # 4,096 training images
    small_val = _fm_subset(FM_VAL_IDX[:2000])       # 2,000 validation images

    def save_checkpoint(path, model, optimizer, scheduler=None, epoch=0, best_metric=None,
                        loader_generator=None, config=None):
        np_state = np.random.get_state()
        ckpt = {
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict() if scheduler is not None else None,
            "epoch": epoch,
            "best_metric": best_metric,
            "config": config or {},
            "rng": {
                "torch": torch.get_rng_state(),
                "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [],
                "python": random.getstate(),
                # NumPy's state contains an array; store it as a list so the safe loader accepts it
                "numpy": (np_state[0], np_state[1].tolist(), *np_state[2:]),
            },
            "loader_generator": loader_generator.get_state() if loader_generator is not None else None,
        }
        tmp = Path(str(path) + ".tmp")
        torch.save(ckpt, tmp)             # write to a temporary file first...
        os.replace(tmp, path)             # ...then rename: an interruption can never leave a half-written checkpoint


    def load_checkpoint(path, model, optimizer=None, scheduler=None, loader_generator=None):
        ckpt = torch.load(path, map_location=device, weights_only=True)   # safe loading: tensors and plain data only
        model.load_state_dict(ckpt["model"])
        if optimizer is not None:
            optimizer.load_state_dict(ckpt["optimizer"])
        if scheduler is not None and ckpt["scheduler"] is not None:
            scheduler.load_state_dict(ckpt["scheduler"])
        rng = ckpt["rng"]
        torch.set_rng_state(rng["torch"].cpu())
        if torch.cuda.is_available() and rng["cuda"]:
            torch.cuda.set_rng_state_all([s.cpu() for s in rng["cuda"]])
        random.setstate(rng["python"])
        name, keys, *rest = rng["numpy"]
        np.random.set_state((name, np.array(keys, dtype=np.uint32), *rest))
        if loader_generator is not None and ckpt["loader_generator"] is not None:
            loader_generator.set_state(ckpt["loader_generator"].cpu())
        return ckpt

    def conv_bn_relu(c_in, c_out):
        return nn.Sequential(nn.Conv2d(c_in, c_out, 3, padding=1, bias=False), nn.BatchNorm2d(c_out), nn.ReLU(inplace=True))


    class SmallCNN(nn.Module):
        def __init__(self, num_classes=10, width=32, p_drop=0.3):
            super().__init__()
            w = width
            self.features = nn.Sequential(
                conv_bn_relu(1, w), conv_bn_relu(w, w), nn.MaxPool2d(2),              # 28x28 -> 14x14
                conv_bn_relu(w, 2 * w), conv_bn_relu(2 * w, 2 * w), nn.MaxPool2d(2),  # 14x14 -> 7x7
                conv_bn_relu(2 * w, 4 * w),                                           # 7x7, 128 channels
                nn.AdaptiveAvgPool2d(1),                                              # -> 1x1
            )
            self.head = nn.Sequential(nn.Flatten(), nn.Dropout(p_drop), nn.Linear(4 * w, num_classes))

        def forward(self, x):
            return self.head(self.features(x))

    CNN_PATH = CKPT_DIR / "smallcnn_fashion.pt"

    def train_small_cnn(epochs=12, seed=0):
        # The recipe of Section 7.7: AdamW with weight-decay groups and a OneCycle schedule.
        set_seed(seed)
        model = SmallCNN().to(device)
        E = n_epochs(epochs)
        opt = torch.optim.AdamW(param_groups_weight_decay(model, 0.05), lr=1e-3)
        sched = S.OneCycleLR(opt, max_lr=3e-3, total_steps=E * len(train_loader))
        hist = fit(model, train_loader, val_loader, opt, E, scheduler=sched,
                   early_stopping=EarlyStopping(patience=100))        # patience=100: just keep the best epoch
        torch.save({"state_dict": model.state_dict()}, CNN_PATH)
        return model, hist

    def get_trained_cnn():
        # Load the CNN trained in Chapter 7 if it was saved; otherwise train it now.
        model = SmallCNN().to(device)
        if CNN_PATH.exists():
            model.load_state_dict(torch.load(CNN_PATH, map_location=device, weights_only=True)["state_dict"])
            print(f"loaded the trained SmallCNN from {CNN_PATH}")
        else:
            print("no saved SmallCNN found: training one with the recipe of Section 7.7 ...")
            model, _ = train_small_cnn()
        return model.eval()
    print("Restored the tools defined in earlier chapters.")

## 12.1 The protocol
Go through these in order; each step is cheap and rules out a whole class of bugs.

| Step | What to do | What it catches |
| --- | --- | --- |
| 1 | **Smoke test** the whole pipeline with `FAST_DEV_RUN=True` | Crashes in rarely run code (evaluation, saving) before you spend GPU hours |
| 2 | **Look at a batch**: images, labels, value ranges, shapes, dtypes | Wrong labels, broken normalisation, misaligned data |
| 3 | **Check the initial loss**: ≈ ln C for classification | Bad initialisation, wrong loss setup |
| 4 | **Overfit one batch** to near-zero loss | Bugs in model, loss or loop (Notebook 1, §7.9) |
| 5 | **Check activations and gradient flow** per layer | Vanishing/exploding signals, dead units, frozen or disconnected layers |
| 6 | **Scale up** and watch the curves, the learning rate and the gradient norm | Overfitting, learning-rate problems, instabilities |

## 12.2 Watching activations with forward hooks
A **forward hook** is a function that PyTorch calls every time a module finishes its forward pass, with the module, its inputs and its output. Hooks let you inspect any layer of any model **without editing its code**, which is invaluable with models written by other people.

```python
handle = module.register_forward_hook(lambda module, inputs, output: ...)
...
handle.remove()            # always remove hooks you no longer need
```

Below, a hook records statistics of every ReLU's output in a 10-layer MLP. We compare PyTorch's default initialisation, a too-small one, and He initialisation.

In [ ]:
def deep_mlp(depth=10, width=256, init=None):
    layers = [nn.Flatten(), nn.Linear(784, width), nn.ReLU()]
    for _ in range(depth - 1):
        layers += [nn.Linear(width, width), nn.ReLU()]
    model = nn.Sequential(*layers, nn.Linear(width, 10))
    if init is not None:
        for m in model.modules():
            if isinstance(m, nn.Linear):
                init(m.weight); nn.init.zeros_(m.bias)
    return model

def activation_report(model, x):
    stats, handles = [], []
    for name, m in model.named_modules():
        if isinstance(m, nn.ReLU):
            def hook(mod, inp, out, name=name):          # name=name freezes the loop variable
                stats.append((name, out.std().item(), (out == 0).float().mean().item()))
            handles.append(m.register_forward_hook(hook))
    with torch.no_grad():
        model(x)
    for h in handles:
        h.remove()
    return stats

x_dbg, y_dbg = next(iter(train_loader))
inits = {"PyTorch default": None,
         "normal(std=0.01)": lambda w: nn.init.normal_(w, std=0.01),
         "He (kaiming_normal_)": lambda w: nn.init.kaiming_normal_(w, nonlinearity="relu")}
plt.figure(figsize=(6.5, 3.5))
for label, init in inits.items():
    set_seed(0)
    stats = activation_report(deep_mlp(init=init), x_dbg)
    plt.semilogy([s[1] for s in stats], "o-", label=label)
plt.xlabel("ReLU layer"); plt.ylabel("std of activations (log)"); plt.legend(); plt.title("Signal propagation through 10 layers")
plt.show()

With `std=0.01` the signal shrinks by a constant factor at every layer and is essentially zero by layer 10: the network's output no longer depends on its input, and the gradients vanish too. PyTorch's default also shrinks the signal, more slowly, and then levels off at a small value: its biases are initialised randomly too, and from some depth on the activations are dominated by the biases rather than by the input. He initialisation, designed for ReLU, keeps the scale roughly constant. This is exactly the kind of problem that hooks reveal in seconds and that would otherwise look like "the model does not learn".

A related check is the fraction of exact zeros after each ReLU (the third value in each `stats` entry). If a layer is almost entirely zero for every input, those units are **dead**: they never activate and get no gradient. High learning rates and large negative biases cause this.

## 12.3 Gradient flow
After `loss.backward()`, every parameter's `.grad` is available. The norm per layer shows whether gradients reach the early layers, and whether some layer gets none at all (a frozen or disconnected parameter).

In [ ]:
def grad_norms(model):
    return {name: p.grad.norm().item() for name, p in model.named_parameters()
            if p.grad is not None and name.endswith("weight")}

plt.figure(figsize=(6.5, 3.5))
for label, init in inits.items():
    set_seed(0)
    m = deep_mlp(init=init)
    F.cross_entropy(m(x_dbg), y_dbg).backward()
    g = grad_norms(m)
    plt.semilogy(list(g.values()), "o-", label=label)
plt.xlabel("Linear layer (input -> output)"); plt.ylabel("gradient norm (log)"); plt.legend()
plt.title("Gradient flow at initialisation"); plt.show()

With the bad initialisation, gradients at the first layers are many orders of magnitude smaller than at the last one, so the early layers effectively do not train. In a healthy network the per-layer gradient norms are within an order of magnitude or two of each other.

Two more tools from the same family:
- **Tensor hooks:** `tensor.register_hook(fn)` calls `fn(grad)` when the gradient of that tensor is computed. It is useful for intermediate activations, which do not keep `.grad`.
- **Logging:** in real projects, log the global gradient norm (Chapter 4's `fit` already does) and per-layer norms every few hundred steps to a dashboard such as TensorBoard or Weights & Biases.

## 12.4 Hunting NaNs
Once a single `NaN` appears, it spreads to every weight within one step. Common causes:

| Cause | Example |
| --- | --- |
| Learning rate too high; exploding loss | the loss rises steeply, then becomes `inf`, then `nan` |
| `log(0)` or division by zero in hand-written code | `torch.log(probs)` where a probability underflowed to 0 |
| Square root at 0 in the backward pass | `x.sqrt()` has an infinite derivative at 0, e.g. in a hand-written norm or distance |
| `float16` overflow without a scaler | activations above 65,504 |
| Bad data | a `NaN` in the input file |

`torch.autograd.detect_anomaly()` makes the backward pass check every operation and report **which forward operation** produced the first `NaN` in the gradient. It is slow, so use it only while debugging.

In [ ]:
import warnings

def pairwise_dist(a, b):
    return ((a - b) ** 2).sum(dim=1).sqrt()                # hand-written Euclidean distance

a = torch.randn(4, 3, requires_grad=True)
b = a.detach().clone() + 1.0                               # every example at distance > 0 ...
b[0] = a[0].detach()                                       # ... except example 0: distance exactly 0
loss = pairwise_dist(a, b).mean()
loss.backward()
print("loss is finite:", torch.isfinite(loss).item(), "| but a.grad has NaN:", torch.isnan(a.grad).any().item())

a.grad = None
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    with torch.autograd.detect_anomaly():
        try:
            pairwise_dist(a, b).mean().backward()
        except RuntimeError as e:
            print("detect_anomaly:", str(e)[:120])

The loss was perfectly finite, yet the gradient contained `NaN`. Follow the backward pass: $\frac{d}{dx}\sqrt{x} = \frac{1}{2\sqrt{x}}$ is **infinite** at 0, so `SqrtBackward0` outputs `inf` (not yet `NaN`). The next node backwards is the square, whose backward multiplies by $2(a-b) = 0$, and $\infty \cdot 0 = \text{NaN}$. Anomaly detection therefore reports `PowBackward0`: the **first** backward operation whose output is `NaN`, together with a traceback pointing at the forward line that created it. The lesson: the reported operation is where the `NaN` first appears; the root cause is often the operation just before it in the forward pass (here, the `sqrt`). The fix is `(... + 1e-12).sqrt()` or the built-in `torch.cdist` / `F.pairwise_distance`, which handle this case. In training loops, a cheap guard is to check `torch.isfinite(loss)` and stop (or skip the batch) with a clear message.

## 12.5 A field guide to symptoms

| Symptom | Likely causes | First check |
| --- | --- | --- |
| Loss exactly flat from the start | `optimizer.step()` missing or before `backward()`; parameters not registered; lr = 0; everything frozen | `count_params(model)`; print the lr; do weights change after one step? |
| Initial loss far from ln C | Wrong output size, unnormalised inputs, bad init, softmax applied twice | Print logits statistics on one batch |
| Accuracy stuck at chance, loss slowly falls | Labels misaligned with inputs; wrong label mapping | Plot images with their labels |
| Cannot overfit one batch | Model bug; loss contract violated (shapes, softmax); too much regularisation | Overfit test without dropout and weight decay |
| Loss spikes, then `nan` | Learning rate too high; no warm-up; `fp16` overflow | Lower the lr, add clipping and warm-up; log the gradient norm |
| Training loss falls, validation flat from the start | Train and validation preprocessed differently; data leakage in the other direction; validation labels wrong | Evaluate the model on a slice of the **training** set with the validation pipeline |
| Validation much better than test | Model selection overfit to validation; distribution shift | Use a fresh split; compare data statistics |
| Validation result changes between identical evaluations | `model.eval()` missing | Evaluate twice and compare |
| Slower each epoch, memory grows | Graph kept alive (losses accumulated without `.item()`), lists of GPU tensors | Check what you append to lists |

## 🧪 Exercise 12.1: debug a broken training script  `★★★ · ~30 min`
The function below trains an MLP on a small, clean subset of Fashion-MNIST. It runs without any error message, yet its validation accuracy is terrible. It contains **four** independent bugs, each of them a mistake that real engineers make. Use the protocol and the field guide to find them. Then write `fixed_training()`, which should reach roughly 80–85% validation accuracy (the subset has only 4,096 training images).

In [ ]:
def buggy_training(epochs=5, seed=0):
    set_seed(seed)
    X = small_ds.tensors[0]                                   # (4096, 1, 28, 28), normalised
    Y = small_ds.tensors[1]
    perm = torch.randperm(len(X))
    train_ds = TensorDataset(X[perm], Y)                      # shuffle the training set
    loader = DataLoader(train_ds, batch_size=128, shuffle=True)

    model = nn.Sequential(nn.Flatten(), nn.Linear(784, 256), nn.ReLU(), nn.Dropout(0.5),
                          nn.Linear(256, 10), nn.Softmax(dim=1)).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        model.train()
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            loss = loss_fn(model(x), y)
            loss.backward()
            opt.step()
        with torch.no_grad():                                 # validation
            logits = model(small_val.tensors[0].to(device))
            acc = (logits.argmax(dim=1).cpu() == small_val.tensors[1]).float().mean().item()
        print(f"epoch {epoch + 1}: val acc {acc:.3f}")

buggy_training()

In [ ]:
# ✍️ Your attempt here.
# Hint: Bugs hide in the data, the model, the loop and the evaluation: one in each.

In [ ]:
# @title ✅ Solution (double-click to reveal) { display-mode: "form" }
# Solutions only execute when RUN_SOLUTIONS = True (setup cell), so "Run all" stays fast.
if RUN_SOLUTIONS:
    def fixed_training(epochs=5, seed=0):
        set_seed(seed)
        X, Y = small_ds.tensors
        # Bug 1: X[perm] with an unshuffled Y destroys the pairing of images and labels -> chance accuracy.
        #        The DataLoader already shuffles (keeping pairs intact), so no manual shuffle is needed.
        loader = DataLoader(TensorDataset(X, Y), batch_size=128, shuffle=True)

        # Bug 2: nn.Softmax before CrossEntropyLoss (double softmax, Notebook 1 §5.3): the model must output logits.
        model = nn.Sequential(nn.Flatten(), nn.Linear(784, 256), nn.ReLU(), nn.Dropout(0.5),
                              nn.Linear(256, 10)).to(device)
        opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
        loss_fn = nn.CrossEntropyLoss()

        for epoch in range(epochs):
            model.train()
            for x, y in loader:
                x, y = x.to(device), y.to(device)
                loss = loss_fn(model(x), y)
                opt.zero_grad()                  # Bug 3: without it, gradients accumulate over all past steps
                loss.backward()
                opt.step()
            model.eval()                         # Bug 4: without eval(), Dropout(0.5) is active during validation
            with torch.no_grad():
                logits = model(small_val.tensors[0].to(device))
                acc = (logits.argmax(dim=1).cpu() == small_val.tensors[1]).float().mean().item()
            print(f"epoch {epoch + 1}: val acc {acc:.3f}")

    fixed_training()
    # How the protocol finds them: plotting a batch with its labels (step 2) reveals bug 1; the
    # initial-loss and overfit-one-batch checks (steps 3-4) expose bug 2 (the loss cannot go below
    # about 1.46); printing whether the weights change sensibly, or the gradient norm growing every
    # step, reveals bug 3; evaluating twice and getting different numbers reveals bug 4.
else:
    print("Solution not executed (RUN_SOLUTIONS = False).")

### 📌 Key takeaways
- Follow the protocol: smoke test, look at data, initial loss, overfit one batch, activations and gradients, then scale.
- Forward hooks inspect any layer of any model without editing it; always remove them.
- Per-layer gradient norms reveal vanishing gradients and disconnected parameters.
- `detect_anomaly()` finds the operation that created the first `NaN` gradient.

### ✔️ Check yourself

Commit to an answer (say it out loud or write it down) *before* you expand it.

**1.** Your model's loss is finite, but after one step all the weights are NaN. Where would you look?

<details>
<summary>Answer</summary>

At the gradients: something in the backward pass produced inf/NaN even though the forward value was finite (e.g. sqrt or log at 0, division by a tiny number). Run one step under `torch.autograd.detect_anomaly()` and inspect the reported operation and the one just before it in the forward pass.

</details>

**2.** Why does a forward hook take the arguments `(module, inputs, output)`, and what happens if you forget `handle.remove()`?

<details>
<summary>Answer</summary>

PyTorch calls it after the module's forward with the module, a tuple of its inputs and its output. Without `remove()`, the hook keeps running on every forward pass, slowing things down and, if it stores tensors, leaking memory.

</details>

**3.** A 10-layer network's first-layer gradient norm is 1e-9 while the last layer's is 1e-1. Name a cause and two fixes.

<details>
<summary>Answer</summary>

Vanishing gradients, e.g. from a too-small initialisation or saturating activations. Fixes: He initialisation for ReLU networks, normalisation layers (BatchNorm/LayerNorm), residual connections.

</details>

---
# Chapter 13. Production essentials

🎯 **In this chapter you will learn to:** prepare a model for inference without the classic mistakes; package preprocessing inside the model; export it with `torch.export` and to ONNX so it runs without your Python code; measure latency; and turn notebook code into a small tested project with a command-line training script.

A model in a notebook is an experiment. A model that someone else can run, on another machine, months later, is a product. Most of the work in between is about removing the hidden assumptions that live in your notebook's state.

🔁 **Starting a fresh session here?** Run the two Chapter 0 cells, then the hidden restore-point cell below. It re-creates the tools from earlier chapters that Chapter 13 uses. In a normal top-to-bottom run it does nothing.

In [ ]:
# @title 🔁 Restore point for Chapter 13 (needed only if you start a fresh session here) { display-mode: "form" }
if "device" not in globals():
    raise RuntimeError("Run the Chapter 0 setup cells first, then this cell.")
if "get_trained_cnn" not in globals():
    from torch.optim import lr_scheduler as S

    def param_groups_weight_decay(model, weight_decay):
        decay, no_decay = [], []
        for name, p in model.named_parameters():
            if not p.requires_grad:
                continue                                     # frozen parameters: leave them out
            (no_decay if p.ndim <= 1 else decay).append(p)   # biases and norm weights are 1-D
        return [{"params": decay, "weight_decay": weight_decay},
                {"params": no_decay, "weight_decay": 0.0}]

    class EarlyStopping:
        # Stop when `metric` has not improved by more than `min_delta` for `patience` epochs.
        # mode="min" for losses, mode="max" for accuracies. Keeps a copy of the best weights.
        def __init__(self, patience=5, min_delta=0.0, mode="min"):
            self.patience, self.min_delta, self.mode = patience, min_delta, mode
            self.best = math.inf if mode == "min" else -math.inf
            self.best_state, self.best_epoch, self.bad_epochs, self.epoch = None, 0, 0, 0

        def improved(self, value):
            if self.mode == "min":
                return value < self.best - self.min_delta
            return value > self.best + self.min_delta

        def step(self, value, model):
            # Call once per epoch. Returns True when training should stop.
            self.epoch += 1
            if self.improved(value):
                self.best, self.best_epoch, self.bad_epochs = value, self.epoch, 0
                self.best_state = copy.deepcopy(model.state_dict())
            else:
                self.bad_epochs += 1
            return self.bad_epochs >= self.patience

        def restore(self, model):
            model.load_state_dict(self.best_state)

    import contextlib

    def train_one_epoch(model, loader, loss_fn, optimizer, scheduler=None, max_grad_norm=None,
                        averaged_model=None, amp_dtype=None, scaler=None):
        # amp_dtype / scaler: mixed precision, explained in Chapter 11. Leave them as None until then.
        model.train()
        total_loss = torch.zeros((), device=device)                  # running sums stay ON the device:
        correct = torch.zeros((), device=device, dtype=torch.long)   # no CPU-GPU synchronisation per step
        n, grad_norms = 0, []
        for x, y in loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            amp = torch.autocast(device_type=device.type, dtype=amp_dtype) if amp_dtype else contextlib.nullcontext()
            with amp:
                logits = model(x)
                loss = loss_fn(logits, y)
            optimizer.zero_grad(set_to_none=True)
            if scaler is not None:                                   # float16 mixed precision (Chapter 11)
                scaler.scale(loss).backward()
                if max_grad_norm is not None:
                    scaler.unscale_(optimizer)                       # clip the REAL gradients, not the scaled ones
            else:
                loss.backward()
            if max_grad_norm is not None:
                grad_norms.append(nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm))
            if scaler is not None:
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()
            if scheduler is not None:                                # per-batch schedules (OneCycleLR, warm-up)
                scheduler.step()
            if averaged_model is not None:                           # weight averaging (Chapter 6)
                averaged_model.update_parameters(model)
            total_loss += loss.detach().float() * len(y)
            correct += (logits.argmax(dim=1) == y).sum()
            n += len(y)
        mean_norm = torch.stack(grad_norms).mean().item() if grad_norms else float("nan")
        return total_loss.item() / n, correct.item() / n, mean_norm   # one synchronisation per epoch


    def fit(model, train_loader, val_loader, optimizer, epochs, loss_fn=nn.CrossEntropyLoss(),
            scheduler=None, scheduler_interval="batch", early_stopping=None, eval_model=None, verbose=True, **train_kw):
        # scheduler_interval: "batch", "epoch" or "plateau" (ReduceLROnPlateau).
        # eval_model: evaluate (and select) a different model than the one trained, e.g. an EMA copy (Chapter 6).
        history = {k: [] for k in ["train_loss", "train_acc", "val_loss", "val_acc", "lr", "grad_norm", "time"]}
        eval_model = eval_model if eval_model is not None else model
        for epoch in range(1, epochs + 1):
            t0 = time.perf_counter()
            lr = optimizer.param_groups[0]["lr"]
            tr_loss, tr_acc, gnorm = train_one_epoch(model, train_loader, loss_fn, optimizer,
                                                     scheduler if scheduler_interval == "batch" else None, **train_kw)
            va_loss, va_acc = evaluate(eval_model, val_loader, loss_fn)
            if scheduler is not None and scheduler_interval == "epoch":
                scheduler.step()
            elif scheduler is not None and scheduler_interval == "plateau":
                scheduler.step(va_loss)
            elapsed = time.perf_counter() - t0
            for k, v in zip(history, (tr_loss, tr_acc, va_loss, va_acc, lr, gnorm, elapsed)):
                history[k].append(v)
            if verbose:
                print(f"epoch {epoch:2d} | lr {lr:.2e} | train {tr_loss:.4f} / {tr_acc:.4f} | "
                      f"val {va_loss:.4f} / {va_acc:.4f} | {elapsed:4.1f}s")
            if early_stopping is not None and early_stopping.step(va_loss, eval_model):
                print(f"early stopping: no improvement for {early_stopping.patience} epochs")
                break
        if early_stopping is not None:                       # select the epoch with the best validation LOSS...
            early_stopping.restore(eval_model)
            sel_epoch = early_stopping.best_epoch
        else:
            sel_epoch = len(history["val_loss"])            # ...or keep the last epoch
        sel_loss, sel_acc = history["val_loss"][sel_epoch - 1], history["val_acc"][sel_epoch - 1]
        history["selected"] = {"epoch": sel_epoch, "val_loss": sel_loss, "val_acc": sel_acc}
        if verbose:
            print(f"selected model: epoch {sel_epoch}, val loss {sel_loss:.4f}, val acc {sel_acc:.4f}")
        return history


    def plot_history(histories, keys=("val_loss", "val_acc", "lr")):
        fig, axes = plt.subplots(1, len(keys), figsize=(5 * len(keys), 3.2))
        for ax, key in zip(axes, keys):
            for name, h in histories.items():
                ax.plot(range(1, len(h[key]) + 1), h[key], "o-", ms=3, label=name)
            ax.set_title(key); ax.set_xlabel("epoch")
        axes[0].legend(fontsize=8); plt.tight_layout(); plt.show()

    # Small in-memory subsets (normalised exactly like fm_transform) for quick experiments
    def _fm_subset(idx):
        x = ((raw_train.data[idx].float() / 255 - FM_MEAN) / FM_STD).unsqueeze(1)
        return TensorDataset(x, raw_train.targets[idx])

    small_ds = _fm_subset(FM_TRAIN_IDX[:4096])      # 4,096 training images
    small_val = _fm_subset(FM_VAL_IDX[:2000])       # 2,000 validation images

    def save_checkpoint(path, model, optimizer, scheduler=None, epoch=0, best_metric=None,
                        loader_generator=None, config=None):
        np_state = np.random.get_state()
        ckpt = {
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict() if scheduler is not None else None,
            "epoch": epoch,
            "best_metric": best_metric,
            "config": config or {},
            "rng": {
                "torch": torch.get_rng_state(),
                "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [],
                "python": random.getstate(),
                # NumPy's state contains an array; store it as a list so the safe loader accepts it
                "numpy": (np_state[0], np_state[1].tolist(), *np_state[2:]),
            },
            "loader_generator": loader_generator.get_state() if loader_generator is not None else None,
        }
        tmp = Path(str(path) + ".tmp")
        torch.save(ckpt, tmp)             # write to a temporary file first...
        os.replace(tmp, path)             # ...then rename: an interruption can never leave a half-written checkpoint


    def load_checkpoint(path, model, optimizer=None, scheduler=None, loader_generator=None):
        ckpt = torch.load(path, map_location=device, weights_only=True)   # safe loading: tensors and plain data only
        model.load_state_dict(ckpt["model"])
        if optimizer is not None:
            optimizer.load_state_dict(ckpt["optimizer"])
        if scheduler is not None and ckpt["scheduler"] is not None:
            scheduler.load_state_dict(ckpt["scheduler"])
        rng = ckpt["rng"]
        torch.set_rng_state(rng["torch"].cpu())
        if torch.cuda.is_available() and rng["cuda"]:
            torch.cuda.set_rng_state_all([s.cpu() for s in rng["cuda"]])
        random.setstate(rng["python"])
        name, keys, *rest = rng["numpy"]
        np.random.set_state((name, np.array(keys, dtype=np.uint32), *rest))
        if loader_generator is not None and ckpt["loader_generator"] is not None:
            loader_generator.set_state(ckpt["loader_generator"].cpu())
        return ckpt

    def conv_bn_relu(c_in, c_out):
        return nn.Sequential(nn.Conv2d(c_in, c_out, 3, padding=1, bias=False), nn.BatchNorm2d(c_out), nn.ReLU(inplace=True))


    class SmallCNN(nn.Module):
        def __init__(self, num_classes=10, width=32, p_drop=0.3):
            super().__init__()
            w = width
            self.features = nn.Sequential(
                conv_bn_relu(1, w), conv_bn_relu(w, w), nn.MaxPool2d(2),              # 28x28 -> 14x14
                conv_bn_relu(w, 2 * w), conv_bn_relu(2 * w, 2 * w), nn.MaxPool2d(2),  # 14x14 -> 7x7
                conv_bn_relu(2 * w, 4 * w),                                           # 7x7, 128 channels
                nn.AdaptiveAvgPool2d(1),                                              # -> 1x1
            )
            self.head = nn.Sequential(nn.Flatten(), nn.Dropout(p_drop), nn.Linear(4 * w, num_classes))

        def forward(self, x):
            return self.head(self.features(x))

    CNN_PATH = CKPT_DIR / "smallcnn_fashion.pt"

    def train_small_cnn(epochs=12, seed=0):
        # The recipe of Section 7.7: AdamW with weight-decay groups and a OneCycle schedule.
        set_seed(seed)
        model = SmallCNN().to(device)
        E = n_epochs(epochs)
        opt = torch.optim.AdamW(param_groups_weight_decay(model, 0.05), lr=1e-3)
        sched = S.OneCycleLR(opt, max_lr=3e-3, total_steps=E * len(train_loader))
        hist = fit(model, train_loader, val_loader, opt, E, scheduler=sched,
                   early_stopping=EarlyStopping(patience=100))        # patience=100: just keep the best epoch
        torch.save({"state_dict": model.state_dict()}, CNN_PATH)
        return model, hist

    def get_trained_cnn():
        # Load the CNN trained in Chapter 7 if it was saved; otherwise train it now.
        model = SmallCNN().to(device)
        if CNN_PATH.exists():
            model.load_state_dict(torch.load(CNN_PATH, map_location=device, weights_only=True)["state_dict"])
            print(f"loaded the trained SmallCNN from {CNN_PATH}")
        else:
            print("no saved SmallCNN found: training one with the recipe of Section 7.7 ...")
            model, _ = train_small_cnn()
        return model.eval()
    print("Restored the tools defined in earlier chapters.")

## 13.1 The inference checklist
| ✔ | Item | Why |
| --- | --- | --- |
| 1 | `model.eval()` | Dropout off, BatchNorm uses its running statistics |
| 2 | `torch.inference_mode()` (or `no_grad`) | No graph is recorded: faster and less memory |
| 3 | **Identical preprocessing** to training | Same resizing, scaling to [0, 1], normalisation statistics, channel order |
| 4 | A batch dimension | Models expect `(N, …)`; add it with `unsqueeze(0)` for single inputs |
| 5 | The right device and dtype | Inputs on the model's device; `float32` (or what the model was exported for) |
| 6 | Output post-processing | Softmax for probabilities; class-index → name mapping stored **with** the model |
| 7 | Weights + config + class names + preprocessing stored together | So the model can be rebuilt and used correctly |

## 13.2 Put the preprocessing inside the model
The most common deployment bug is preprocessing that differs between training and serving: someone forgets the normalisation, or uses different statistics. The robust fix is to make preprocessing **part of the model**, so that the deployed artifact takes raw input. Normalisation constants become **buffers** (Notebook 1, §4.5): saved with the weights, moved with `.to()`, never trained.

In [ ]:
class FashionClassifier(nn.Module):
    # Deployable model: takes raw uint8 images (N, 1, 28, 28) with values 0..255, returns class probabilities.
    def __init__(self, net, mean, std, class_names):
        super().__init__()
        self.net = net
        self.register_buffer("mean", torch.tensor(mean, dtype=torch.float32))
        self.register_buffer("std", torch.tensor(std, dtype=torch.float32))
        self.class_names = list(class_names)            # plain attribute: NOT in state_dict or exports (see 13.3)

    def forward(self, images):
        x = (images.float() / 255.0 - self.mean) / self.std     # exactly the training preprocessing
        return torch.softmax(self.net(x), dim=1)

cnn = get_trained_cnn()                                        # the network trained in Chapter 7
deploy = FashionClassifier(copy.deepcopy(cnn).cpu(), FM_MEAN, FM_STD, CLASSES).eval()

test_raw = datasets.FashionMNIST(DATA_DIR, train=False)
raw_batch = test_raw.data[:8].unsqueeze(1)                     # uint8, (8, 1, 28, 28): straight from the file
with torch.inference_mode():
    probs = deploy(raw_batch)
    reference = torch.softmax(cnn.cpu().eval()(torch.stack([fm_transform(test_raw[i][0]) for i in range(8)])), dim=1)
cnn.to(device)
print("same output as the training pipeline:", torch.allclose(probs, reference, atol=1e-5))
print("predictions:", [deploy.class_names[i] for i in probs.argmax(dim=1)])

## 13.3 `torch.export`: a self-contained graph
Loading a `state_dict` requires the model's **Python class**. For deployment you often want an artifact that contains the computation itself. `torch.export.export(model, example_inputs)` traces the model into an **`ExportedProgram`**: a graph of PyTorch operations plus the weights, with no dependence on your source code. It can be saved to a `.pt2` file, loaded elsewhere, run, and compiled further (for example for mobile with ExecuTorch, or with AOTInductor).

By default the traced graph is specialised to the example's shapes. To accept any batch size, declare the batch dimension as **dynamic**.

⚠️ **Metadata is not saved automatically.** `class_names` is a plain Python attribute: it is not a tensor, so it is in neither the `state_dict` nor the exported graph. A file that outputs "class 7" without saying what class 7 is fails item 7 of the checklist. Store such metadata explicitly: in the dictionary you `torch.save` next to the `state_dict`, and as `extra_files` inside the `.pt2` file:

In [ ]:
example = test_raw.data[:4].unsqueeze(1)                       # uint8 (4, 1, 28, 28)
batch = torch.export.Dim("batch", min=1, max=1024)
exported = torch.export.export(deploy, (example,), dynamic_shapes={"images": {0: batch}})
import json
meta = {"classes": CLASSES, "input": "uint8 (N, 1, 28, 28), values 0-255", "output": "class probabilities (N, 10)"}
torch.export.save(exported, DATA_DIR / "fashion_classifier.pt2", extra_files={"meta.json": json.dumps(meta)})
torch.save({"state_dict": deploy.state_dict(), "meta": meta}, DATA_DIR / "fashion_classifier_weights.pt")

extra = {"meta.json": ""}                                     # names of the extra files to read back
program = torch.export.load(DATA_DIR / "fashion_classifier.pt2", extra_files=extra)
loaded = program.module()                                     # no FashionClassifier class needed
loaded_classes = json.loads(extra["meta.json"])["classes"]
print("class names read back from the .pt2 file:", loaded_classes[:3], "...")
for n in (1, 7):
    xi = test_raw.data[:n].unsqueeze(1)
    with torch.inference_mode():
        print(f"batch {n}: matches the eager model:", torch.allclose(loaded(xi), deploy(xi), atol=1e-6),
              "| first prediction:", loaded_classes[loaded(xi).argmax(dim=1)[0]])
print("\nfirst lines of the exported graph:")
print("\n".join(str(exported.graph_module.code).strip().splitlines()[:6]))

The graph shows that BatchNorm's running statistics and our `mean`/`std` buffers were captured as inputs of the program, just like the weights. `torch.export` is strict about what it can capture: data-dependent Python control flow, for example, must be rewritten (with `torch.cond`) or removed. When export fails, its error messages name the offending line.

🏭 You will still meet **TorchScript** (`torch.jit.script` / `torch.jit.trace`) in older code bases. It solved the same problem and is now in maintenance mode; new projects use `torch.export`.

## 13.4 ONNX: run without PyTorch
**ONNX** is an open format for neural-network graphs. An ONNX file runs in **ONNX Runtime** (C++, C#, Java, JavaScript, Python), in NVIDIA TensorRT, in browsers and on many edge devices, all without PyTorch installed. Since PyTorch 2.5, `torch.onnx.export(..., dynamo=True)` builds on `torch.export`.

In [ ]:
%pip install -q onnx onnxscript onnxruntime

In [ ]:
try:
    import onnxruntime as ort
    onnx_path = DATA_DIR / "fashion_classifier.onnx"
    torch.onnx.export(deploy, (example,), onnx_path, dynamo=True, dynamic_shapes={"images": {0: batch}})
    import onnx                                                     # add the metadata to the ONNX file
    onnx_model = onnx.load(str(onnx_path))
    onnx.helper.set_model_props(onnx_model, {k: json.dumps(v) for k, v in meta.items()})
    onnx.save(onnx_model, str(onnx_path))

    session = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
    onnx_classes = json.loads(session.get_modelmeta().custom_metadata_map["classes"])
    print("class names read back from the ONNX file:", onnx_classes[:3], "...")
    inp = session.get_inputs()[0]
    print("ONNX input:", inp.name, inp.shape, inp.type)
    for n in (1, 7):
        xi = test_raw.data[:n].unsqueeze(1)
        out = session.run(None, {inp.name: xi.numpy()})[0]            # NumPy in, NumPy out: no PyTorch involved
        with torch.inference_mode():
            print(f"batch {n}: ONNX Runtime matches PyTorch:", np.allclose(out, deploy(xi).numpy(), atol=1e-5))
    HAVE_ORT = True
except ImportError as e:
    HAVE_ORT = False
    print("ONNX packages missing; run the %pip cell above.", e)

## 13.5 Latency: how fast is one prediction?
Throughput (examples per second, with large batches) and **latency** (time for one request, often batch size 1) are different goals. Interactive services care about latency, and especially about the slow tail (the 95th or 99th percentile), not just the average. Measure with many repetitions, after a warm-up, and report the median and a high percentile.

In [ ]:
def latency_ms(fn, x, reps=200):
    for _ in range(10):
        fn(x)                                                   # warm-up
    times = []
    for _ in range(reps):
        t0 = time.perf_counter(); fn(x); times.append((time.perf_counter() - t0) * 1e3)
    t = torch.tensor(times)
    return t.median().item(), t.quantile(0.95).item()

one = test_raw.data[:1].unsqueeze(1)
with torch.inference_mode():
    rows = {"PyTorch eager (CPU)": latency_ms(deploy, one), "torch.export (CPU)": latency_ms(loaded, one)}
if HAVE_ORT:
    rows["ONNX Runtime (CPU)"] = latency_ms(lambda x: session.run(None, {inp.name: x.numpy()}), one)
for name, (med, p95) in rows.items():
    print(f"{name:22s} median {med:6.3f} ms | p95 {p95:6.3f} ms")

For small models, ONNX Runtime is often noticeably faster than eager PyTorch at batch size 1, because it has less per-operation overhead. The ranking depends on the model, the hardware and the thread settings, which is why you measure.

## 13.6 From notebook to project
Notebooks are excellent for exploring and terrible for running the same experiment reliably 50 times. The usual next step is a small project:

```
fmnist_project/
├── fmnist/
│   ├── __init__.py
│   └── model.py          # model definitions (importable, testable)
├── tests/
│   └── test_model.py     # fast unit tests, run on every change (e.g. in CI)
└── train.py              # command-line entry point: config in, checkpoint and logs out
```

Principles: **configuration in one place** (a dataclass, filled from command-line arguments or a YAML file; tools like Hydra do this at scale), **every hyper-parameter saved with the checkpoint**, **machine-readable logs** (here one JSON line per epoch; at scale, TensorBoard or Weights & Biases), and **tests** for the things that break silently. The next cell writes such a project to disk and runs its tests and a smoke-test training run. Open the files in Colab's file browser to read them.

In [ ]:
import subprocess, textwrap
proj = Path("fmnist_project")
for folder in (proj / "fmnist", proj / "tests"):
    folder.mkdir(parents=True, exist_ok=True)
(proj / "fmnist" / "__init__.py").write_text("")
(proj / "pytest.ini").write_text("[pytest]\npythonpath = .\n")

def write_file(relative_path, text):
    (proj / relative_path).write_text(text.lstrip("\n"))
    print(f"wrote {proj / relative_path} ({len(text.splitlines())} lines)")

In [ ]:
write_file('fmnist/model.py', r'''
"""Model definitions for the Fashion-MNIST project: the same code as Section 7.7 of the notebook."""
import torch
import torch.nn as nn


def conv_bn_relu(c_in, c_out):
    return nn.Sequential(nn.Conv2d(c_in, c_out, 3, padding=1, bias=False), nn.BatchNorm2d(c_out), nn.ReLU(inplace=True))


class SmallCNN(nn.Module):
    def __init__(self, num_classes=10, width=32, p_drop=0.3):
        super().__init__()
        w = width
        self.features = nn.Sequential(
            conv_bn_relu(1, w), conv_bn_relu(w, w), nn.MaxPool2d(2),              # 28x28 -> 14x14
            conv_bn_relu(w, 2 * w), conv_bn_relu(2 * w, 2 * w), nn.MaxPool2d(2),  # 14x14 -> 7x7
            conv_bn_relu(2 * w, 4 * w),                                           # 7x7, 128 channels
            nn.AdaptiveAvgPool2d(1),                                              # -> 1x1
        )
        self.head = nn.Sequential(nn.Flatten(), nn.Dropout(p_drop), nn.Linear(4 * w, num_classes))

    def forward(self, x):
        return self.head(self.features(x))
''')

In [ ]:
write_file('train.py', r'''
"""Train SmallCNN on Fashion-MNIST.  Usage: python train.py --epochs 12 --lr 3e-3 --out runs/cnn"""
import argparse, json, random
from dataclasses import dataclass, asdict
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets
from torchvision.transforms import v2 as T

from fmnist.model import SmallCNN


@dataclass
class Config:
    epochs: int = 12
    lr: float = 3e-3
    batch_size: int = 128
    weight_decay: float = 0.05
    seed: int = 0
    data_dir: str = "./data"
    out: str = "runs/cnn"
    fast_dev_run: bool = False


def parse_args() -> Config:
    p = argparse.ArgumentParser(description=__doc__)
    for name, default in asdict(Config()).items():
        if isinstance(default, bool):
            p.add_argument(f"--{name.replace('_', '-')}", action="store_true")
        else:
            p.add_argument(f"--{name.replace('_', '-')}", type=type(default), default=default)
    return Config(**vars(p.parse_args()))


def get_loaders(cfg: Config):
    raw = datasets.FashionMNIST(cfg.data_dir, train=True, download=True)
    perm = torch.randperm(len(raw), generator=torch.Generator().manual_seed(0))
    val_idx, train_idx = perm[:5000], perm[5000:]
    px = raw.data[train_idx].float() / 255
    mean, std = px.mean().item(), px.std().item()
    tf = T.Compose([T.ToImage(), T.ToDtype(torch.float32, scale=True), T.Normalize([mean], [std])])
    full = datasets.FashionMNIST(cfg.data_dir, train=True, transform=tf)
    n = 512 if cfg.fast_dev_run else None
    tr, va = Subset(full, train_idx[:n].tolist()), Subset(full, val_idx[:n].tolist())
    return (DataLoader(tr, cfg.batch_size, shuffle=True, drop_last=True, num_workers=2),
            DataLoader(va, 512, num_workers=2), {"mean": mean, "std": std})


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    correct = n = 0
    for x, y in loader:
        correct += (model(x.to(device)).argmax(1).cpu() == y).sum().item()
        n += len(y)
    return correct / n


def main():
    cfg = parse_args()
    if cfg.fast_dev_run:
        cfg.epochs = 1
    random.seed(cfg.seed); np.random.seed(cfg.seed); torch.manual_seed(cfg.seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    out = Path(cfg.out); out.mkdir(parents=True, exist_ok=True)

    train_loader, val_loader, norm = get_loaders(cfg)
    model = SmallCNN().to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)   # OneCycleLR sets the actual lr
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=cfg.lr, total_steps=cfg.epochs * len(train_loader))

    best = -1.0
    for epoch in range(1, cfg.epochs + 1):
        model.train()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            loss = F.cross_entropy(model(x), y)
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
        acc = evaluate(model, val_loader, device)
        print(json.dumps({"epoch": epoch, "val_acc": round(acc, 4), "lr": sched.get_last_lr()[0]}))
        if acc > best:
            best = acc
            torch.save({"state_dict": model.state_dict(), "config": asdict(cfg), "normalization": norm},
                       out / "best.pt")
    print(f"best val acc {best:.4f}; saved {out / 'best.pt'}")


if __name__ == "__main__":
    main()
''')

In [ ]:
write_file('tests/test_model.py', r'''
"""Fast unit tests for the model. Run with:  python -m pytest -q tests"""
import torch
import torch.nn.functional as F

from fmnist.model import SmallCNN


def test_output_shape():
    model = SmallCNN(num_classes=10)
    assert model(torch.randn(3, 1, 28, 28)).shape == (3, 10)


def test_works_for_other_input_sizes():
    # global average pooling makes the network independent of the input resolution
    assert SmallCNN()(torch.randn(2, 1, 40, 40)).shape == (2, 10)


def test_can_overfit_a_tiny_batch():
    torch.manual_seed(0)
    model = SmallCNN(p_drop=0.0)
    x, y = torch.randn(16, 1, 28, 28), torch.arange(16) % 10
    opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
    for _ in range(60):
        loss = F.cross_entropy(model(x), y)
        opt.zero_grad(); loss.backward(); opt.step()
    assert loss.item() < 0.1, f"could not overfit 16 examples (loss {loss.item():.3f})"


def test_eval_mode_is_deterministic_and_batch_independent():
    model = SmallCNN().eval()
    x = torch.randn(8, 1, 28, 28)
    with torch.no_grad():
        full = model(x)
        assert torch.allclose(full, model(x))                       # dropout off
        assert torch.allclose(full[:3], model(x[:3]), atol=1e-5)    # BatchNorm uses running stats
''')

In [ ]:
def run(cmd):
    r = subprocess.run(cmd, cwd=proj, capture_output=True, text=True)
    print("$", " ".join(cmd)); print(textwrap.indent((r.stdout + r.stderr).strip()[-1500:], "  "))
    return r.returncode

run([sys.executable, "-m", "pytest", "-q", "tests"])
run([sys.executable, "train.py", "--fast-dev-run", "--data-dir", str(DATA_DIR.resolve()), "--out", "runs/smoke"])

# One definition, used everywhere: the project's class loads the weights trained in the notebook
sys.path.insert(0, str(proj.resolve()))
from fmnist.model import SmallCNN as ProjectSmallCNN
ProjectSmallCNN().load_state_dict(cnn.state_dict())
print("the notebook's trained weights load into the project's SmallCNN: same architecture")

The four cells above write the project's files, so the notebook stays self-contained; in a real project you would edit the files directly. The model file contains **exactly** the code of §7.7, and the last check proves it: the weights trained in the notebook load into the project's class. In a real project the notebook would `import` the model from the package instead of defining it again, so that there is a single definition to maintain.

Two of the tests are worth copying into every project you build: **"can it overfit a tiny batch?"** (catches most model and loss bugs in seconds) and **"is evaluation deterministic and independent of batch composition?"** (catches a forgotten `eval()` and BatchNorm problems).

🏭 **Frameworks.** Libraries such as **PyTorch Lightning** and Hugging Face **Accelerate** package the training loop you wrote (device placement, AMP, checkpointing, logging, multi-GPU) behind a standard interface. Much research code uses them. Because you have written every piece yourself, you can read such code and know what happens underneath.

## 13.7 Where production goes from here
- **Serving:** wrap the model in a web service (e.g. FastAPI) or use an inference server such as NVIDIA Triton; batch incoming requests to use the hardware well.
- **Optimisation:** quantisation (8-bit or lower weights), pruning, distillation into a smaller model, TensorRT or ONNX Runtime optimisations.
- **Monitoring:** log inputs and predictions; watch for **data drift** (inputs that no longer look like the training data) and falling accuracy; retrain on fresh data.
- **Documentation:** a model card stating the training data, intended use, evaluation results and known limitations.

## 🧪 Exercises

**Exercise 13.1: a reusable deployment test.** `★★☆ · ~15 min`
Write `is_batch_independent(model, x, atol=1e-5)` that returns `True` if, in the model's **current** mode and with gradients disabled, the predictions for `x[:2]` computed alone equal the first two rows of the predictions for the whole batch `x`. Then check that it returns `True` for `cnn.eval()` and `False` for `cnn.train()` (BatchNorm in training mode uses batch statistics). Restore `cnn.eval()` afterwards.

In [ ]:
# ✍️ Your attempt here.

In [ ]:
# 🔎 Self-check for Exercise 13.1
def _test(fn):
    m = SmallCNN(p_drop=0.0).to(device)
    x = torch.randn(16, 1, 28, 28, device=device) * 3
    m.eval()
    assert fn(m, x) is True or fn(m, x) == True, "should be True in eval mode"
    m.train()
    assert not fn(m, x), "should be False in train mode (BatchNorm uses batch statistics)"
    assert m.training, "the function must not change the model's mode"
self_check("is_batch_independent", _test)

In [ ]:
# @title ✅ Solution (double-click to reveal) { display-mode: "form" }
# Solutions only execute when RUN_SOLUTIONS = True (setup cell), so "Run all" stays fast.
if RUN_SOLUTIONS:
    def is_batch_independent(model, x, atol=1e-5):
        with torch.no_grad():
            full = model(x)
            part = model(x[:2])
        return bool(torch.allclose(full[:2], part, atol=atol))

    print("eval: ", is_batch_independent(cnn.eval(), next(iter(val_loader))[0].to(device)))
    print("train:", is_batch_independent(cnn.train(), next(iter(val_loader))[0].to(device)))
    cnn.eval()
else:
    print("Solution not executed (RUN_SOLUTIONS = False).")

**Exercise 13.2: what static shapes cost you.** `★☆☆ · ~10 min`
Export `deploy` again **without** `dynamic_shapes`, using the batch-of-4 example. Call the exported module with a batch of 4, then with a batch of 7. What happens, and why?

In [ ]:
# ✍️ Your attempt here.

In [ ]:
# @title ✅ Solution (double-click to reveal) { display-mode: "form" }
# Solutions only execute when RUN_SOLUTIONS = True (setup cell), so "Run all" stays fast.
if RUN_SOLUTIONS:
    static = torch.export.export(deploy, (example,)).module()
    print("batch 4:", static(test_raw.data[:4].unsqueeze(1)).shape)
    try:
        static(test_raw.data[:7].unsqueeze(1))
    except Exception as e:
        print("batch 7:", type(e).__name__, str(e)[:150])
    # Without dynamic_shapes the batch size 4 is baked into the graph as a constant, and the exported
    # program checks its inputs against those shapes. Declare every dimension that can vary.
else:
    print("Solution not executed (RUN_SOLUTIONS = False).")

### 📌 Key takeaways
- Inference = `eval()` + `inference_mode()` + identical preprocessing + a batch dimension.
- Put preprocessing inside the model as buffers, and store class names and config with the weights.
- `torch.export` gives a code-free graph (`.pt2`); ONNX runs in ONNX Runtime without PyTorch. Declare dynamic dimensions.
- Measure latency with warm-up, many repetitions, median and tail percentiles.
- Move stable code into importable modules with a CLI, a config and fast tests.

### ✔️ Check yourself

Commit to an answer (say it out loud or write it down) *before* you expand it.

**1.** Why is putting normalisation inside the model (as buffers) better than documenting the statistics?

<details>
<summary>Answer</summary>

The statistics then travel with the weights in every saved or exported artifact, and callers cannot apply different preprocessing by mistake. Documentation is easy to miss; a buffer is impossible to forget.

</details>

**2.** Your exported model fails on batch size 7 but works on batch size 4. What did you forget?

<details>
<summary>Answer</summary>

To declare the batch dimension as dynamic (`dynamic_shapes` with `torch.export.Dim`). Otherwise the example's shapes are fixed in the graph.

</details>

**3.** Which two unit tests catch the most silent ML bugs cheaply?

<details>
<summary>Answer</summary>

Overfitting a tiny batch (model, loss and training-loop bugs) and checking that evaluation is deterministic and batch-independent (forgotten `eval()`, BatchNorm issues).

</details>

---
# Chapter 14. ★ Optional deep dives

These sections are optional and independent of each other. Each needs only the Chapter 0 cells and the restore point below (which re-creates `fit`, `SmallCNN` and the other tools), so you can jump straight to any of them.

| Section | Topic | Time |
| --- | --- | --- |
| 14.1 | ★ Project: a ResNet on CIFAR-10 (residual connections, augmentation, AMP) | 45 min, about 10 min of GPU |
| 14.2 | ★ Extending autograd: custom `autograd.Function`, straight-through estimators | 20 min |
| 14.3 | ★ Are the probabilities trustworthy? Calibration and temperature scaling | 25 min |
| 14.4 | ★ Profiling with `torch.profiler` | 15 min |
| 14.5 | ★ Multiple GPUs: a preview of DDP | 15 min |
| 14.6 | ★ Interview questions and an error-message decoder | 30 min |

## 14.1 ★ Project: a ResNet on CIFAR-10
CIFAR-10 has 60,000 colour images (32×32) in 10 classes: airplane, automobile, bird, cat, deer, dog, frog, horse, ship and truck. It is much harder than Fashion-MNIST (colour, cluttered backgrounds, large variation within each class), and it is the standard benchmark for small CNNs.

### Why residual connections
Stacking more plain convolution layers eventually makes training **worse**, not better: the signal and the gradients have to pass through every layer, and they degrade. A **residual block** computes $y = x + f(x)$: the input skips around the layers and is added back. The block then only has to learn a *correction* $f(x)$, and the identity path gives gradients a direct route backwards, since $\partial y / \partial x = I + \partial f / \partial x$. Residual connections made networks with hundreds of layers trainable (ResNet, 2015), and they are in nearly every modern architecture, transformers included.

We build a compact "ResNet-9", a design commonly reported to reach the low-to-mid 90s on CIFAR-10 in a few minutes of GPU time (record what yours reaches).

🔁 **Starting a fresh session here?** Run the two Chapter 0 cells, then the hidden restore-point cell below. It re-creates the tools from earlier chapters that Chapter 14 uses. In a normal top-to-bottom run it does nothing.

In [ ]:
# @title 🔁 Restore point for Chapter 14 (needed only if you start a fresh session here) { display-mode: "form" }
if "device" not in globals():
    raise RuntimeError("Run the Chapter 0 setup cells first, then this cell.")
if "get_trained_cnn" not in globals():
    from torch.optim import lr_scheduler as S

    def param_groups_weight_decay(model, weight_decay):
        decay, no_decay = [], []
        for name, p in model.named_parameters():
            if not p.requires_grad:
                continue                                     # frozen parameters: leave them out
            (no_decay if p.ndim <= 1 else decay).append(p)   # biases and norm weights are 1-D
        return [{"params": decay, "weight_decay": weight_decay},
                {"params": no_decay, "weight_decay": 0.0}]

    class EarlyStopping:
        # Stop when `metric` has not improved by more than `min_delta` for `patience` epochs.
        # mode="min" for losses, mode="max" for accuracies. Keeps a copy of the best weights.
        def __init__(self, patience=5, min_delta=0.0, mode="min"):
            self.patience, self.min_delta, self.mode = patience, min_delta, mode
            self.best = math.inf if mode == "min" else -math.inf
            self.best_state, self.best_epoch, self.bad_epochs, self.epoch = None, 0, 0, 0

        def improved(self, value):
            if self.mode == "min":
                return value < self.best - self.min_delta
            return value > self.best + self.min_delta

        def step(self, value, model):
            # Call once per epoch. Returns True when training should stop.
            self.epoch += 1
            if self.improved(value):
                self.best, self.best_epoch, self.bad_epochs = value, self.epoch, 0
                self.best_state = copy.deepcopy(model.state_dict())
            else:
                self.bad_epochs += 1
            return self.bad_epochs >= self.patience

        def restore(self, model):
            model.load_state_dict(self.best_state)

    import contextlib

    def train_one_epoch(model, loader, loss_fn, optimizer, scheduler=None, max_grad_norm=None,
                        averaged_model=None, amp_dtype=None, scaler=None):
        # amp_dtype / scaler: mixed precision, explained in Chapter 11. Leave them as None until then.
        model.train()
        total_loss = torch.zeros((), device=device)                  # running sums stay ON the device:
        correct = torch.zeros((), device=device, dtype=torch.long)   # no CPU-GPU synchronisation per step
        n, grad_norms = 0, []
        for x, y in loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            amp = torch.autocast(device_type=device.type, dtype=amp_dtype) if amp_dtype else contextlib.nullcontext()
            with amp:
                logits = model(x)
                loss = loss_fn(logits, y)
            optimizer.zero_grad(set_to_none=True)
            if scaler is not None:                                   # float16 mixed precision (Chapter 11)
                scaler.scale(loss).backward()
                if max_grad_norm is not None:
                    scaler.unscale_(optimizer)                       # clip the REAL gradients, not the scaled ones
            else:
                loss.backward()
            if max_grad_norm is not None:
                grad_norms.append(nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm))
            if scaler is not None:
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()
            if scheduler is not None:                                # per-batch schedules (OneCycleLR, warm-up)
                scheduler.step()
            if averaged_model is not None:                           # weight averaging (Chapter 6)
                averaged_model.update_parameters(model)
            total_loss += loss.detach().float() * len(y)
            correct += (logits.argmax(dim=1) == y).sum()
            n += len(y)
        mean_norm = torch.stack(grad_norms).mean().item() if grad_norms else float("nan")
        return total_loss.item() / n, correct.item() / n, mean_norm   # one synchronisation per epoch


    def fit(model, train_loader, val_loader, optimizer, epochs, loss_fn=nn.CrossEntropyLoss(),
            scheduler=None, scheduler_interval="batch", early_stopping=None, eval_model=None, verbose=True, **train_kw):
        # scheduler_interval: "batch", "epoch" or "plateau" (ReduceLROnPlateau).
        # eval_model: evaluate (and select) a different model than the one trained, e.g. an EMA copy (Chapter 6).
        history = {k: [] for k in ["train_loss", "train_acc", "val_loss", "val_acc", "lr", "grad_norm", "time"]}
        eval_model = eval_model if eval_model is not None else model
        for epoch in range(1, epochs + 1):
            t0 = time.perf_counter()
            lr = optimizer.param_groups[0]["lr"]
            tr_loss, tr_acc, gnorm = train_one_epoch(model, train_loader, loss_fn, optimizer,
                                                     scheduler if scheduler_interval == "batch" else None, **train_kw)
            va_loss, va_acc = evaluate(eval_model, val_loader, loss_fn)
            if scheduler is not None and scheduler_interval == "epoch":
                scheduler.step()
            elif scheduler is not None and scheduler_interval == "plateau":
                scheduler.step(va_loss)
            elapsed = time.perf_counter() - t0
            for k, v in zip(history, (tr_loss, tr_acc, va_loss, va_acc, lr, gnorm, elapsed)):
                history[k].append(v)
            if verbose:
                print(f"epoch {epoch:2d} | lr {lr:.2e} | train {tr_loss:.4f} / {tr_acc:.4f} | "
                      f"val {va_loss:.4f} / {va_acc:.4f} | {elapsed:4.1f}s")
            if early_stopping is not None and early_stopping.step(va_loss, eval_model):
                print(f"early stopping: no improvement for {early_stopping.patience} epochs")
                break
        if early_stopping is not None:                       # select the epoch with the best validation LOSS...
            early_stopping.restore(eval_model)
            sel_epoch = early_stopping.best_epoch
        else:
            sel_epoch = len(history["val_loss"])            # ...or keep the last epoch
        sel_loss, sel_acc = history["val_loss"][sel_epoch - 1], history["val_acc"][sel_epoch - 1]
        history["selected"] = {"epoch": sel_epoch, "val_loss": sel_loss, "val_acc": sel_acc}
        if verbose:
            print(f"selected model: epoch {sel_epoch}, val loss {sel_loss:.4f}, val acc {sel_acc:.4f}")
        return history


    def plot_history(histories, keys=("val_loss", "val_acc", "lr")):
        fig, axes = plt.subplots(1, len(keys), figsize=(5 * len(keys), 3.2))
        for ax, key in zip(axes, keys):
            for name, h in histories.items():
                ax.plot(range(1, len(h[key]) + 1), h[key], "o-", ms=3, label=name)
            ax.set_title(key); ax.set_xlabel("epoch")
        axes[0].legend(fontsize=8); plt.tight_layout(); plt.show()

    # Small in-memory subsets (normalised exactly like fm_transform) for quick experiments
    def _fm_subset(idx):
        x = ((raw_train.data[idx].float() / 255 - FM_MEAN) / FM_STD).unsqueeze(1)
        return TensorDataset(x, raw_train.targets[idx])

    small_ds = _fm_subset(FM_TRAIN_IDX[:4096])      # 4,096 training images
    small_val = _fm_subset(FM_VAL_IDX[:2000])       # 2,000 validation images

    def save_checkpoint(path, model, optimizer, scheduler=None, epoch=0, best_metric=None,
                        loader_generator=None, config=None):
        np_state = np.random.get_state()
        ckpt = {
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict() if scheduler is not None else None,
            "epoch": epoch,
            "best_metric": best_metric,
            "config": config or {},
            "rng": {
                "torch": torch.get_rng_state(),
                "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [],
                "python": random.getstate(),
                # NumPy's state contains an array; store it as a list so the safe loader accepts it
                "numpy": (np_state[0], np_state[1].tolist(), *np_state[2:]),
            },
            "loader_generator": loader_generator.get_state() if loader_generator is not None else None,
        }
        tmp = Path(str(path) + ".tmp")
        torch.save(ckpt, tmp)             # write to a temporary file first...
        os.replace(tmp, path)             # ...then rename: an interruption can never leave a half-written checkpoint


    def load_checkpoint(path, model, optimizer=None, scheduler=None, loader_generator=None):
        ckpt = torch.load(path, map_location=device, weights_only=True)   # safe loading: tensors and plain data only
        model.load_state_dict(ckpt["model"])
        if optimizer is not None:
            optimizer.load_state_dict(ckpt["optimizer"])
        if scheduler is not None and ckpt["scheduler"] is not None:
            scheduler.load_state_dict(ckpt["scheduler"])
        rng = ckpt["rng"]
        torch.set_rng_state(rng["torch"].cpu())
        if torch.cuda.is_available() and rng["cuda"]:
            torch.cuda.set_rng_state_all([s.cpu() for s in rng["cuda"]])
        random.setstate(rng["python"])
        name, keys, *rest = rng["numpy"]
        np.random.set_state((name, np.array(keys, dtype=np.uint32), *rest))
        if loader_generator is not None and ckpt["loader_generator"] is not None:
            loader_generator.set_state(ckpt["loader_generator"].cpu())
        return ckpt

    def conv_bn_relu(c_in, c_out):
        return nn.Sequential(nn.Conv2d(c_in, c_out, 3, padding=1, bias=False), nn.BatchNorm2d(c_out), nn.ReLU(inplace=True))


    class SmallCNN(nn.Module):
        def __init__(self, num_classes=10, width=32, p_drop=0.3):
            super().__init__()
            w = width
            self.features = nn.Sequential(
                conv_bn_relu(1, w), conv_bn_relu(w, w), nn.MaxPool2d(2),              # 28x28 -> 14x14
                conv_bn_relu(w, 2 * w), conv_bn_relu(2 * w, 2 * w), nn.MaxPool2d(2),  # 14x14 -> 7x7
                conv_bn_relu(2 * w, 4 * w),                                           # 7x7, 128 channels
                nn.AdaptiveAvgPool2d(1),                                              # -> 1x1
            )
            self.head = nn.Sequential(nn.Flatten(), nn.Dropout(p_drop), nn.Linear(4 * w, num_classes))

        def forward(self, x):
            return self.head(self.features(x))

    CNN_PATH = CKPT_DIR / "smallcnn_fashion.pt"

    def train_small_cnn(epochs=12, seed=0):
        # The recipe of Section 7.7: AdamW with weight-decay groups and a OneCycle schedule.
        set_seed(seed)
        model = SmallCNN().to(device)
        E = n_epochs(epochs)
        opt = torch.optim.AdamW(param_groups_weight_decay(model, 0.05), lr=1e-3)
        sched = S.OneCycleLR(opt, max_lr=3e-3, total_steps=E * len(train_loader))
        hist = fit(model, train_loader, val_loader, opt, E, scheduler=sched,
                   early_stopping=EarlyStopping(patience=100))        # patience=100: just keep the best epoch
        torch.save({"state_dict": model.state_dict()}, CNN_PATH)
        return model, hist

    def get_trained_cnn():
        # Load the CNN trained in Chapter 7 if it was saved; otherwise train it now.
        model = SmallCNN().to(device)
        if CNN_PATH.exists():
            model.load_state_dict(torch.load(CNN_PATH, map_location=device, weights_only=True)["state_dict"])
            print(f"loaded the trained SmallCNN from {CNN_PATH}")
        else:
            print("no saved SmallCNN found: training one with the recipe of Section 7.7 ...")
            model, _ = train_small_cnn()
        return model.eval()
    print("Restored the tools defined in earlier chapters.")

In [ ]:
class Residual(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.f = nn.Sequential(conv_bn_relu(channels, channels), conv_bn_relu(channels, channels))

    def forward(self, x):
        return x + self.f(x)                      # the skip connection

def resnet9(num_classes=10):
    return nn.Sequential(
        conv_bn_relu(3, 64),
        conv_bn_relu(64, 128), nn.MaxPool2d(2), Residual(128),       # 32 -> 16
        conv_bn_relu(128, 256), nn.MaxPool2d(2),                     # 16 -> 8
        conv_bn_relu(256, 512), nn.MaxPool2d(2), Residual(512),      # 8 -> 4
        nn.AdaptiveMaxPool2d(1), nn.Flatten(), nn.Linear(512, num_classes),
    )

print(f"ResNet-9 parameters: {count_params(resnet9()):,}")
print("output shape:", resnet9()(torch.randn(2, 3, 32, 32)).shape)

**Data and augmentation.** We hold out 5,000 training images for validation and compute the per-channel normalisation statistics on the remaining 45,000. The training transform uses the two standard CIFAR augmentations: a random crop after padding by 4 pixels (small random translations) and a horizontal flip.

⚠️ **The transform trap.** If you split one dataset object with `Subset`, both subsets share its transform, so the validation images would be randomly cropped and flipped too. Create **two** dataset objects over the same files, one with the training transform and one with the evaluation transform, and index both with the same split.

In [ ]:
cifar_raw = datasets.CIFAR10(DATA_DIR, train=True, download=True)          # about 170 MB the first time
perm = torch.randperm(len(cifar_raw), generator=torch.Generator().manual_seed(0))
c_val_idx, c_train_idx = perm[:5000], perm[5000:]
pix = torch.tensor(cifar_raw.data[c_train_idx.numpy()[:20000]]).float() / 255  # (N, 32, 32, 3); a 20k sample is plenty
C_MEAN, C_STD = pix.mean(dim=(0, 1, 2)).tolist(), pix.std(dim=(0, 1, 2)).tolist()
print("per-channel mean:", [round(v, 3) for v in C_MEAN], "std:", [round(v, 3) for v in C_STD])

norm = [T.ToImage(), T.ToDtype(torch.float32, scale=True), T.Normalize(C_MEAN, C_STD)]
cifar_train_tf = T.Compose([T.RandomCrop(32, padding=4, padding_mode="reflect"), T.RandomHorizontalFlip(), *norm])
cifar_eval_tf = T.Compose(norm)

c_train = maybe_subset(Subset(datasets.CIFAR10(DATA_DIR, train=True, transform=cifar_train_tf), c_train_idx.tolist()))
c_val = maybe_subset(Subset(datasets.CIFAR10(DATA_DIR, train=True, transform=cifar_eval_tf), c_val_idx.tolist()))
c_test = maybe_subset(datasets.CIFAR10(DATA_DIR, train=False, download=True, transform=cifar_eval_tf))
kw = dict(num_workers=NUM_WORKERS, pin_memory=PIN, persistent_workers=NUM_WORKERS > 0)
c_train_loader = DataLoader(c_train, batch_size=256, shuffle=True, drop_last=True, **kw)
c_val_loader = DataLoader(c_val, batch_size=512, **kw)
c_test_loader = DataLoader(c_test, batch_size=512, **kw)

**The recipe** combines most of this notebook: SGD with Nesterov momentum and weight-decay parameter groups (Chapter 3), a OneCycle schedule stepped per batch and gradient clipping (Chapters 3–4), label smoothing (Notebook 1, §5.6), mixed precision and channels-last memory layout (Chapter 11), and keeping the best epoch (Chapter 4). Expect roughly 20–30 seconds per epoch on a T4 and a test accuracy in the low-to-mid 90s.

In [ ]:
set_seed(0)
resnet = resnet9().to(device).to(memory_format=torch.channels_last)
C_EPOCHS = n_epochs(20)
opt = torch.optim.SGD(param_groups_weight_decay(resnet, 5e-4), lr=0.1, momentum=0.9, nesterov=True)   # OneCycleLR sets the actual lr
sched = S.OneCycleLR(opt, max_lr=0.1, total_steps=C_EPOCHS * len(c_train_loader))
scaler = torch.amp.GradScaler(device.type) if AMP_DTYPE == torch.float16 else None
t0 = time.perf_counter()
hist_c = fit(resnet, c_train_loader, c_val_loader, opt, C_EPOCHS,
             loss_fn=nn.CrossEntropyLoss(label_smoothing=0.1), scheduler=sched, max_grad_norm=2.0,
             early_stopping=EarlyStopping(patience=100), amp_dtype=AMP_DTYPE, scaler=scaler)
print(f"training took {(time.perf_counter() - t0) / 60:.1f} min")
record("Ch14 ResNet-9 on CIFAR-10, test accuracy", evaluate(resnet, c_test_loader)[1])

Extensions to try: an EMA of the weights (Chapter 6), more epochs, or test-time augmentation (average the predictions for each test image and its mirror image).

## 14.2 ★ Extending autograd: custom `autograd.Function`
Occasionally you need an operation whose backward pass **you** define: a more numerically stable gradient, a custom GPU kernel, or a deliberately "wrong" gradient. Subclass `torch.autograd.Function` and implement two static methods: `forward` computes the output and saves what the backward pass needs; `backward` receives $\partial L/\partial \text{output}$ and returns $\partial L/\partial \text{input}$.

**Example: the straight-through estimator (STE).** Rounding has zero gradient almost everywhere, so a network with rounded (quantised) weights cannot be trained by backpropagation directly. The STE uses rounding in the forward pass but pretends it was the identity in the backward pass. This trick is used for quantisation-aware training and in VQ-VAEs.

In [ ]:
class RoundSTE(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        return torch.round(x)

    @staticmethod
    def backward(ctx, grad_output):
        return grad_output                       # pretend d round(x)/dx = 1

x = torch.tensor([0.2, 1.7, -0.6], requires_grad=True)
torch.round(x).sum().backward()
print("gradient through torch.round:", x.grad)  # zeros: nothing would train
x.grad = None
RoundSTE.apply(x).sum().backward()
print("gradient through RoundSTE:   ", x.grad)  # ones: gradients pass straight through
print("you will often see the same trick written as:  x + (torch.round(x) - x).detach()")

When you write a custom backward that is meant to be the **true** derivative, verify it with `torch.autograd.gradcheck`, which compares it with finite differences (in `float64`):

In [ ]:
class StableSoftplus(torch.autograd.Function):
    # softplus(x) = log(1 + e^x), computed without overflow; derivative = sigmoid(x)
    @staticmethod
    def forward(ctx, x):
        ctx.save_for_backward(x)                 # stash what backward needs
        return torch.clamp(x, min=0) + torch.log1p(torch.exp(-x.abs()))

    @staticmethod
    def backward(ctx, grad_output):
        (x,) = ctx.saved_tensors
        return grad_output * torch.sigmoid(x)

z = torch.randn(6, dtype=torch.float64, requires_grad=True)
print("matches F.softplus:", torch.allclose(StableSoftplus.apply(z), F.softplus(z)))
print("gradcheck passed:  ", torch.autograd.gradcheck(StableSoftplus.apply, (z,)))
print("no overflow at x = 1000:", StableSoftplus.apply(torch.tensor([1000.0])).item())

## 14.3 ★ Are the probabilities trustworthy? Calibration
A classifier is **calibrated** if, among all predictions made with 80% confidence, about 80% are correct. Modern networks trained with cross-entropy tend to be **over-confident**. This matters whenever a probability drives a decision: flagging cases for human review, combining models, or estimating risk.

**Expected calibration error (ECE):** sort predictions into confidence bins (0–10%, 10–20%, …), and in each bin compare the average confidence with the accuracy; ECE is the weighted average of the gaps.

**Temperature scaling** is the simplest fix: divide all logits by one scalar $T > 0$, fitted on the **validation** set by minimising the negative log-likelihood. $T > 1$ softens over-confident predictions. Since dividing by $T$ does not change which logit is largest, **accuracy is unchanged**.

In [ ]:
@torch.no_grad()
def collect_logits(model, loader):
    model.eval()
    out, ys = [], []
    for x, y in loader:
        out.append(model(x.to(device)).float().cpu()); ys.append(y)
    return torch.cat(out), torch.cat(ys)

def ece(logits, y, n_bins=15):
    conf, pred = torch.softmax(logits, dim=1).max(dim=1)
    correct = (pred == y).float()
    bins = torch.linspace(0, 1, n_bins + 1)
    total = 0.0
    for lo, hi in zip(bins[:-1], bins[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            total += m.float().mean() * (conf[m].mean() - correct[m].mean()).abs()
    return float(total)

cnn = get_trained_cnn()                                       # the Chapter 7 network
val_logits, val_y = collect_logits(cnn, val_loader)
test_logits, test_y = collect_logits(cnn, test_loader)

log_t = torch.zeros(1, requires_grad=True)                 # optimise log T so that T stays positive
opt_t = torch.optim.LBFGS([log_t], lr=0.1, max_iter=100, line_search_fn="strong_wolfe")
def closure():
    opt_t.zero_grad()
    loss = F.cross_entropy(val_logits / log_t.exp(), val_y)
    loss.backward()
    return loss
opt_t.step(closure)
temperature = log_t.exp().item()

print(f"fitted temperature T = {temperature:.3f}")
print(f"test ECE before: {ece(test_logits, test_y):.4f} | after: {ece(test_logits / temperature, test_y):.4f}")
record("Ch14 SmallCNN test ECE after temperature scaling", ece(test_logits / temperature, test_y))
print(f"test NLL before: {F.cross_entropy(test_logits, test_y):.4f} | after: {F.cross_entropy(test_logits / temperature, test_y):.4f}")

In deployment, fold the temperature into the model (divide the logits by $T$ inside `forward`) so that the reported probabilities are the calibrated ones.

## 14.4 ★ Profiling with `torch.profiler`
Chapter 11 timed whole steps. The profiler breaks a step down **by operation**, on the CPU and the GPU, which tells you which layers or kernels dominate:

In [ ]:
from torch.profiler import profile, ProfilerActivity

model = SmallCNN().to(device)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
xb, yb = next(iter(train_loader))
xb, yb = xb.to(device), yb.to(device)
activities = [ProfilerActivity.CPU] + ([ProfilerActivity.CUDA] if device.type == "cuda" else [])
with profile(activities=activities) as prof:
    for _ in range(5):
        loss = F.cross_entropy(model(xb), yb)
        opt.zero_grad(); loss.backward(); opt.step()
    sync()
sort_key = "self_device_time_total" if device.type == "cuda" else "self_cpu_time_total"
print(prof.key_averages().table(sort_by=sort_key, row_limit=10))

Read the table from the top: the operations that take the most time. For this CNN you should see convolution kernels (forward and backward) and BatchNorm near the top. `prof.export_chrome_trace("trace.json")` writes a timeline you can open in `chrome://tracing` or Perfetto, which shows gaps where the GPU waits for the CPU (often the data loader).

## 14.5 ★ Multiple GPUs: a preview of DDP
**Distributed Data Parallel (DDP)** is the standard way to train on several GPUs. One process runs per GPU; each holds a full copy of the model and trains on a different slice of every batch; after `backward()`, DDP averages the gradients across processes (an "all-reduce"), so all copies take identical steps. Larger models that do not fit on one GPU use **FSDP** (fully sharded data parallel), which also splits the weights, gradients and optimizer state across GPUs.

The changes to a training script are small. This code is not run here (it needs several GPUs); it is launched with `torchrun --nproc_per_node=4 train.py`:

```python
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data.distributed import DistributedSampler

dist.init_process_group("nccl")                         # torchrun provides rank and world size
rank = dist.get_rank(); local_rank = int(os.environ["LOCAL_RANK"])
torch.cuda.set_device(local_rank)

model = DDP(MyModel().cuda(local_rank), device_ids=[local_rank])
sampler = DistributedSampler(train_ds, shuffle=True)    # each process gets a different shard
loader = DataLoader(train_ds, batch_size=per_gpu_batch, sampler=sampler)

for epoch in range(epochs):
    sampler.set_epoch(epoch)                            # ⚠️ otherwise every epoch uses the same shuffle
    for x, y in loader:
        ...                                             # the usual five steps; DDP syncs gradients in backward()
    if rank == 0:                                       # only one process saves and logs
        torch.save(model.module.state_dict(), "ckpt.pt")   # note .module: DDP wraps the model ("module." prefix again)
dist.destroy_process_group()
```

Things to know: the **effective batch size** is the per-GPU batch × the number of GPUs (scale the learning rate accordingly, often with warm-up); BatchNorm statistics are per GPU unless you convert to `nn.SyncBatchNorm`; and metrics computed on one process cover only its shard.

## 14.6 ★ Interview questions and an error-message decoder

### The error-message decoder
| Message (abridged) | Usual cause | Fix |
| --- | --- | --- |
| `Expected all tensors to be on the same device` | Model on GPU, data on CPU (or a tensor created inside `forward` without `device=`) | `.to(device)` for the batch; create new tensors with `device=x.device` |
| `mat1 and mat2 shapes cannot be multiplied (64x3136 and 784x10)` | The input size of a `Linear` layer does not match the flattened features | Print the shape before the layer; fix `in_features` |
| `mat1 and mat2 must have the same dtype` / `expected scalar type Float but found Double` | NumPy `float64` data | `.float()` at the NumPy boundary |
| `expected scalar type Long` (in a loss) | Class labels stored as floats | `y.long()` |
| `CUDA error: device-side assert triggered` | A label outside `[0, C)`, or an index out of range | Check `y.min(), y.max()`; rerun on CPU for a readable error; restart the runtime |
| `view size is not compatible with input tensor's size and stride` | `view` on a non-contiguous tensor | `.reshape(...)` or `.contiguous().view(...)` |
| `Trying to backward through the graph a second time` | Reusing a graph, often by carrying a tensor from the previous step | Recompute the forward pass; `.detach()` carried-over tensors |
| `one of the variables needed for gradient computation has been modified by an inplace operation` | An in-place edit of a tensor autograd saved | Use the out-of-place version |
| `Can't call numpy() on Tensor that requires grad` | Converting a tensor that is part of the graph | `.detach().cpu().numpy()` |
| `CUDA out of memory` | Batch too large, graphs kept alive, evaluation without `no_grad` | Smaller batch + accumulation, AMP, `no_grad` for evaluation, `.item()` for logging |
| `Error(s) in loading state_dict ... Missing key(s) ... Unexpected key(s)` | Architecture mismatch, or a `module.` / `_orig_mod.` prefix | Compare keys; strip prefixes; `strict=False` only with inspection |
| `DataLoader worker (pid …) exited unexpectedly` | Crash inside a worker; on Windows/macOS notebooks, workers cannot import the `Dataset` class | `num_workers=0` to see the real error; move the class into a `.py` file |

### ✔️ Check yourself

Commit to an answer (say it out loud or write it down) *before* you expand it.

**1.** What exactly does `optimizer.zero_grad()` do, and why is it needed?

<details>
<summary>Answer</summary>

It clears (by default, sets to `None`) the `.grad` of every parameter the optimizer manages. It is needed because `backward()` accumulates into `.grad`.

</details>

**2.** What are the differences between `model.eval()`, `torch.no_grad()` and `torch.inference_mode()`?

<details>
<summary>Answer</summary>

`eval()` changes layer behaviour (Dropout off, BatchNorm uses running statistics) but does not affect autograd. `no_grad()` stops graph recording. `inference_mode()` also stops recording and is a little faster, but its tensors can never be used in autograd later.

</details>

**3.** Why does a classifier output logits instead of probabilities?

<details>
<summary>Answer</summary>

Losses such as `CrossEntropyLoss` combine softmax and log in a numerically stable way internally. Applying softmax first is both less stable and wrong (double softmax).

</details>

**4.** `view` vs `reshape` vs `permute`?

<details>
<summary>Answer</summary>

`view` reinterprets the memory with a new shape and never copies (fails on incompatible strides). `reshape` does the same when possible and copies otherwise. `permute` reorders dimensions by permuting strides, without copying, producing a usually non-contiguous view.

</details>

**5.** What is the difference between a parameter and a buffer? Give an example of each.

<details>
<summary>Answer</summary>

Both are saved in the `state_dict` and moved by `.to()`. Parameters are trained by the optimizer (`Linear.weight`); buffers are not (BatchNorm `running_mean`, normalisation constants).

</details>

**6.** How would you fine-tune a pretrained ResNet on 2,000 images of a new task?

<details>
<summary>Answer</summary>

Replace the head; start with a linear probe or train the head with the backbone frozen (keeping frozen BatchNorm in eval mode), then fine-tune the whole network with a small backbone learning rate (e.g. 10× smaller than the head), warm-up, augmentation and weight decay; select by validation; test once.

</details>

**7.** What does `GradScaler` do, and when do you need it?

<details>
<summary>Answer</summary>

With `float16` autocast, it scales the loss up before `backward()` so small gradients do not underflow, unscales before the step, and skips steps with inf/NaN gradients while adapting the scale. It is not needed for `bfloat16`.

</details>

**8.** Your training loss decreases but validation accuracy stays at chance. Name three possible causes.

<details>
<summary>Answer</summary>

Labels misaligned with inputs in the validation (or training) data; different preprocessing for validation; evaluation code bugs such as comparing with the wrong labels or forgetting `eval()` with heavy dropout; severe overfitting to a tiny training set.

</details>

**9.** Why must BatchNorm statistics be recomputed after SWA?

<details>
<summary>Answer</summary>

The running statistics belong to the individual snapshots; the averaged weights produce different activations, so the statistics are wrong until recomputed with `update_bn`.

</details>

**10.** How do you resume training exactly where it stopped?

<details>
<summary>Answer</summary>

Restore model, optimizer and scheduler `state_dict`s, the epoch counter, the best metric, the RNG states (torch, CUDA, Python, NumPy) and the data loader's generator, then continue from the next epoch.

</details>

---
# Summary: the complete modern training loop
Every technique in this notebook fits into one skeleton. When you read a research repository, look for these pieces; when you write one, include them.

```python
# ---- configuration and reproducibility ----
cfg = Config(...); set_seed(cfg.seed)

# ---- data: split first, statistics from the training split, augmentation only for training ----
train_loader = DataLoader(train_ds, cfg.batch_size, shuffle=True, drop_last=True,
                          num_workers=4, pin_memory=True, persistent_workers=True)

# ---- model, optimizer with parameter groups, schedule, extras ----
model = build_model(cfg).to(device)                                    # pretrained? replace head, freeze?
optimizer = torch.optim.AdamW(param_groups_weight_decay(model, cfg.wd), lr=cfg.lr)
scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, cfg.lr, total_steps=cfg.epochs * len(train_loader))
scaler = torch.amp.GradScaler("cuda")                                  # float16 only
ema = AveragedModel(model, multi_avg_fn=get_ema_multi_avg_fn(0.999))
start_epoch = load_checkpoint_if_exists(...)                           # resume support

for epoch in range(start_epoch, cfg.epochs):
    model.train()
    for x, y in train_loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        with torch.autocast("cuda", dtype=torch.float16):
            loss = loss_fn(model(x), y)
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer); scaler.update()
        scheduler.step()
        ema.update_parameters(model)
        # log loss.item(), grad_norm, lr
    val_metrics = evaluate(ema, val_loader)                            # eval() + no_grad inside
    save_checkpoint("last.pt", ...); maybe_save_best("best.pt", ...)
    if early_stopping.step(val_metrics["loss"], ema): break

# ---- once: test, export, document ----
```

## Your results
Every number below was measured by **your** run of this notebook. Values quoted in the text are typical outcomes, not guarantees; when they disagree with yours, your run is the ground truth (and a discrepancy is worth understanding).

In [ ]:
for name, value in RESULTS.items():
    print(f"{name:62s} {value:.4f}")

## Where to go next
- **The official PyTorch tutorials and documentation** (pytorch.org/tutorials, pytorch.org/docs): the reference for every API used here.
- **Sequence models and attention:** embeddings, recurrent networks and transformers use exactly the tools of this notebook (`einsum`, `view`/`transpose`/`contiguous`, masking, LayerNorm, warm-up schedules, clipping).
- **Libraries you will meet:** `timm` (pretrained vision models), Hugging Face `transformers` and `accelerate`, PyTorch Lightning, `torchmetrics`, Weights & Biases / TensorBoard.
- **Capstone ideas:** fine-tune a pretrained model on a dataset from your own field (defect images, spectrograms of machine sounds, satellite tiles); reproduce one table from a paper; or deploy your Fashion-MNIST classifier as a small web service with ONNX Runtime.

You now know how PyTorch works underneath, not just which functions to call. That is what lets you read other people's code, debug what goes wrong, and build what comes next.